# ★ V9 是什么：多 seed × 轮数扫描（V8 主干的稳定性与欠训练跟进）

**一句话**：XLM-RoBERTa-large + 编码器标准配方**一字不改**，只扫两个训练编排变量：
**种子（42/43/44）× 轮数（4/5）**，共 6 次独立联合训练；数据切分种子恒为 42 ⇒
全部运行的验证集逐行对齐（跨 seed 对照与多 seed 集成的前提）。

**为什么要扫（全部来自 V8 云端实测，`v8/云端实跑_V8结果与结论.md`）**：

| 事实 | 数值 | 含义 |
|---|---|---|
| 逐轮合计指标1 | e1 0.4906 → e2 0.5303 → e3 **0.5783**（仍在涨） | 欠训练，4/5 轮可能还有增量 |
| 历史新高的显著性 | 单种子单次运行，+0.0801/+0.0838 vs v4.4/V7 | 需多 seed 复验（种子间差异多大？） |
| 训练成本 | 34.2 分钟/次、峰值 6.10 GiB、评估 33 秒 | 扫描 6 次的成本 ≈ Qwen 单次的 1/3 |
| 预算 | 87 ms/样本（实测）× 27 个样本·轮 × 1.3 ≈ **6.7 h** | ≤ 8 h 用户口径 |

**Step10 出什么（判读全部用既有阈值：合计 0.0652/0.1526/0.0463，单设置另表）**：

| 表 | 回答的问题 |
|---|---|
| 表 1 逐运行汇总 | 6 次运行各自的合计三指标 + 相对 V8（3 轮 seed42）的 Δm1 判读 |
| 表 2 逐轮轨迹 | 欠训练是否持续到 4/5 轮（末轮仍涨与否） |
| 表 3 跨 seed 稳定性 | 同轮数档内 3 个 seed 的均值±SD 与极差（V8 新高是不是种子运气） |
| 表 4 多 seed 集成 | 逐例句分数均值（e4 组 / e5 组 / 全部 6 个）vs 最佳单运行 |

**预注册判读（可证伪）**：
- **R1** 若 3 seed 极差 ≤ 0.0652 ⇒ V8 的 +0.08 跨 seed 稳定；否则如实报「种子敏感」；
- **R2** 若 4→5 轮的 seed 均值差 > 0.0652 或轨迹末轮仍涨 ⇒ 欠训练确认，5 轮档为提交候选；
  若两者皆否 ⇒ 3–4 轮即平台，V8 的 3 轮已够；
- **R3** 多 seed 集成相对最佳单运行的 Δm1 > 0.0652 才算显著；否则集成不进提交；
- **R4（红线）** 每个运行与每个集成臂的指标2 非 nan、组内 std 退化目标数 = 0
  （集成是预测均值，会改变组内 std ⇒ 退化目标数必须逐位检查）。

**与 V7/V8 校准机制的关系**：V8 云端判读「校准在强主干上无剩余空间」
（shrink/full Δm1 = −0.012/−0.014，噪声内）⇒ V9 **不含校准**；
V8 的校准管线在 `v8/` 冻结可回，测试集阶段如需再按「全验证重估 β」流程启用。

# ComPartMENT 机制实验：XLM-RoBERTa-large（V9 多 seed × 轮数扫描） 的组合性预测管线

**任务**：给定一个多词表达（MWE，如 `account book` / `crack down`）和一句真实语料，
输出该表达**在这条语境里**的组合性分数（0–5 连续值）。

**这份 notebook 交付什么**：一套机制实验训练管线，主干是
`XLM-RoBERTa-large（V9 多 seed × 轮数扫描）`（`FacebookAI/xlm-roberta-large`），**关闭全部结构模块**（m1/m2/m4 均不启用）与机制损失（softcl/ce 为 0），
只复刻已验证的编码器标准配方；V9 的实验变量只有**训练种子与轮数**（训练编排层），
模型结构与损失配方零改动。

| 模块 | 做什么 | 开关 |
|---|---|---|
| **模块 1** 序数-分布头 | 回归头输出 6 类（0–5）logits，读数 = 期望值 `Σ k·p(k)`，训练用软标签交叉熵；分布宽度 = 不确定度通道 | `COMPARTMENT_V5_MODULES` 含 `m1` |
| **模块 2** 类型-语境双流 + 敏感度门 | 类型流只编码 MWE、语境流编码整串，`分数 = (1−g)·μ̂_type + g·δ_ctx`；门 g 只看类型表征 ⇒ 逐目标 | 含 `m2` |
| **模块 4** 角色嵌入 + 交换一致性 | 共享主干 + 角色向量 ⇒ Mod/Head 变成「同一 MWE 的两个视角」；再加「成分交换后第 0 列 ≈ 原来第 1 列」的一致性正则 | 含 `m4` |

**怎么只跑一部分模块**（云端消融，不用改代码）：
`COMPARTMENT_V5_MODULES="m1"` / `"m1,m2"` / `"m1,m2,m4"`（默认全开）。
实际生效的开关会写进 `outputs/summary/run_settings_*.json`。

本文件验证的机制是 **多 seed × 轮数扫描 + 多 seed 预测均值集成（V8 主干的稳定性与欠训练跟进）**（见 Step4 / Step7 的说明）。

**跑什么（V9）**：打开 notebook 一路执行到底，它会**自动完成 6 次联合训练**——
XLM-RoBERTa-large 编码器标准配方（t3 全参、有效 batch 16、结构模块与机制损失
全关）在 **3 个训练种子（42/43/44）× 2 个轮数档（4/5）** 上各训一遍；数据切分
恒为 seed 42 ⇒ 全部运行验证集逐行对齐。跑完在 **Step10** 出四张表：逐运行汇总、
逐轮轨迹、跨 seed 稳定性、多 seed 预测均值集成。
**不需要手动切换任何开关**（网格在配置区 `V9_SWEEP`，改它即改扫描计划）。

**与另一份 notebook（DeBERTa / Qwen）的关系**：两份文件共用同一份数据处理、
超参、指标与绘图，只切换 `MODEL_KEY` 一个变量；「读出方式 / 机制项」按实验各自设定，
其余逐字相同 ⇒ 差异能干净地归因到机制本身。

---

## 五个部分

| 部分 | 对应章节 | 内容 |
|---|---|---|
| 问题与背景 | 本节 + Step1 | 三项官方指标各自的含义，指标 2 为什么是硬门槛 |
| 原理与设计 | Step3 / Step4 | 输入怎么构造、目标怎么定位、主干怎么接回归头、损失怎么写 |
| 实现代码 | Step1–Step8 | 全部可执行代码，注释说明每段的作用 |
| 结果分析 | Step9 | **评估标准**（模型在 6 个评测设置上的三项指标）与**参考对照**（随机初始化 / type-level oracle）分区呈现 |
| 结论与延伸 | 文末 | 运行前置条件、显存降级顺序、判断实验有没有做对的三个指标 |

---

## 全局数据流（谁只读、谁被更新）

```
train/{subset}-train.tsv
        │  ① 载入：删 ContextID + 物理打乱 + 去重            （Step2）
        ▼
   DataFrame
        │  ② 定位 MWE → 标记上下文 → 拼输入模板              （Step3）
        ▼
   {"text", "labels[2]", "loss_mask[2]", "group_index"}
        │  ③ 分词（动态 padding）→ 目标级互斥切分             （Step3）
        ▼
   datasets.Dataset ──Collator──► batch{input_ids, attention_mask, labels, group_index}
        │
        ▼
   AutoModel 主干（只读参数，全部更新）
        │  ④ 池化 → 线性回归头 → sigmoid×5                   （Step4）
        ▼
   logits ∈ R^{B×2}
        │  ⑤ L = Huber(掩码加权) + λ·组内排序 + μ·组内std     （Step7）
        ▼
   反向传播 → 更新主干与回归头
        │
        ▼
   评估：指标1 ρ / 指标2 ρ / 指标3 RMSE                     （Step5, Step9）
```

- **只读**：数据文件、标签列。
- **被更新**：主干全部参数 + 回归头。
- **不参与训练**：目标字符串只作为 `group_index`（分组键）与输入里的 MWE 文本，
  **不作为任何数值特征**；`ContextID` 在载入时就被删除，防止行序泄漏。

## Step1 导入包

**这一节要解决什么**：把「会随机器、随人变」的东西全部钉死在一个地方——
模型下载端点、随机种子、设备、精度、序列长度、批大小、模型 id、学习率、运行计划。
其余 8 个 Step 一律不写死任何常量，只从这里取。

**为什么必须这样**：本项目要跑 2 个模型 × 4 个子集 × 后续若干组消融。
如果 `lr=2e-5` 写在 Step6、`max_length=160` 写在 Step3，
那么「换个模型跑一遍」就要改 6 个地方，改漏一处就会得到一张**无法归因**的对照表。
对照实验的可信度 100% 取决于配置是否只有一个入口。

**先说一件本项目特有的事：模型权重从哪儿下。**

云端 GPU（Kaggle / Colab / 阿里云等海外实例）**可以直连 `huggingface.co`**，
本机直连通常不通。所以默认策略是：

1. **先探测官方源**（`https://huggingface.co`）是否真的可达——用一次真实的 HTTPS 请求，
   不是 ping，也不是 socket 探测（很多网络环境 443 端口能连但 HTTP 被重置）；
2. 官方源可达 → 用官方；不可达 → 换镜像站 `https://hf-mirror.com`；
3. **下载时再兜一层**：即便探测时通了、下载时断了，也会自动换另一个端点重试。

探测与端点切换都发生在 `import transformers` **之前**或**显式传参**，
不依赖 `HF_ENDPOINT` 全局变量的缓存时机——这是「我明明改了镜像怎么还在走官方」
这类现象最常见的来源。

In [ ]:
# ============ 0. 端点选择：必须在 import transformers 之前完成 ============
# huggingface_hub 在 import 时缓存端点地址，之后再改环境变量无效；
# 因此在 import 前定好端点，下载权重时用 snapshot_download(endpoint=...) 显式传参。
import os
import urllib.request

# PYTORCH_CUDA_ALLOC_CONF 只在 CUDA 上下文初始化之前设置才生效；
# expandable_segments 能减少显存碎片。
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# =============================================================================
# 只让本进程看见一张卡（必须在 import torch 之前设置）：
# 进程可见 GPU ≥ 2 时，transformers Trainer 会自动套 nn.DataParallel，
# 其反向传播要额外复制一整份梯度（1.5B 模型 +5.75 GiB）→ 单卡装不下的必然 OOM。
# nn.DataParallel 是数据并行、不省显存；真正合并多卡显存需要 ZeRO-3/FSDP 多进程启动，
# 单进程跑不了 ⇒ 每条基线绑定一张卡，靠两个进程并行提速。
# CUDA_VISIBLE_DEVICES 在 CUDA 初始化时就被读走，之后再改无效。
# =============================================================================
_GPU_INDEX = int(os.environ.get("COMPARTMENT_GPU", "0"))
os.environ["CUDA_VISIBLE_DEVICES"] = str(_GPU_INDEX)
print(f"[环境] CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} "
      f"（本进程只看见这 1 张卡 → Trainer 不会套 nn.DataParallel → 梯度显存不翻倍）")

OFFICIAL_ENDPOINT = "https://huggingface.co"
MIRROR_ENDPOINT = "https://hf-mirror.com"

# COMPARTMENT_HF_MODE 决定下载端点策略：
#   "auto"     → 探测官方源，能连就用官方，不能连就换镜像（**默认**）
#   "official" → 强制官方源
#   "mirror"   → 强制镜像站
#   "cache"    → 只用本地 HuggingFace 缓存，不联网（离线复现 / 冒烟测试用）
HF_MODE = os.environ.get("COMPARTMENT_HF_MODE", "auto").strip().lower()


def probe_http(url, timeout=8):
    """发一次真实的 HTTPS 请求，判断 URL 是否可达。
    【为什么不 ping / 不测 socket】很多网络环境 TCP 443 能握手但 HTTP 被 RST，
    只有真实请求才能反映「能不能下模型」。"""
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "compartment-probe"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return 200 <= resp.status < 500
    except Exception:
        return False


def choose_endpoints(model_id):
    """返回按优先级排序的端点列表；第一个失败会自动尝试下一个。"""
    if HF_MODE == "cache":
        print("端点策略: cache（只用本地缓存，不联网）")
        return []
    if HF_MODE == "mirror":
        print("端点策略: mirror（强制镜像站）")
        return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]
    if HF_MODE == "official":
        print("端点策略: official（强制官方源）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    api = f"{OFFICIAL_ENDPOINT}/api/models/{model_id}"
    ok = probe_http(api)
    print(f"官方源连通性探测: {'通' if ok else '不通'}  ({api})")
    if ok:
        print("→ 使用官方 huggingface.co（云端 GPU 直连即走这条）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    print("→ 官方源不可达，改用镜像站 hf-mirror.com")
    return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]


os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # 短句场景收益极低；fork 在 macOS 上还刷屏

In [ ]:
# ============ 1. 导入其余依赖 ============
import gc
import inspect
import json
import math
import platform
import re
import shutil
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.stats import ConstantInputWarning

import matplotlib.pyplot as plt
from matplotlib import font_manager

from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.modeling_outputs import ModelOutput
from torch.utils.data import Sampler

print("torch", torch.__version__)

In [ ]:
# ============ 2. 绘图字体与图注语言：统一使用英文 ============
# 云端环境通常没有 CJK 字体，matplotlib 缺字不报错、中文会静默变成方块；
# 字体只挑能显示拉丁字母与德语变音的通用字体，图注固定英文。
_FONT_CANDIDATES = [
    "Helvetica Neue", "Helvetica", "Arial", "Arial Unicode MS",
    "Liberation Sans", "DejaVu Sans", "Verdana", "Tahoma",
]


def _pick_latin_font():
    """挑一个系统里已装、能显示拉丁字母与德语变音的字体。"""
    try:
        installed = {f.name for f in font_manager.fontManager.ttflist}
    except Exception:
        installed = set()
    for _name in _FONT_CANDIDATES:
        if _name in installed:
            return _name
    return "DejaVu Sans"          # matplotlib 自带，100% 可用


_CHOSEN_FONT = _pick_latin_font()
plt.rcParams["font.sans-serif"] = [_CHOSEN_FONT] + [f for f in _FONT_CANDIDATES if f != _CHOSEN_FONT]
plt.rcParams["axes.unicode_minus"] = False   # 用 ASCII 连字符，避免变音符号缺失
print("绘图字体:", _CHOSEN_FONT, "| 图注语言: en (English only)")

# 图上文字的语言：**固定英文**（理由见上）
PLOT_LANG = "en"

_L10N = {
    "zh": {
        "gold": "人工评分", "pred": "模型预测",
        "gold_std": "人工评分的组内标准差（金标准）", "pred_std": "模型预测的组内标准差",
        "m1": "指标1_ρ", "m2": "指标2_ρ", "m3": "指标3_RMSE",
        "rho_up": "Spearman ρ（越大越好）", "rmse_down": "RMSE（越小越好）",
        "figA": "图A  预测 vs 真值", "figB": "图B  评估标准", "figC": "图C  指标2 的可视化",
        "figD": "图D  五次运行的合计指标对照", "figE": "图E  参考对照",
        "not_standard": "非评估标准", "trained": "训练后模型", "random_init": "随机初始化",
        "oracle": "type-level oracle", "single": "单子集训练", "joint": "联合训练（覆盖全部设置）",
        "joint_short": "联合\n(四子集)", "single_short": "单子集\n{short}",
        "m2rho": "指标2 ρ =", "n_targets": "目标数 =",
        "const_std": "预测 std 恒为 0 的目标数 =", "n_settings": "个评测设置",
        "gold_axis": "横轴金标准、纵轴系统输出", "eval_std": "评估标准",
        "short": {},
    },
    "en": {
        "gold": "Gold score", "pred": "Prediction",
        "gold_std": "Gold within-target std", "pred_std": "Predicted within-target std",
        "m1": "Metric1 rho", "m2": "Metric2 rho", "m3": "Metric3 RMSE",
        "rho_up": "Spearman rho (higher is better)", "rmse_down": "RMSE (lower is better)",
        "figA": "Fig A  Prediction vs Gold", "figB": "Fig B  Evaluation standard",
        "figC": "Fig C  Metric 2", "figD": "Fig D  Aggregate metrics across runs",
        "figE": "Fig E  Reference systems",
        "figF": "Fig F  Training loss curves",
        "loss": "Loss", "step": "Step", "epoch": "Epoch",
        "train_loss": "Train loss", "eval_loss": "Eval loss",
        "loss_all_runs": "Training loss of every run",
        "loss_main": "Train vs eval loss (joint run)",
        "untrained": "untrained", "overfit_gap": "train - eval gap at last epoch",
        "not_standard": "reference only", "trained": "Trained", "random_init": "Random init",
        "oracle": "type-level oracle", "single": "Single-subset", "joint": "Joint (all settings)",
        "joint_short": "Joint\n(4 subsets)", "single_short": "single\n{short}",
        "m2rho": "Metric2 rho =", "n_targets": "targets =",
        "const_std": "targets with constant pred std =", "n_settings": "settings",
        "gold_axis": "gold (x) vs system (y)", "eval_std": "Evaluation standard",
        "short": {"英·复合词": "EN-NN", "德·复合词": "DE-NN",
                  "英·小品词": "EN-PV", "德·小品词": "DE-PV"},
    },
}


def T(key, **kw):
    """取图上文字：有中文字体用中文，没有就用英文。"""
    v = _L10N[PLOT_LANG].get(key, key)
    return v.format(**kw) if kw else v


def short_of(subset):
    """子集的短标签（跟随图上语言）。"""
    zh = SUBSET_SPECS[subset]["short"]
    return _L10N[PLOT_LANG]["short"].get(zh, zh)

In [ ]:
# ============ 3. 全局配置：整个 notebook 只有这一处可以改数值 ============
SEED = 42

# ---- 本文件的「默认档位」（生成脚本按产物注入；环境变量优先级更高，上传即可跑对应配置）----
# 各键含义：arm 产物隔离臂名 / modules 结构模块开关 / ce、swap、softcl 损失占比覆盖 /
# plan 运行计划 / v6 结论收敛配方开关 / epochs 轮数覆盖 / data_mode: orig/aug/synth
_DEFAULT_CFG = {'arm': '', 'modules': '', 'ce': '', 'swap': '', 'softcl': '', 'plan': 'joint', 'v6': '', 'epochs': '', 'data_mode': 'orig'}
# ---- 模型清单：只在这里改模型 id ----
# head_mode："regression" = 池化后接 Linear 回归头；"lm_head" = 不加回归头，
#   用模型自带的 LM 头读出分数（把分数当成要生成的数字，在数字 token 分布上取期望）。
# mech：本实验要验证的机制名（只影响损失项开关，不影响数据管线）。
MODEL_REGISTRY = {
    "deberta": dict(
        model_id="microsoft/deberta-v2-xlarge",   # 900M 双向编码器
        pooling="mean",                            # 编码器：掩码均值池化
        attn_impls=("eager",),                     # DeBERTa-v2 不支持 SDPA
        head_mode="regression",                    # 池化 + Linear 回归头
        mech="softcl",                             # 机制：软标签对比学习
    ),
    "qwen": dict(
        model_id="Qwen/Qwen2.5-1.5B-Instruct",    # 1.5B 因果解码器（Instruct 版）
        pooling="last",                            # 解码器：取最后一个真实 token
        attn_impls=("sdpa", "eager"),              # 解码器：优先 SDPA，失败自动回退
        head_mode="lm_head",                       # 不加回归头，用 LM 头读出分数
        mech="lm_head",                            # 机制：无回归头读出
    ),
    "xlmr": dict(
        model_id="FacebookAI/xlm-roberta-large",  # 560M 多语双向编码器（100 语，含德语）
        pooling="mean",                            # 编码器：掩码均值池化（同 deberta 路径）
        attn_impls=("sdpa", "eager"),              # RoBERTa 系支持 SDPA，失败自动回退
        head_mode="regression",                    # 池化 + Linear 回归头（编码器标准读出）
        mech="sweep",                              # 本实验：seed×轮数扫描（训练侧实验，无推理期机制）
    ),
}

# 本文件是 XLM-RoBERTa-large（V9 多 seed × 轮数扫描） 的专属实验；其余（数据处理/超参/指标/绘图）与另一份完全共用，
# 只有「机制项」按 MODEL_KEY 分叉。允许用环境变量指定基线（优先；未设则用文件内默认值）。
MODEL_KEY = (os.environ.get("COMPARTMENT_MODEL_KEY", "").strip() or "xlmr")   # "deberta" | "qwen" | "xlmr"

# 读出方式与机制由 MODEL_KEY 决定
HEAD_MODE = MODEL_REGISTRY[MODEL_KEY]["head_mode"]   # "regression" | "lm_head"
MECH_NAME = MODEL_REGISTRY[MODEL_KEY]["mech"]        # "softcl" | "lm_head"
print(f"读出方式 head_mode = {HEAD_MODE} | 机制 = {MECH_NAME}")

# ---- 运行计划：自动跑完 5 次，不需要手动切换任何开关 ----
# 【为什么写成计划表】把「跑哪几次」显式列出来，读代码的人一眼知道
#   总共会产生几组结果、每组覆盖哪些评测设置、产物落在哪个目录。
#   想少跑几次（云端时间紧张）就直接注释掉对应行，其余代码一行不用动。
#   ("single", 子集名) = 只用该子集训练；("joint", None) = 四个子集拼起来联合训练。
RUN_PLAN = [
    ("single", "en-nn"),
    ("single", "de-nn"),
    ("single", "en-pv"),
    ("single", "de-pv"),
    ("joint", None),
]

# ---- 用环境变量/注入档裁剪运行计划（不改代码）----
#   "all" = 完整 5 次（默认）；"joint" = 只跑联合（提交用模型，覆盖全部 6 个评测设置）；
#   "single_en-nn,joint" = 只跑这两次。名字与 run_name 一致；
#   裁剪只影响「跑哪几次」，不影响切分/损失/指标/绘图任何口径。
_RUN_PLAN_ENV_RAW = os.environ.get("COMPARTMENT_RUN_PLAN", "").strip()
_RUN_PLAN_ENV = _RUN_PLAN_ENV_RAW or _DEFAULT_CFG["plan"]


def _filter_run_plan(plan, spec):
    """按环境变量裁剪运行计划。spec 为空或 "all" ⇒ 原样返回。"""
    if (not spec) or spec.lower() == "all":
        return list(plan)
    want = {x.strip() for x in spec.split(",") if x.strip()}
    out = [(m, s) for (m, s) in plan
           if (m if s is None else f"single_{s}") in want]
    if not out:
        raise ValueError(
            f"COMPARTMENT_RUN_PLAN={spec!r} 没有匹配到任何运行；"
            f"可选名字：{['joint'] + [f'single_{s}' for _m, s in plan if s]}")
    return out


RUN_PLAN = _filter_run_plan(RUN_PLAN, _RUN_PLAN_ENV)

# ---- V9：多 seed × 轮数扫描网格 ----
# 【动机（V8 云端实测，v8/云端实跑_V8结果与结论.md）】① 逐轮合计指标1
# 0.4906 → 0.5303 → 0.5783 到第 3 轮仍在涨 ⇒ 欠训练；② +0.08 的历史新高
# 是单种子单次运行，需复验。单次训练 34 分钟 ⇒ 6 次扫描一次 session 可容纳。
# 【设计】6 次独立联合训练 = 3 seed（42/43/44）× 2 轮数档（4/5）。
# 数据切分种子恒为 42（load_subset/split_by_target 不动）⇒ 全部运行的
# 验证集逐行对齐 —— 跨 seed 对照与多 seed 集成的前提。
# 【预算】V8 云端实测吞吐 87 ms/样本 × 0.8×9,825×(4×3+5×3) 个样本·轮
# × 1.3 开销系数 ≈ 6.7 h ≤ 8 h。
V9_SWEEP = [(42, 4), (42, 5), (43, 4), (43, 5), (44, 4), (44, 5)]
V9_CLOUD_SPS = 0.087   # V8 云端实测（34.2 min 训练 / 23,580 样本轮），预算投影的校准锚点
# 运行级种子/轮数表：名字规范 joint_s{seed}_e{epochs}；未列出的运行一律
# seed 42 / JOINT_EPOCHS。joint_s43 仅为冒烟计划的种子覆盖回归路径保留。
RUN_SEED_OVERRIDE = {"joint_s43": 43}
RUN_EPOCHS_OVERRIDE = {}
RUN_SEED_OVERRIDE.update({f"joint_s{s}_e{e}": s for s, e in V9_SWEEP})
RUN_EPOCHS_OVERRIDE.update({f"joint_s{s}_e{e}": e for s, e in V9_SWEEP})
_grid_names = [f"joint_s{s}_e{e}" for s, e in V9_SWEEP]
# 把过滤后计划里的「联合运行」替换成网格（单子集运行保留在前；
# 用户显式把计划裁到不含 joint 时不展开，尊重 COMPARTMENT_RUN_PLAN）。
if any(s is None for _m, s in RUN_PLAN):
    RUN_PLAN = [(m, s) for (m, s) in RUN_PLAN if s is not None] \
              + [(n, None) for n in _grid_names]
print(f"V9 扫描网格（seed × 轮数）：{V9_SWEEP} → {len(_grid_names)} 次联合训练")

# ---- 臂名：把一次运行的产物隔离到 outputs/arms/<臂名>/，多档运行互不覆盖；默认空 = 不隔离 ----
ARM_NAME = (os.environ.get("COMPARTMENT_ARM_NAME", "").strip()
           or _DEFAULT_CFG["arm"])


def resolve_arm_output_root(root, arm_name):
    """把输出根目录按臂名隔离；臂名为空 ⇒ 原样返回。"""
    if not (arm_name or "").strip():
        return root
    return root / "arms" / arm_name.strip()

# ---- 子集登记表：每个子集的「目标键 / 标签列 / 语言 / 任务」----
SUBSET_SPECS = {
    "en-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], stds=["ModStd", "HeadStd"], lang="en", task="nn", short="英·复合词"),
    "de-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], stds=["ModStd", "HeadStd"], lang="de", task="nn", short="德·复合词"),
    "en-pv": dict(key="ParticleVerb", cols=["Avg"],             stds=["Std"],           lang="en", task="pv", short="英·小品词"),
    "de-pv": dict(key="ParticleVerb", cols=["Avg"],             stds=["Std"],           lang="de", task="pv", short="德·小品词"),
}
SUBSET_ORDER = ["en-nn", "de-nn", "en-pv", "de-pv"]   # 固定顺序，保证多次运行之间编号一致

# 统一输出维度：复合词 2 列（修饰语/中心语），小品词动词只有 1 列有效。
# 【为什么统一成 2 列而不是让 pv 用 1 列】这样四种运行模式、四个子集
#   共用**完全相同**的模型结构、损失与指标代码；模型输出的第 2 个位置在 pv 上
#   用 loss_mask 屏蔽掉即可。少一个特例，就少一处能出错的地方。
N_OUTPUTS = 2

# ---- 训练超参：所有运行共用同一套数值 ----
MAX_LENGTH   = 160
# TARGET_EFFECTIVE_BATCH 是有效 batch（micro-batch × 梯度累积），不同显存的卡上
# 分解方式可以不同，但乘积必须相等，否则梯度尺度与对照口径都变了。
# 两个机制实验各自独立选口径：DeBERTa 16；Qwen 8 —— Qwen 只能用 accum=1，
# 因为梯度累积 ≥2 时绑定 embedding 的稠密梯度会在反向同时存在两份
#（151936×1536×4 B ≈ 890 MiB），t3 档在单张 T4 上装不下。
EFFECTIVE_BATCH_BY_MODEL = {"deberta": 16, "qwen": 8, "xlmr": 16}
TARGET_EFFECTIVE_BATCH = EFFECTIVE_BATCH_BY_MODEL[MODEL_KEY]
BATCH_SIZE   = 8
# 梯度累积由有效 batch 推导，不写死，避免「有效 batch 改了、累积没跟着改」的口径漂移。
GRAD_ACCUM   = max(1, int(round(TARGET_EFFECTIVE_BATCH / BATCH_SIZE)))
PROBE_MEMORY = True      # 云端 GPU 规格不一，先实测能跑多大 batch 再开训
LR           = 2e-5
# ---- 轮数：单子集与联合运行同为 3 轮 ----
EPOCHS       = 3
JOINT_EPOCHS = 3
# 轮数覆盖：环境变量 COMPARTMENT_EPOCHS 或注入档 epochs 优先；空 = 不覆盖；SMOKE 仍强制 1 轮。
_EP_OVERRIDE = (os.environ.get("COMPARTMENT_EPOCHS", "").strip()
                or str(_DEFAULT_CFG.get("epochs", "") or ""))
if _EP_OVERRIDE:
    EPOCHS = int(float(_EP_OVERRIDE))
    JOINT_EPOCHS = int(float(_EP_OVERRIDE))
    print(f"★ 轮数覆盖生效：EPOCHS = JOINT_EPOCHS = {EPOCHS}")
VAL_RATIO    = 0.2
WEIGHT_DECAY = 0.01
# warmup ratio 0.1：语言模型微调的常用起步保护（DeBERTa 官方 recipe 同值）；
# 步数按实际 max_steps 计算并有断言防口径漂移。
WARMUP_RATIO = 0.1
# Huber δ=1.0：单条例句标签噪声 sd≈0.36–0.50 分（标签=10 个评分的均值），
# δ=1 让 |err|>1 走线性段（梯度恒为 ±1），对标注噪声与离群更稳健；主指标是秩相关，
# 抗噪的估计器更契合。
HUBER_DELTA  = 1.0

# ---- 优化器与显存相关的开关 ----
# 全参微调的「固定开销」与 batch 无关（权重 4 + 梯度 4 + AdamW 状态 8 = 16 B/param），
# 1.5B 模型 fp32 AdamW ≈ 23 GiB；换 8bit 优化器（状态 2 B/param）是唯一能降固定开销的办法。
# USE_8BIT_OPT="auto" 只取决于环境有无 bitsandbytes（同机结果可复现）；
# 不做「显存不够就自动切」——那会让前后两次运行的数值不可比，不够时明确报错。
USE_8BIT_OPT   = "auto"   # "auto" | True | False
USE_GRAD_CKPT  = False    # True = 开 gradient checkpointing（激活降到约 1/N，慢约 30%）

# ---- 训练档位（tier）----
# 档位 = 「优化器 / 精度 / 训练范围」的预设组合。写死 t3（全参微调 · Adafactor · 关 AMP）：
# 训练范围可复现、不会在显存紧张时悄悄降档冻住一半主干（那等于换了一个实验）；
# t3 真跑不通就报错并打印诊断，而不是自动降档。
# 单卡装得下的原因：Qwen 有效 batch=8（accum=1）⇒ 绑定矩阵的稠密梯度全程只占一份。
FIT_MODE       = "t3"      # 写死 t3（全参微调，禁止自动降档）；"auto" = 按显存自动选档
FIT_REF_PARAMS = 1.55e9      # 选档参考参数量（取两条基线中较大者；仅 auto 模式用）
FIT_SAFETY     = 0.90        # 显存预算安全系数（留 10% 给碎片与 CUDA context）
# ---- 显存探测的「余量」----
# 探针通过的条件 = 「峰值 + 余量 ≤ 卡容量」；余量不足时改用更小 micro-batch，
# 全都不满足则保留档位并告警（不降档，正式训练里还有 OOM 自动重试兜底）。
# 余量 0.8 GiB 只需覆盖碎片与 CUDA context：探针与正式训练同模型结构、同梯度累积；
# 设得太大反而会把 micro-batch 逼小、白白拖长训练。
FIT_PROBE_RESERVE_GB = 0.8   # 探测优先采用的显存余量（GiB）；不满足时保留档位并告警

# ---- 用哪张卡：两条基线各占一张并行（同型号卡对数值无影响，只影响调度）----
# CUDA 显存不能跨卡合并；DDP 不省显存且会让分组采样取到重复样本；
# 真正分片显存的 ZeRO-3/FSDP 需要多进程启动，单进程 notebook 跑不了。
GPU_INDEX      = int(os.environ.get("COMPARTMENT_GPU", "0"))

# ---- 训练时长预算（Kaggle 单次运行上限 12 小时）----
TIME_BUDGET_H = 8.0        # 投影超过这个小时数就报警（留 1 小时给评估与出图）

# ---- 损失配方：三个官方指标各有对应项，按子集/按模型差异化 ----
# 每个标签是 10 个 0–5 整数评分的均值 ⇒ 单条例句标签噪声 sd≈0.36–0.50 分；
# 不做「组内 std 等值匹配」（等于要求模型复现标注噪声），方向是：
#   指标 1（主）  ← order 组内排序（只用符号，抗噪）+ huber 绝对值锚
#   指标 2（次要）← guard 反收缩 hinge（保「组内 std 非 0」的 nan 红线）+ xrank 跨目标 std 序
#   指标 3（补充）← mse 显式二次项
# 官方指标 2 的现实天花板（蒙特卡洛）：nn 0.00–0.24（不可追，只防塌陷）、
# pv 0.26–0.57（可达，适度优化）⇒ 权重按子集差异化。
# 权重是「目标占比」而非原始权重：LossBalancer 按各项实时量级在线标定（见 cells_b），
# 避免「猜权重」（太小没效果、太大压过主项）。
LOSS_PROFILE_BY_MODEL = {
    # ============ DeBERTa：机制 = 软标签对比学习 L_softcl ============
    "deberta": {
        # 各项 = 在总损失中的目标占比（0 = 关闭）；kappa = 反收缩 hinge 的下界系数
        #（预测组内 std 不得低于「人工 std × kappa」）。
        # softcl：把「标签相近」的句子在表征空间拉近、「标签相远」的推开，
        # 相似度目标由标签距离连续决定 ⇒ 用标签距离做成对监督，比纯符号（order）
        # 与逐样本绝对误差（huber/mse，被标注噪声主导）更稳。
        # 只在 pv 开：nn 的组内标签方差 93–129% 是标注噪声（做对比=拟合噪声），
        # pv 的组内差异约一半是真实语境效应且指标 2 天花板可达。
        "en-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "de-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "en-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.06, ce=0.00),
        "de-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.06, ce=0.00),
    },
    # ============ Qwen：机制 = 不加回归头、用 LM 头自己读出分数 ============
    "qwen": {
        # ce：数字 token 上的软目标交叉熵。LM 头给出「下一个 token 是 0–5 的分布」，
        # 训练目标两层：期望值层（huber）让 Σk·p_k 等于标签；分布层（ce）让分布
        # 平滑地落在标签附近（标签 3.5 ⇒ 「3 与 4 的混合」，而不是尖锐的 one-hot）。
        # ce 在四个子集都开：它是「让 LM 头学会输出分数」本身的要求，与子集无关。
        "en-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.10),
        "de-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.10),
        "en-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.10),
        "de-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.10),
    },
    # ============ XLM-R（V8/V9）：编码器标准配方，机制实验全关 ============
    # 公共底座（order/mse/guard/xrank/kappa）与 deberta/qwen 逐字同值；
    # softcl 与 ce 都关 —— 新机制全部在训练编排（V9 扫描）而非损失配方里。
    "xlmr": {
        "en-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "de-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "en-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.00),
        "de-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.00),
    },
}
# 本实验实际使用的配置（按 MODEL_KEY 选；两个 notebook 用同一份真源，只是 MODEL_KEY 不同）
SUBSET_LOSS_PROFILE = LOSS_PROFILE_BY_MODEL[MODEL_KEY]

# kappa = 「组内 std 不得低于人工 std 的多少倍」。人工 std 被标注噪声膨胀
# 2.7%–13.6% ⇒ 正常训练的模型 std ≈ 人工 std × 0.86–0.97；
# nn κ=0.15 / pv κ=0.50 都低于该值 ⇒ 平时不触发，只在真塌陷（std→0）时发力（保险丝）。
AUTO_BALANCE = True   # True = 按上面的占比在线标定权重；False = 用固定权重
ORDER_TAU    = 1.0    # 组内排序的 margin 温度（softplus(-s·Δ/τ)）
XRANK_MARGIN = 0.05   # 跨目标比较只取「人工 std 差异 > 此值」的目标对
XRANK_TAU    = 2.0    # 跨目标 std 序比较的温度
BALANCE_EMA_BETA  = 0.99  # 量级 EMA 的衰减系数
# ---- L_tierank：跨目标分数序（标签方差 68–88% 在目标之间，既有损失无一覆盖这一层）----
# 与 xrank 同构：batch 内凑不出跨目标对 ⇒ 与历史目标比（实现见 cells_b.tierank_parts）。
TIERANK_MARGIN = 0.3  # 只比较「目标均值差 > 此值」的对（目标级噪声 ≈ 单句噪声/√n ⇒ 0.3 稳）
TIERANK_TAU    = 1.0  # softplus 温度（同 ORDER_TAU 量纲）

# ---- 机制结构参数 ----
# softcl：目标相似度 w_ij = cos(π·|y_i-y_j| / SPAN)（带符号：差 0 分→+1、2.5 分→0、
# 5 分→−1，才能区分「正交」与「重合」）；对比在投影空间做，不污染主干表征。
SOFTCL_LABEL_SPAN = 5.0
SOFTCL_PROJ_DIM  = 128
# ce：目标分布的高斯带宽（标签 3.5 在 σ=0.8 下 ⇒ 3 与 4 各约 0.40、2 与 5 各约 0.08）。
CE_SOFT_SIGMA = 0.8
# lm_head 读出的「提示锚点」：每个标签列一个锚点，文本结尾接上它，
# 模型在锚点末 token 处的下一个 token 分布就是该列分数分布
#（Qwen2.5 的 "0"–"5" 均为单 token；锚点以空格结尾 ⇒ 下一个 token 就是数字本身）。
# 两列各一个锚点：修饰语分与中心语分是两个不同的值，同一锚点读两次只会得到同一个数。
READOUT_CUES = ["\nModifier: ", "\nHead: "]
DIGIT_VALUES = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

# ---- 三个模型结构模块（m1/m2/m4）的开关：逐个独立，便于归因每个模块的收益 ----
# 可用 COMPARTMENT_V5_MODULES="m1" / "m1,m2" / "m1,m2,m4"（默认全开）指定；
# 实际生效的开关写进 outputs/summary/run_settings_*.json（口径不靠记忆，靠文件）。
# 真值由 configure_experiment() 在运行时设定（同一 notebook 顺序跑多档时必须能重算）；
# 下面的初值只是占位。
V5_MODULES = {"m1": False, "m2": False, "m4": False}
M1_DIST_HEAD = False                # 模块 1：序数-分布头
M2_DUAL_STREAM = False              # 模块 2：类型-语境双流 + 语境敏感度门
M4_ROLE_EMB = False                 # 模块 4：角色嵌入 + 成分交换一致性
# 模块 1：M1_CE_WEIGHT = 分布头软标签交叉熵的目标占比；M1_CLASSES = 6（分数量程 0–5）。
M1_CE_WEIGHT = 0.10
M1_CLASSES = 6
# 模块 2：M2_TYPE_MAX_LEN = 类型流（只编码 MWE）的最大长度——t3 档显存余量小，
# 第二次前向必须用短序列。M2_GATE_SUPERVISION 选门的监督方式：
#   "none"  = 自由门（默认），g 由主损失自己学，「g 在四个子集是否分化」是一个真诊断；
#   "share" = 用「语境有效度 share」（组内观测方差去掉标注噪声后的比例）做逐目标回归监督，
#             只用训练集算出（推理不需要）。默认自由门：带监督会把诊断变成同义反复。
M2_TYPE_MAX_LEN = 24
M2_GATE_SUPERVISION = "none"
M2_GATE_WEIGHT = 0.05
# 模块 4：M4_SWAP_MAX_ROWS = 每 batch 最多多少行构造「成分交换视图」——交换视图需要
# 第三次前向，必须封顶防 OOM；抽中是随机的 ⇒ 一个 epoch 覆盖所有可构造的行。
# M4_SWAP_WEIGHT = 交换一致性的目标占比（正则、非主目标，太大=强制两成分同分）。
M4_SWAP_MAX_ROWS = 4
M4_SWAP_WEIGHT = 0.05

def apply_recipe_overrides(profile, ce=None, swap=None, softcl=None):
    """把显式指定的损失占比覆盖写进按子集的配方表；空串/None ⇒ 不覆盖。
    必须在模块开关推导之后调用，否则模块推导会把显式覆盖顶回去。"""
    for _sn in SUBSET_ORDER:
        if ce:
            profile[_sn]["ce"] = float(ce)
        if swap:
            profile[_sn]["swap"] = float(swap)
        if softcl:
            profile[_sn]["softcl"] = float(softcl)
    return profile


# ---- 结论收敛配方（V6 提交配置）----
# nn：order→0（组内差异 93–129% 是标注噪声，组内排序=拟合噪声）+ tierank 0.10；
# pv：guard→0.10（顶住组内 std 塌陷）+ tierank 0.05；全部子集 swap=0、softcl=0（无收益）。
# 做成「开关 + 函数」而不直接改配方表：配方表被多组产物共享，直接改会静默换掉历史对照。
def apply_v6_recipe(profile):
    for _sn in SUBSET_ORDER:
        _task = "nn" if SUBSET_SPECS[_sn]["task"] == "nn" else "pv"
        if _task == "nn":
            profile[_sn]["order"] = 0.0
            profile[_sn]["tierank"] = 0.10
        else:
            profile[_sn]["guard"] = 0.10
            profile[_sn]["tierank"] = 0.05
        profile[_sn]["swap"] = 0.0
        profile[_sn]["softcl"] = 0.0
    return profile


V6_RECIPE_ENABLED = bool(os.environ.get("COMPARTMENT_V6_RECIPE", "").strip()
                         or str(_DEFAULT_CFG.get("v6", "") or ""))


def configure_experiment(model_key, modules_spec, ce_override="", swap_override="",
                         softcl_override=""):
    """可重入地设定「本条基线 + 本档位」的结构开关与损失配方。

    同一 notebook 顺序跑多档/多条基线时，模块开关、读出方式、有效 batch、
    配方都不同 ⇒ 派生量必须能在运行时重算，否则「换了档位但结构没换」会静默失真。

    覆盖顺序（不得改）：按基线取默认配方 → 模块开关推导新增项 → 结论收敛配方 →
    显式覆盖（ce/swap/softcl）在最后，保证档位声明的配方一定生效。

    返回本档位的口径快照（落盘进 run_settings_*.json）。
    """
    global MODEL_KEY, HEAD_MODE, MECH_NAME, TARGET_EFFECTIVE_BATCH
    global V5_MODULES, M1_DIST_HEAD, M2_DUAL_STREAM, M4_ROLE_EMB
    global SUBSET_LOSS_PROFILE

    MODEL_KEY = model_key
    HEAD_MODE = MODEL_REGISTRY[MODEL_KEY]["head_mode"]
    MECH_NAME = MODEL_REGISTRY[MODEL_KEY]["mech"]
    TARGET_EFFECTIVE_BATCH = EFFECTIVE_BATCH_BY_MODEL[MODEL_KEY]

    _mods = {x.strip().lower() for x in str(modules_spec).split(",") if x.strip()}
    V5_MODULES = {k: (k in _mods) for k in ("m1", "m2", "m4")}
    M1_DIST_HEAD = V5_MODULES["m1"]
    M2_DUAL_STREAM = V5_MODULES["m2"]
    M4_ROLE_EMB = V5_MODULES["m4"]

    # ★ **深拷贝**：LOSS_PROFILE_BY_MODEL 里的表是共享引用，
    #   若原地改它，下一档 / 下一条基线会拿到被上一档改过的配方 ⇒ 口径漂移。
    SUBSET_LOSS_PROFILE = {k: dict(v) for k, v in LOSS_PROFILE_BY_MODEL[MODEL_KEY].items()}

    # ---- ① 模块开关推导出的新增配方项 ----
    # 【模块 1】分布头必须有分布层监督，否则模型可以把期望值凑对、分布退化成任意形状。
    #   只在「该项当前为 0」时补 ⇒ 两条基线的 CE 权重口径一致（0.10）。
    if M1_DIST_HEAD:
        for _sn in SUBSET_ORDER:
            if float(SUBSET_LOSS_PROFILE[_sn].get("ce", 0.0)) <= 0.0:
                SUBSET_LOSS_PROFILE[_sn]["ce"] = M1_CE_WEIGHT
    # 【模块 2】门控监督项（可选档）：默认 "none" ⇒ 权重保持 0 ⇒ 这一项不产生梯度，
    #   门 g 只由主损失驱动（自由门）⇒ 「g 是否分化」是一个**真诊断**。
    if M2_DUAL_STREAM and M2_GATE_SUPERVISION == "share":
        for _sn in SUBSET_ORDER:
            SUBSET_LOSS_PROFILE[_sn]["gate"] = M2_GATE_WEIGHT
    # 【模块 4】成分交换一致性：权重写上，但实际是否生效由「这一批有没有可构造的行」决定
    #   （de-nn 是单词形复合词、pv 只有一列 ⇒ 那两个子集上分母为 0 ⇒ 整项退出）。
    if M4_ROLE_EMB:
        for _sn in SUBSET_ORDER:
            SUBSET_LOSS_PROFILE[_sn]["swap"] = M4_SWAP_WEIGHT
    # ---- 结论收敛配方（在模块推导之后、显式覆盖之前 ⇒ 两头都顶不掉它）----
    if V6_RECIPE_ENABLED:
        apply_v6_recipe(SUBSET_LOSS_PROFILE)

    # ---- ② 显式覆盖（最后一道）----
    apply_recipe_overrides(SUBSET_LOSS_PROFILE, ce=ce_override,
                           swap=swap_override, softcl=softcl_override)

    print(f"★ 配置：基线={MODEL_KEY} 模块={V5_MODULES} "
          f"| ce={ {s: SUBSET_LOSS_PROFILE[s].get('ce') for s in SUBSET_ORDER} } "
          f"| swap={ {s: SUBSET_LOSS_PROFILE[s].get('swap') for s in SUBSET_ORDER} } "
          f"| 有效 batch={TARGET_EFFECTIVE_BATCH}")
    return dict(v5_modules=dict(V5_MODULES),
                loss_profile={k: dict(v) for k, v in SUBSET_LOSS_PROFILE.items()})


# ---- 初始化：按「本文件的默认档位」设定一次 ----
_V5_MOD_ENV = (os.environ.get("COMPARTMENT_V5_MODULES", "").strip()
              or _DEFAULT_CFG["modules"])
_CE_OVERRIDE = (os.environ.get("COMPARTMENT_CE_WEIGHT", "").strip()
                or _DEFAULT_CFG["ce"])
_SWAP_OVERRIDE = (os.environ.get("COMPARTMENT_SWAP_WEIGHT", "").strip()
                  or _DEFAULT_CFG["swap"])
_SOFTCL_OVERRIDE = (os.environ.get("COMPARTMENT_SOFTCL_WEIGHT", "").strip()
                    or _DEFAULT_CFG["softcl"])
configure_experiment(MODEL_KEY, _V5_MOD_ENV, ce_override=_CE_OVERRIDE,
                     swap_override=_SWAP_OVERRIDE, softcl_override=_SOFTCL_OVERRIDE)
print("结构模块开关:", V5_MODULES,
      "| 分布头 CE 占比:", {s: SUBSET_LOSS_PROFILE[s].get("ce") for s in SUBSET_ORDER},
      "| 门控占比:", {s: SUBSET_LOSS_PROFILE[s].get("gate", 0.0) for s in SUBSET_ORDER},
      "| swap 占比:", {s: SUBSET_LOSS_PROFILE[s].get("swap", 0.0) for s in SUBSET_ORDER},
      "| softcl 占比:", {s: SUBSET_LOSS_PROFILE[s].get("softcl", 0.0) for s in SUBSET_ORDER},
      "| tierank 占比:", {s: SUBSET_LOSS_PROFILE[s].get("tierank", 0.0) for s in SUBSET_ORDER})
if V6_RECIPE_ENABLED:
    print("★ 结论收敛配方已启用：nn order=0 + tierank=0.10"
          "｜pv guard=0.10 + tierank=0.05｜swap=softcl=0")
if _CE_OVERRIDE or _SWAP_OVERRIDE or _SOFTCL_OVERRIDE:
    print(f"★ 损失配方覆盖生效：ce={_CE_OVERRIDE or '（未覆盖）'} "
          f"swap={_SWAP_OVERRIDE or '（未覆盖）'} softcl={_SOFTCL_OVERRIDE or '（未覆盖）'}")
# 训练期每多少步验一次 loss 是否有限（0 = 不检查）。
# 无 AMP 的档位没有 GradScaler 兜底，NaN 梯度会被直接写进权重且训练不报错；
# 每 50 步查一次，把「几小时后才发现」变成「50 步内就停」。
LOSS_FINITE_CHECK_EVERY = 50

# ---- 模型选择指标（best checkpoint 用哪个指标挑）----
# 综合分在 z 分数空间等权求和：三项各除以自己的运行间噪声 sd，
# 在「可检测性」上等权（指标1 与指标2 的噪声 sd 差 2.4 倍，等权相加会让
# 噪声更大的一项主导选择）。指标 2 为 nan 按 −1 计 ⇒ 大负 z 分 ⇒ 必然选不中。
BEST_METRIC_KEY = "合计|_综合分"
Z_M1 = 0.023   # 指标 1（Spearman）的运行间噪声 sd
Z_M2 = 0.055   # 指标 2（组内 std 的 Spearman）的运行间噪声 sd
Z_M3 = 0.046   # RMSE 的运行间噪声 sd

# ---- 其它开关 ----
GROUPED_BATCHING = True   # 让同一目标的例句尽量落在同一个 batch
# 冒烟模式用环境变量驱动（父进程与子进程共用同一个变量），
# 不用「文本替换 SMOKE=False」那种脆弱机制。
SMOKE = (os.environ.get("COMPARTMENT_SMOKE", "0").strip() == "1")   # True = 冒烟模式（2 层随机模型，不下载权重）
SMOKE_N_ROWS = 400        # 冒烟模式每个子集只取这么多行

# ---- 保存哪些运行的模型权重 ----
# 【为什么默认只存联合运行】900M 模型一个 final 就 ~3.6GB，5 次运行全存 = 18GB，
#   会直接撑爆 Kaggle 磁盘。联合运行覆盖全部 6 个评测设置，也是提交要用的那一个。
#   想把单子集的权重也留下来做后续分析，把对应 run_name 加进来即可（注意磁盘）。
# V9：6 个联合运行的 final 各 ≈2.2 GB，全存会撑爆 Kaggle 磁盘；
# 扫描只消费逐例句预测（ALL_RUNS + 导出），不需要任何 final 权重。
SAVE_FINAL_FOR = set()

# ---- 冒烟模式：把规模压到秒级，代码路径与正式运行完全一致（只改规模与跑几次）----
if SMOKE:
    MAX_LENGTH = 64
    EPOCHS = 1
    JOINT_EPOCHS = 1
    BATCH_SIZE = 4
    # 冒烟固定跑 3 次（en-nn / de-nn / joint）：de-nn 是「全局子集编号在单子集运行
    # 里越界」那条回归路径的唯一触发点（子集编号是全局的、子集名清单是本次运行的）。
    # V9：冒烟跑 3 个联合运行 —— joint（裸）/ joint_s43（只种子覆盖）/
    # joint_s44_e4（种子+轮数双覆盖，网格命名路径本机只有这里能测到）
    RUN_PLAN = [("single", "en-nn"), ("single", "de-nn"),
                ("joint", None), ("joint_s43", None), ("joint_s44_e4", None)]
    # 冒烟也尊重显式指定的 RUN_PLAN（只看环境变量原始值，不含默认档），
    # 否则「裁剪运行计划」这条路径在本机永远测不到。
    if _RUN_PLAN_ENV_RAW and _RUN_PLAN_ENV_RAW.lower() != "all":
        RUN_PLAN = _filter_run_plan(RUN_PLAN, _RUN_PLAN_ENV_RAW)

print(json.dumps(dict(MODEL_KEY=MODEL_KEY, HEAD_MODE=HEAD_MODE, MECH_NAME=MECH_NAME,
                      MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM,
                      TARGET_EFFECTIVE_BATCH=TARGET_EFFECTIVE_BATCH,
                      LR=LR, EPOCHS=EPOCHS, JOINT_EPOCHS=JOINT_EPOCHS,
                      WARMUP_RATIO=WARMUP_RATIO, HUBER_DELTA=HUBER_DELTA,
                      SUBSET_LOSS_PROFILE=SUBSET_LOSS_PROFILE, AUTO_BALANCE=AUTO_BALANCE,
                      ORDER_TAU=ORDER_TAU, XRANK_MARGIN=XRANK_MARGIN, XRANK_TAU=XRANK_TAU,
                      BALANCE_EMA_BETA=BALANCE_EMA_BETA,
                      SOFTCL_LABEL_SPAN=SOFTCL_LABEL_SPAN,
                      SOFTCL_PROJ_DIM=SOFTCL_PROJ_DIM, CE_SOFT_SIGMA=CE_SOFT_SIGMA,
                      READOUT_CUES=READOUT_CUES,
                      V5_MODULES=V5_MODULES, M1_CE_WEIGHT=M1_CE_WEIGHT,
                      M1_CLASSES=M1_CLASSES,
                      M2_TYPE_MAX_LEN=M2_TYPE_MAX_LEN,
                      M2_GATE_SUPERVISION=M2_GATE_SUPERVISION,
                      M2_GATE_WEIGHT=M2_GATE_WEIGHT,
                      M4_SWAP_MAX_ROWS=M4_SWAP_MAX_ROWS, M4_SWAP_WEIGHT=M4_SWAP_WEIGHT,
                      Z_M1=Z_M1, Z_M2=Z_M2, Z_M3=Z_M3,
                      LOSS_FINITE_CHECK_EVERY=LOSS_FINITE_CHECK_EVERY,
                      BEST_METRIC_KEY=BEST_METRIC_KEY,
                      SMOKE=SMOKE, RUN_PLAN=[list(x) for x in RUN_PLAN]),
                 ensure_ascii=False, indent=1))

In [ ]:
# ============ 4. 硬件探测与精度自动降级 ============
def detect_device():
    """返回 (device, use_bf16, use_fp16, use_tf32)，三档自动降级不写死。

    bf16 判据以算力为准：torch.cuda.is_bf16_supported() 会把「软件模拟」也算支持，
    T4 等老卡照样返回 True ⇒ 只有 major ≥ 8 才用原生 bf16，major ≥ 7 用 fp16。"""
    if torch.cuda.is_available():
        _n_gpu = torch.cuda.device_count()
        _gi = GPU_INDEX % max(1, _n_gpu)
        # 显式选卡：多卡机器上两个进程必须各占一张，否则互相挤兑显存；
        # set_device 必须在任何显存分配之前调用。
        torch.cuda.set_device(_gi)
        _cap = torch.cuda.get_device_capability(_gi)
        _native_bf16 = (_cap[0] >= 8) and torch.cuda.is_bf16_supported()
        _fp16 = (not _native_bf16) and (_cap[0] >= 7)
        return "cuda", _native_bf16, _fp16, True
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        # MPS 的 bf16 在部分算子上会退化成 fp32，速度反而不升；统一走 fp32 更稳
        return "mps", False, False, False
    return "cpu", False, False, False


DEVICE, USE_BF16, USE_FP16, USE_TF32 = detect_device()
# 把「硬件本来支持什么」单独存一份：训练档位可能为省显存关掉混合精度，
# 档位需要靠它来在「开 AMP / 关 AMP」之间切换。
HW_BF16, HW_FP16 = USE_BF16, USE_FP16
# num_workers=0 时必须同步降低 persistent_workers / multiprocessing_context / prefetch_factor，
# 否则 DataLoader 抛「multiprocessing_context can only be used with multi-process loading」。
NUM_WORKERS = 0 if platform.system() == "Darwin" else max(1, (os.cpu_count() or 2) // 2)
DATALOADER_PERSISTENT = NUM_WORKERS > 0

# TF32 只在 CUDA 上有意义，CPU/MPS 上开它没有收益
if USE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

set_seed(SEED)
print({
    "device": DEVICE, "bf16": USE_BF16, "fp16": USE_FP16, "tf32": USE_TF32,
    "workers": NUM_WORKERS, "torch": torch.__version__, "platform": platform.system(),
    "python": platform.python_version(),
})
# 打印 GPU 清单：卡数/型号/容量/算力，用于判断单卡是否够用、是否分卡并行。
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor} | "
              f"原生bf16={_p.major >= 8} | fp16={_p.major >= 7}")
    _total_cards = sum(torch.cuda.get_device_properties(_i).total_memory
                        for _i in range(torch.cuda.device_count())) / (1024 ** 3)
    print(f"物理卡：{torch.cuda.device_count()} 张可见（合计 {_total_cards:.1f} GiB）；"
          f"本 notebook 绑定物理 GPU{GPU_INDEX}，在进程内编号为 cuda:0")
    print("→ 单卡模式：Trainer 看到 n_gpu=1 → **不会**自动套 nn.DataParallel"
          "（DataParallel 的反向传播会额外复制一整份梯度，Qwen-1.5B 要多花 5.75 GiB）")

## Step2 加载数据集

**这一节要解决什么**：用最少的假设把原始 TSV 变成规整的 DataFrame，
并且**在载入这一步就把会泄漏答案的东西扔掉**。

**为什么载入阶段就要处理反泄漏**：本项目实测发现，
同一个 MWE 的各条例句在文件里是**按 ContextID 顺序连续排列**的，
而 ContextID 又与该目标的平均分高度相关（组内 Spearman 达 0.985）。
换句话说，**「这条例句在文件里的位置」本身就带着答案**。
如果先建特征再打乱，任何从行号派生的东西（下标、位次、窗口邻域）都会变成作弊通道。
所以顺序必须是：**先删 ContextID → 再物理打乱**。

**要保留什么**：`Compound` / `ParticleVerb`（MWE 本身）、`Mod`/`Head`/`Base`/`Particle`（成分）、
`ModAvg`/`HeadAvg`/`Avg`（标签）、`Context`（句子）。

In [ ]:
# ============ 数据目录解析：按「环境变量 → Kaggle → 仓库」顺序探测，不改代码即可换数据源 ============
CLOUD_TRAIN_DIR = "/kaggle/input/datasets/kintsugi0v0/compartment-data/train"
# Kaggle 数据集 compartment-data-v2 的布局：train_data_v2/ 根目录是 4 个原始 TSV，合成增广数据在 filtered/ 子目录
CLOUD_V2_DATA_ROOT = "/kaggle/input/datasets/kintsugi0v0/compartment-data-v2/train_data_v2"
CLOUD_AUG_DIR = CLOUD_V2_DATA_ROOT + "/filtered"


def resolve_data_root(start: Path) -> Path:
    """返回第一个同时含 4 个 *-train.tsv 的目录（原始训练数据）。

    兜底用递归 glob 全盘找 en-nn-train.tsv，但排除 filtered/ 目录下的命中——
    那是合成增广数据（与原始文件同名），不能被当成原始数据，否则增广模式会把
    合成数据拼两遍。
    """
    import glob as _glob
    cands = []
    env = os.environ.get("COMPARTMENT_TRAIN_DIR")
    if env:
        cands.append(Path(env).expanduser())
    cands.append(Path(CLOUD_V2_DATA_ROOT))
    cands.append(Path(CLOUD_TRAIN_DIR))
    root_env = os.environ.get("COMPARTMENT_ROOT")
    if root_env:
        cands.append(Path(root_env).expanduser() / "train")
    p = start.resolve()
    for cand in [p] + list(p.parents):
        if (cand / "train" / "en-nn-train.tsv").exists():
            cands.append(cand / "train")
            break
    for _hit in sorted(map(Path, _glob.glob("/kaggle/input/**/en-nn-train.tsv", recursive=True))):
        if "filtered" not in _hit.parts:
            cands.append(_hit.parent)
    _need = [f"{_n}-train.tsv" for _n in SUBSET_ORDER]
    for c in dict.fromkeys(cands):
        if all((c / _f).exists() for _f in _need):
            assert "filtered" not in c.parts, f"原始数据目录不能是 filtered/：{c}"
            return c
    raise FileNotFoundError(
        "找不到原始训练数据（需同时含 " + " / ".join(_need) + "）。已依次尝试：\n  "
        + "\n  ".join(str(c) for c in cands)
        + "\n请检查 Kaggle 数据集挂载（原始 TSV 应在 .../train_data_v2/ 根目录下），"
          "或设置环境变量 COMPARTMENT_TRAIN_DIR 指向该目录。")


WORK_DIR = Path.cwd()
DATA_ROOT = resolve_data_root(WORK_DIR)
# 输出目录可用环境变量覆盖；Kaggle 上默认落在 /kaggle/working（可写）
OUTPUT_ROOT = Path(os.environ.get("COMPARTMENT_OUTPUT", str(WORK_DIR / "outputs")))
# 有臂名时把整棵产物树放进 <OUTPUT_ROOT>/arms/<臂名>/ 下，多档运行互不覆盖。
OUTPUT_ROOT = resolve_arm_output_root(OUTPUT_ROOT, ARM_NAME)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("WORK_DIR  =", WORK_DIR)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
if ARM_NAME:
    print(f"★ 消融臂名 = {ARM_NAME}（产物隔离在 outputs/arms/{ARM_NAME}/ 下）")

# ---- 合成数据增广：aug = 原始 + train_data_v2/filtered（过滤版合成数据），synth = 只用合成 ----
# 合成行与原始行在 load_subset 里合并成同一目标组；找不到合成目录 ⇒ 报错（不静默降级）。
DATA_MODE = (os.environ.get("COMPARTMENT_DATA_MODE", "").strip()
             or str(_DEFAULT_CFG.get("data_mode", "") or "orig")).lower()
if DATA_MODE not in ("orig", "aug", "synth"):
    raise ValueError(f"DATA_MODE={DATA_MODE!r} 不合法；可选 orig / aug / synth")

# ---- 数据身份断言：官方人类标注 TSV 的 md5 指纹 ----
# 【为什么必须有】曾发生「云端挂载的原始数据目录实为全量合成数据」的事故：
#   训练/验证一行人类数据都没进，全部指标口径静默失效。文件名相同不代表身份相同，
#   必须按内容指纹校验；不匹配时当场报错并列出实际加载的文件。
# 指纹 = 仓库 train/ 四个官方 TSV 的 md5（与云端事故复盘文档给出的前缀逐位一致）。
ORIG_TSV_MD5 = {
    "en-nn": "58695bd9d5a1f1d39ac14a2d3364f00a",
    "de-nn": "0ed2245c5fe2d7b354d97d0a75ef1039",
    "en-pv": "0648c8a441ec731e5d4dd81b774933ee",
    "de-pv": "7c6e353097593d850c21c38885bd0767",
}
if DATA_MODE == "orig" and os.environ.get("COMPARTMENT_SKIP_MD5", "") != "1":
    import hashlib
    _bad_md5 = []
    for _n, _want in ORIG_TSV_MD5.items():
        _h = hashlib.md5((DATA_ROOT / f"{_n}-train.tsv").read_bytes()).hexdigest()
        if _h != _want:
            _bad_md5.append((_n, _want, _h))
    if _bad_md5:
        _msg = ["数据身份校验失败：以下文件与官方人类标注数据指纹不符",
                "（大概率挂成了合成/替换数据集 —— 这会让全部指标口径静默失效）："]
        _msg += [f"  {n}: 期望 {w} 实得 {h}" for n, w, h in _bad_md5]
        _msg += ["请挂载官方数据集，或设 COMPARTMENT_SKIP_MD5=1 显式跳过（自担口径风险）。"]
        raise RuntimeError("\n".join(_msg))
    print("数据身份校验通过：4 个官方 TSV 的 md5 指纹逐位匹配")


def resolve_aug_root():
    """返回第一个含 4 个 *-train.tsv 的合成增广目录（train_data_v2/filtered/）。找不到 ⇒ raise。"""
    import glob as _glob
    cands = []
    _env = os.environ.get("COMPARTMENT_AUG_DIR")
    if _env:
        cands.append(Path(_env).expanduser())
    cands.append(Path(CLOUD_AUG_DIR))
    cands.append(DATA_ROOT / "filtered")
    cands.append(DATA_ROOT / "train_data_v2" / "filtered")
    cands.append(DATA_ROOT.parent / "train_data_v2" / "filtered")
    cands.append(DATA_ROOT.parent.parent / "train_data_v2" / "filtered")
    _root_env = os.environ.get("COMPARTMENT_ROOT")
    if _root_env:
        cands.append(Path(_root_env).expanduser() / "train_data_v2" / "filtered")
    for _g in sorted(_glob.glob("/kaggle/input/**/train_data_v2/filtered", recursive=True)) \
            + sorted(_glob.glob("/kaggle/input/*/filtered")):
        cands.append(Path(_g))
    for c in dict.fromkeys(cands):
        try:
            if (c / "en-nn-train.tsv").exists():
                return c
        except OSError:
            continue
    raise FileNotFoundError(
        "DATA_MODE=aug 但找不到合成增广目录 train_data_v2/filtered/（应含 4 个 *-train.tsv）。\n"
        "已探测：\n  " + "\n  ".join(str(c) for c in cands)
        + "\n请设置 COMPARTMENT_AUG_DIR 指向该目录，或在 Kaggle 挂载对应 dataset。")


AUG_ROOT = None
AUG_ROWS = {}
if DATA_MODE in ("aug", "synth"):
    AUG_ROOT = resolve_aug_root()
    for _n in SUBSET_ORDER:
        AUG_ROWS[_n] = int(sum(1 for _ in open(AUG_ROOT / f"{_n}-train.tsv", "r",
                                               encoding="utf-8")) - 1)
    print(f"★ 数据增广已启用：DATA_MODE={DATA_MODE} | 合成数据目录 = {AUG_ROOT} "
          f"| 各子集合成行数 = {AUG_ROWS}")
else:
    print("DATA_MODE=orig（只用原始数据）")

In [ ]:
# ============ 演示：原始数据长什么样 ============
# 【为什么先看原始数据】不看一眼就上模型，最常见的翻车是「列名/列序/编码」与预期不符，
# 而这类错误往往不会报错，只会安静地给出离谱的分数。
_demo_name = SUBSET_ORDER[0]
_demo_spec = SUBSET_SPECS[_demo_name]
raw = pd.read_csv(DATA_ROOT / f"{_demo_name}-train.tsv", sep="\t")
print(f"{_demo_name} 原始形状: {raw.shape}")
print("列名:", list(raw.columns))
print(raw[[_demo_spec["key"]] + _demo_spec["cols"] + ["Context"]].head(3).to_string()[:700])

# 标签分布与句子长度分位：判断这个子集「值不值得建模」
print("\n--- 标签分布 ---")
for c in _demo_spec["cols"]:
    s = raw[c]
    print(f"{c}: 均值 {s.mean():.3f} | 标准差 {s.std():.3f} | 最小 {s.min():.2f} | 最大 {s.max():.2f}")
wlen = raw["Context"].str.split().str.len()
print("\n--- 句子长度（词）分位 ---")
print(dict(zip(["p50", "p90", "p95", "p99", "max"],
                np.percentile(wlen, [50, 90, 95, 99, 100]).round(0).tolist())))
print("→ 句子很短，MAX_LENGTH=160 足够，不会因为截断丢掉关键信息。")

In [ ]:
# ============ 封装：载入一个子集并做反泄漏处理 ============
def compute_ctx_share(df, sp, n_raters=10):
    """逐目标算「**语境有效度 share**」= 1 − 标注噪声方差 / 组内观测方差。

    【这个量是什么】
      每个标签是 **10 个整数评分的均值** ⇒ 该条例句标签的**噪声方差 = Std²/10**
      （Std 是数据集自带的评分者分歧列：ModStd / HeadStd / Std）。
      组内观测方差 = 同一个目标的几条例句之间，标签的方差。
      ⇒ share = 1 − 噪声/观测 = 「这个目标的组内差异里，有多少**不是**标注噪声」
      = **真实的语境效应占比**。

    【实测（analysis/v5_design_stats.json，四个子集的平均）】
      en-nn 0.183 / 0.213、de-nn 0.094 / 0.117、en-pv 0.441、de-pv 0.322
      ⇒ nn 上「同一目标不同句子分数不同」几乎全是标注噪声；
        pv 上约三分之一到一半是真实语境效应。

    【★ 为什么不用「人工 std 的目标级均值（歧义度）」当门控目标】
      实测 ρ(歧义度, 组内std)：en-nn 0.398/0.507、de-nn 0.739/0.639、
                              en-pv 0.132、de-pv 0.176
      ⇒ 歧义度在 nn 上与组内 std 高度相关（因为 nn 的组内 std 主要就是噪声），
        在 pv 上几乎不相关 ⇒ 用它监督「语境敏感度门」会把门**在 nn 上撑开**
        （方向正好相反）、在 pv 上又给不出信号 ⇒ 换成 share。

    【★ 合法性】只用**训练集**的标签与 Std 列算；官方测试集没有 Std 列，
      而门 g 在推理时只是「从 MWE 表征读出的一个标量」⇒ 不读任何测试集标签列。"""
    key = sp["key"]
    cols, stds = sp["cols"], sp.get("stds", [])
    per_target = {}
    for tgt, sub in df.groupby(key):
        shares = []
        for c, sc in zip(cols, stds):
            if c not in sub.columns or sc not in sub.columns:
                continue
            y = sub[c].to_numpy(dtype=float)
            if len(y) < 2:
                continue
            obs_var = float(np.var(y, ddof=1))
            noise_var = float(np.mean((sub[sc].to_numpy(dtype=float) ** 2) / n_raters))
            shares.append(float(np.clip(1.0 - noise_var / obs_var, 0.0, 1.0))
                          if obs_var > 0 else 0.0)
        per_target[tgt] = float(np.mean(shares)) if shares else 0.0
    return df[key].map(per_target).fillna(0.0).astype(float)


def load_subset(name, seed=SEED):
    """载入一个子集，返回反泄漏处理后的 DataFrame。
    【三步处理，顺序不能换】
    1) 去重：同一 (目标, 句子) 只保留一条，否则同一句话会被重复计权。
    2) 丢弃 ContextID：它是行号。本项目实测组内 Spearman(ContextID, 目标均分)
       高达 0.985，任何行号派生特征都会把「行序」冒充成「语境」。
    3) 物理打乱：让「同一目标的例句连续出现」在数据层面就不成立。
    【为什么要补统一标签列 y0/y1】四个子集的标签列名不一样
       （复合词是 ModAvg/HeadAvg，小品词动词只有 Avg）。
       统一成「两列 + 掩码」之后，模型结构、损失、指标、绘图全都不需要再判断子集类型。
       小品词动词的第 2 列用 m1=0 屏蔽：它不产生标签，也不进损失，但占住输出位。"""
    sp = SUBSET_SPECS[name]
    df = pd.read_csv(DATA_ROOT / f"{name}-train.tsv", sep="\t")
    _n_orig = len(df)
    # ---- 合成数据增广：在去重之前拼接 ⇒「原始 ∪ 合成」整体去重；表头缺列直接报错 ----
    if DATA_MODE in ("aug", "synth") and AUG_ROOT is not None:
        _syn = pd.read_csv(AUG_ROOT / f"{name}-train.tsv", sep="\t")
        _missing = [c for c in df.columns if c not in _syn.columns]
        if _missing:
            raise ValueError(
                f"[{name}] 合成数据缺列 {_missing}；"
                f"表头必须与原始逐字相同")
        if DATA_MODE == "synth":
            df = _syn[df.columns].copy()
        else:
            df = pd.concat([df, _syn[df.columns]], ignore_index=True)
    df = df.drop_duplicates(subset=[sp["key"], "Context"]).reset_index(drop=True)
    if "ContextID" in df.columns:
        df = df.drop(columns=["ContextID"])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df["y0"] = df[sp["cols"][0]].astype(float)
    df["m0"] = 1.0
    if len(sp["cols"]) > 1:
        df["y1"] = df[sp["cols"][1]].astype(float)
        df["m1"] = 1.0
    else:
        df["y1"] = 0.0
        df["m1"] = 0.0
    # 模块 2：目标级「语境有效度 share」，只在门控要监督时计算（默认档不需要）；
    # 只用训练集的标签与 Std 列 ⇒ 推理路径不依赖任何标签列。
    if M2_DUAL_STREAM and M2_GATE_SUPERVISION == "share":
        df["ctx_share"] = compute_ctx_share(df, sp)
    else:
        df["ctx_share"] = 0.0
    if SMOKE and SMOKE_N_ROWS:
        # 按目标取（保留组结构）；aug 模式按「目标中位行数」反推目标数，锚住总行数
        if DATA_MODE == "orig":
            _n_keep = max(1, SMOKE_N_ROWS // 5)
        else:
            _per_tgt = float(df.groupby(sp["key"]).size().median())
            _n_keep = max(1, int(round(SMOKE_N_ROWS / max(1.0, _per_tgt))))
        keep = df[sp["key"]].drop_duplicates().head(_n_keep)
        df = df[df[sp["key"]].isin(set(keep))].reset_index(drop=True)
    return df


for _n in SUBSET_ORDER:
    _d = load_subset(_n)
    _sp = SUBSET_SPECS[_n]
    print(f"[{_n}] 形状 {_d.shape} | 目标数 {_d[_sp['key']].nunique()} | "
          f"每目标例句数中位 {_d.groupby(_sp['key']).size().median()} | "
          f"ContextID 是否还在: {'ContextID' in _d.columns}")

## Step3 数据预处理

这一节做四件事，每件都有明确的动机：

| 环节 | 做什么 | 为什么 |
|---|---|---|
| ① 定位 MWE | 在句子里标出 MWE 出现的那个 span | 任务问的是「这个 MWE 在**这句**里是不是被当成整体理解」。不告诉模型句中哪个 span 对应 MWE，它只能靠整句语义猜 |
| ② 拼输入模板 | `MWE: <mwe> \| Context: <ctx>` | 两个模型必须看到**逐字相同**的输入，否则「模型差异」里会混进「模板差异」 |
| ③ 目标级互斥切分 | 按 MWE 切 train/val，并复刻官方「先分箱后分层抽样」 | 按行切会把同一 MWE 的例句分到两边，模型等于「见过这个 MWE」再去预测它，指标被高估 |
| ④ 分词 + 动态 padding | 批量分词，batch 内补齐到 8 的倍数 | 定长补齐到 `max_length` 会把每条的注意力开销抬到 O(L²) 的上限，动态补齐只按本批最长序列付费 |

**定位为什么难**：本项目实测过定位率——
英语复合词 98.6%、德语复合词 95.9% 可以字面命中；
但英语小品词动词只有 33.6%（`crack down` 在句子里常写成 `cracking down`），
德语更只有 20.8%（德语可分动词写作 `abgehauen`，粒子粘在动词前，且中间插前缀 `ge-`）。
所以定位必须做**分级级联**，而不是一次精确匹配。

In [ ]:
# ============ 文本归一化与词干工具（定位级联的零件）============
# MWE 在真实语料里的写法会变（德语 ä/ae、大小写、屈折后缀、可分动词的粒子黏着），
# 直接字符串相等匹配命中率会掉一大截；实现与 analysis/run_eda.py 同源，
# 定位覆盖率可与《数据分析报告》的数字直接对照。
def fold(s, ascii_mode=False):
    """小写 + NFKD 归一 + 去标点；ascii_mode=True 时把 ä/ö/ü 换成 ae/oe/ue。"""
    s = unicodedata.normalize("NFKD", str(s).lower()).replace("ß", "ss")
    s = re.sub(r"[^a-z0-9äöüß']", "", s)
    if ascii_mode:
        for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue")]:
            s = s.replace(a, b)
    return s


def stem_de(b):
    """德语词干：去掉常见后缀（ern/eln/en/er/e/t/ten/n）。"""
    for suf in ["ern", "eln", "en", "er", "e", "t", "ten", "n"]:
        if b.endswith(suf) and len(b) - len(suf) >= 4:
            return b[:-len(suf)]
    return b


def stem_en(w):
    """英语词干：去掉 ing/edly/ed/es/s/y/e。"""
    for suf in ["ing", "edly", "ed", "es", "s", "y", "e"]:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]
    return w


def stem_keys(base, lang):
    """取词干的前 5/4/3 个字符作为宽松匹配键。
    【为什么要多个长度】词干可能因屈折被切掉一截（crack → cracking），
    只用完整词干会漏；只用 3 个字符又太松。3/4/5 三档一起用，命中任一即算。"""
    ks = set()
    for v in (fold(base), fold(base, True)):
        st = stem_de(v) if lang == "de" else stem_en(v)
        for L in (5, 4, 3):
            if len(st) >= L:
                ks.add(st[:L])
    return ks


WORD_RE = re.compile(r"[a-zA-ZäöüÄÖÜß]+(?:[-'][a-zA-ZäöüÄÖÜß]+)*")
print("归一化与词干工具就绪")

In [ ]:
# ============ 演示：一条样本的定位过程 ============
# 定位是本任务最容易「看起来能跑但悄悄失准」的环节，先摊开看单条样本的每一步。
_demo_df = load_subset(_demo_name)
_sample_row = _demo_df.iloc[0].to_dict()
_mwe = _sample_row[_demo_spec["key"]]
_context = _sample_row["Context"]
print("MWE   :", _mwe)
print("原句  :", _context[:170])

# ① 先把句子切成词：看清「目标长什么样、句子里有哪些词」
_toks = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(_context.lower())]
print(f"\n句子切成 {len(_toks)} 个词:", [t[0] for t in _toks][:14])

# ② 归一化与词干：MWE 在语料里的大小写/变音/屈折都会变
print("\n归一化与词干:")
print("  fold(MWE) =", fold(_mwe))
if _demo_spec["task"] == "nn":
    print("  词干键(复合词) =", sorted(stem_keys(_mwe, _demo_spec["lang"])))
else:
    print("  fold(基干)   =", fold(_sample_row["Base"]))
    print("  词干键(基干) =", sorted(stem_keys(_sample_row["Base"], _demo_spec["lang"])))
    print("  粒子         =", _sample_row["Particle"])

# ③ 最小可运行版本：只试「精确匹配」这一个策略
_m = re.search(re.escape(_mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"),
               _context.lower())
print("\n最小版本（只做 S0 精确匹配）:", ("命中 → " + repr(_context[_m.start():_m.end()])) if _m else "未命中")
print("→ 精确匹配在复合词上够用，但在小品词动词上只能覆盖约三分之一；")
print("   所以下一格把级联封装成一个函数，逐级放宽。")

In [ ]:
# ============ 封装：定位 + 标记 + 输入模板 ============
def _mark(text, start, end):
    """在 text 的 [start, end) 两侧插入 [[ ]] 标记。
    【为什么用 ASCII 方括号而不是 « »】DeBERTa 用 SentencePiece，
    没有收录的字符会被拆成 byte fallback，既浪费 token 又干扰模型；
    ASCII 方括号在两套 tokenizer 里都是正常字符。"""
    return text[:start] + "[[" + text[start:end] + "]]" + text[end:]


def locate_target(row, sp):
    """在上下文中定位 MWE，返回 (标记后的句子, 策略标签)。
    级联：S0 精确匹配 → S1 历史转写还原 → S2a 德语前缀屈折 → S2b 德语后置分离
    → S3 英语词干前缀 → S4 相邻窗口 → S5 未定位（与 analysis/run_eda.py 同源）。
    精确匹配覆盖率视子集差异极大（复合词 96–99%，英语小品词动词仅 34%），
    必须逐级放宽；策略标签标出每条样本的定位置信度，出问题能一眼定位是哪一环。"""
    ctx = str(row["Context"])
    low = ctx.lower()
    mwe = str(row[sp["key"]])
    if sp["task"] == "nn":
        base, part = mwe, ""
    else:
        base, part = str(row["Base"]), str(row["Particle"])

    # S0 精确匹配（连字符/空格容错：compound-xxx 与 compound xxx 都算）
    m = re.search(re.escape(mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"), low)
    if m:
        return _mark(ctx, m.start(), m.end()), "S0 精确匹配"

    toks = [(mm.group(0), mm.start(), mm.end()) for mm in WORD_RE.finditer(low)]
    if not toks:
        return ctx, "S5 未定位"

    # S1 历史转写还原（德语 ä/ae 一类的历史拼写差异；能判定存在但定位不到具体 span）
    if fold(mwe, True) and fold(mwe, True) in fold(low, True):
        return ctx, "S1 历史转写还原"

    ks = stem_keys(base, sp["lang"])
    p = fold(part)
    pw = {fold(part), fold(part, True)}
    fw = [fold(t[0]) for t in toks]
    fa = [fold(t[0], True) for t in toks]

    if sp["lang"] == "de":
        # S2a 前缀型：粒子黏着词首 + 词干后随（abhauen / abgehauen）
        for i2, t in enumerate(toks):
            if not fw[i2].startswith(p):
                continue
            r, ra = fw[i2][len(p):], fa[i2][len(p):]
            if any(k and (k in r or k in ra) for k in ks):
                return _mark(ctx, t[1], t[2]), "S2a 德语前缀型(屈折)"
        # S2b 后置型：动词 + 粒子分离（最多跨 3 个 token）
        for i2 in range(len(toks)):
            if not any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
                continue
            for j2 in range(i2 + 1, min(i2 + 4, len(toks))):
                if fw[j2] in pw or fa[j2] in pw:
                    return _mark(ctx, toks[i2][1], toks[j2][2]), "S2b 德语后置型(动词分离)"
        return ctx, "S5 未定位"

    # S3 英语词干前缀命中
    for i2, t in enumerate(toks):
        if any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
            return _mark(ctx, t[1], t[2]), "S3 英文词干前缀"
    # S4 相邻双词窗口（分写短语被拆开）
    for i2 in range(len(toks) - 1):
        two = fw[i2] + fw[i2 + 1]
        if any(k and k in two for k in ks):
            return _mark(ctx, toks[i2][1], toks[i2 + 1][2]), "S4 相邻窗口"
    return ctx, "S5 未定位"


def build_input(mwe, marked_context):
    """最终送进模型的文本。
    【为什么模板固定成这样】
      1) 两个模型必须收到**逐字相同**的输入，否则对照实验的变量就不干净；
      2) MWE 放在前面：分词截断从尾部截，MWE 在前可保证它一定进得了模型；
      3) 用 `|` 而不是换行：换行在两套 tokenizer 里的切分行为不同。"""
    return f"MWE: {mwe} | Context: {marked_context}"


# ============ 模块 4：成分交换视图 ============
def build_swap_view(row, sp):
    """构造「成分交换视图」：把复合词的两个成分同时换序（MWE 字段 + 语境里的 span）。

    交换一致性是结构性监督：换序后「第一个成分的分」应等于原来「第二个成分的分」。
    只在语境里能字面命中整个复合词时才构造（德语 nn 是单词形、pv 只有一列 ⇒ 不可构造，
    造不出交换视图就不造，避免造出假句子）。
    返回 (text_swap, ok)；ok=False 表示这条样本没有可用的交换视图。"""
    if sp["task"] != "nn":
        return None, False
    comp = str(row[sp["key"]])
    mod = str(row.get("Mod", "") or "").strip()
    head = str(row.get("Head", "") or "").strip()
    if not mod or not head:
        return None, False
    mwe_swap = f"{head} {mod}"
    ctx = str(row["Context"])
    low = ctx.lower()
    pat = re.escape(comp.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s")
    m = re.search(pat, low)
    if not m:
        return None, False
    matched = ctx[m.start():m.end()]
    words = matched.split()
    if len(words) != 2:
        return None, False          # 不是两词复合词（或是带连字符的）⇒ 不构造
    swapped_span = f"{words[1]} {words[0]}"
    new_ctx = ctx[:m.start()] + swapped_span + ctx[m.end():]
    marked = (new_ctx[:m.start()] + "[[" + swapped_span + "]]"
              + new_ctx[m.start() + len(swapped_span):])
    return build_input(mwe_swap, marked), True


# 用同一句样本验证封装结果
_marked, _strat = locate_target(_sample_row, _demo_spec)
print("策略  :", _strat)
print("标记后:", _marked[:200])
print("最终输入:", build_input(_mwe, _marked)[:220])

In [ ]:
# ============ 定位覆盖率统计：把「标注好不好」变成可量化的数字 ============
# 级联实现与《数据分析报告》同源 ⇒ 覆盖率数字应能对上；对不上说明实现被改坏了。
for _n in SUBSET_ORDER:
    _sp = SUBSET_SPECS[_n]
    _d = load_subset(_n)
    _cnt = Counter()
    for _row in _d.to_dict("records"):
        _, _st = locate_target(_row, _sp)
        _cnt[_st] += 1
    _total = sum(_cnt.values())
    _loc = _total - _cnt.get("S5 未定位", 0) - _cnt.get("S1 历史转写还原", 0)
    print(f"{_n} 定位覆盖（n={_total}）成功标出 span {_loc / _total:.1%} | " +
          " | ".join(f"{k} {v}" for k, v in _cnt.most_common()))

In [ ]:
# ============ 切分：目标级互斥 + 复刻官方「先分箱后分层抽样」 ============
def split_by_target(df, sp, val_ratio=VAL_RATIO, seed=SEED, n_bins=5):
    """按「目标」互斥切分，并复刻官方的「先分箱后分层抽样」协议。
    【为什么不能按行切分】同一 MWE 的例句一旦分散在训练集和验证集，
       模型就等于「见过这个 MWE」再去预测它的另一句话，指标会被显著高估。
    【为什么要分箱】各目标的平均组合性差别很大（本项目实测德语复合词修饰语
       79.4% 的目标落在 [4,5]，几乎是天花板标签），直接随机切会让验证集的
       难度分布偏离训练集，分数不可比。
    【为什么分箱在目标层面做】官方协议是先按组合性分箱、再在目标层面按箱配额抽样；
       在目标层面分箱才能保证同一箱里的目标难度可比。"""
    key = sp["key"]
    tgt_mean = df.groupby(key)[sp["cols"]].mean().mean(axis=1)
    try:
        bins = pd.qcut(tgt_mean, n_bins, labels=False, duplicates="drop").to_numpy()
    except ValueError:
        bins = np.zeros(len(tgt_mean), dtype=int)

    rng = np.random.default_rng(seed)
    val_targets = []
    for b in np.unique(bins):
        idx = tgt_mean.index[bins == b].to_numpy()
        rng.shuffle(idx)
        k = max(1, int(round(len(idx) * val_ratio)))
        val_targets.extend(idx[:k].tolist())

    is_val = df[key].isin(set(val_targets)).to_numpy()
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:
# ============ 模型权重的下载与解析（端点自动回退）============
# 【为什么不用 from_pretrained 直接下】from_pretrained 走的是 import 时缓存的全局端点，
#   一旦缓存成错的端点，后面没法切。用 snapshot_download(endpoint=...) 显式指定，
#   端点切换是「每次调用级别」的行为；并且下载失败能自动换端点重试。
_ALLOW_PATTERNS = ["*.json", "*.txt", "*.model", "*.safetensors", "*.bin", "*.py", "*.tiktoken"]


def resolve_model_dir(repo_id):
    """把模型仓库下载到本地目录，返回本地路径。端点失败会自动回退。"""
    from huggingface_hub import snapshot_download
    endpoints = choose_endpoints(repo_id)
    if not endpoints:                      # cache 模式
        return repo_id
    last_err = None
    for ep in endpoints:
        try:
            path = snapshot_download(repo_id, endpoint=ep, allow_patterns=_ALLOW_PATTERNS)
            print(f"权重就绪: {repo_id} @ {ep}\n         -> {path}")
            return path
        except Exception as e:
            last_err = e
            print(f"从 {ep} 下载失败: {type(e).__name__}: {str(e)[:180]}")
    raise last_err


# 冒烟模式不下载任何权重（用本地缓存里的微型随机模型），正式模式才下。
if SMOKE:
    # xlmr 的冒烟源 = 真实仓库（与 qwen 同一模式）：冒烟只用它的 config+tokenizer
    # （shrink 成 2 层随机模型，见 cells_b.build_tiny_backbone），从不下载 2.2GB 权重。
    _SMOKE_IDS = {"deberta": "hf-internal-testing/tiny-random-DebertaV2Model",
                  "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
                  "xlmr": "FacebookAI/xlm-roberta-large"}
    MODEL_SOURCE = _SMOKE_IDS[MODEL_KEY]
    print("SMOKE=True：使用微型随机主干，不下载权重 ->", MODEL_SOURCE)
else:
    MODEL_SOURCE = resolve_model_dir(MODEL_REGISTRY[MODEL_KEY]["model_id"])

In [ ]:
# ============ 分词：把文本变成 input_ids / attention_mask ============
def load_tokenizer(model_key, source):
    """加载 tokenizer 并统一左右 padding。
    【为什么统一 padding_side="right"】本 notebook 的解码器分支用「最后一个真实 token」
       做池化，只有右 padding 时 attention_mask.sum(1)-1 才等于最后一个真实 token 的下标。
    【为什么显式设 pad_token】Qwen2.5 的 tokenizer 默认没有 pad token，
       不设的话 collator 会报错；直接复用 eos 是社区标准做法。"""
    tok = AutoTokenizer.from_pretrained(source)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    tok.model_max_length = 512
    return tok


def build_dataset(frame, tokenizer, max_length=MAX_LENGTH):
    """DataFrame → datasets.Dataset（四个子集可以混在一起）。
    每条记录含：
      text        送模型的字符串（已定位标记）
      labels      两个连续标签 [y0, y1]
      loss_mask   两个掩码 [m0, m1]（小品词动词的 m1=0，表示这一位没有标签）
      group_index 目标编号：**(子集, 目标) 的全局唯一整数**
      subset_index 子集编号
    【group_index 要带子集】group_index 决定「哪些例句属于同一个目标」，
      指标 2 的组内 std 就是按它分组算的；联合模式下不同子集可能存在同名目标，
      编号必须带子集前缀防撞号。
    【lm_head 模式下多存一列 input_ids2】不加回归头时，分数是模型自己生成的下一个 token，
      每个标签列各有一个「提示锚点」，两列锚点不同 ⇒ 两套 token 序列。
      锚点以 token id 形式直接接在正文后（正文按 max_length − len(锚点) 截断）：
      若拼进字符串再整体分词，截断可能切掉锚点，或把「": "」与数字合并成一个 token。"""
    records = []
    for row in frame.to_dict("records"):
        sp = SUBSET_SPECS[row["subset_name"]]
        mwe = str(row[sp["key"]])
        marked, _ = locate_target(row, sp)
        # 模块 4：成分交换视图（不可构造时 text_swap=None、swap_ok=0）
        _swap_txt, _swap_ok = (build_swap_view(row, sp) if M4_ROLE_EMB else (None, False))
        records.append({
            "text": build_input(mwe, marked),
            # 模块 2：类型流输入，只含 MWE、不含语境 ⇒ 表征是「类型级」的
            "text_type": f"MWE: {mwe}",
            # 模块 4：成分交换视图
            "text_swap": _swap_txt,
            "swap_ok": 1.0 if _swap_ok else 0.0,
            "labels": [float(row["y0"]), float(row["y1"])],
            "loss_mask": [float(row["m0"]), float(row["m1"])],
            # 模块 2：该目标的语境有效度 share（门控的可选监督目标）
            "ctx_share": float(row.get("ctx_share", 0.0)),
            "group_index": 0,          # 稍后统一重编号
            "group_key": f'{row["subset_name"]}||{mwe}',
            "subset_index": int(row["subset_id"]),
        })
    ds = Dataset.from_list(records)

    # 把 (子集, 目标) 映射成连续整数：损失按它做组内配对，指标 2 按它算组内 std
    codes = {k: i for i, k in enumerate(sorted(set(frame["group_key"].astype(str))))}
    ds = ds.map(lambda b: {"group_index": [codes[k] for k in b["group_key"]]}, batched=True)
    ds = ds.remove_columns(["group_key"])

    if HEAD_MODE == "lm_head":
        # ---- 无回归头：两列各带一个读出锚点 ----
        cue_ids = [tokenizer(c, add_special_tokens=False).input_ids for c in READOUT_CUES]
        assert len(cue_ids) == N_OUTPUTS, (
            f"读出锚点个数 {len(cue_ids)} 与输出列数 {N_OUTPUTS} 不一致")
        budget = max_length - max(len(c) for c in cue_ids)

        def _tok_lm(b):
            ids0, ids1 = [], []
            for t in b["text"]:
                base = tokenizer(t, truncation=True, max_length=budget).input_ids
                ids0.append(list(base) + list(cue_ids[0]))
                ids1.append(list(base) + list(cue_ids[1]))
            return {"input_ids": ids0, "input_ids2": ids1}

        ds = ds.map(_tok_lm, batched=True, remove_columns=["text"])
    else:
        # 批量分词：batched=True 一次处理一批，truncation 截断到 max_length
        ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_length),
                    batched=True, remove_columns=["text"])
    # ---- 模块 2：类型流的 token（只含 MWE，必须与语境分开、且用短序列省显存）----
    if M2_DUAL_STREAM:
        def _tok_type(b):
            _enc = tokenizer(b["text_type"], truncation=True, max_length=M2_TYPE_MAX_LEN)
            return {"input_ids_type": _enc["input_ids"],
                    "attention_mask_type": _enc["attention_mask"]}
        ds = ds.map(_tok_type, batched=True, remove_columns=["text_type"])
    # ---- 模块 4：成分交换视图的 token（第三条输入序列，仅训练期用于一致性正则）----
    if M4_ROLE_EMB:
        def _tok_swap(b):
            # 不可构造的样本用空串占位；它们的 swap_ok=0 ⇒ 损失不会用到这一行。
            _texts = [t if t is not None else "" for t in b["text_swap"]]
            _enc = tokenizer(_texts, truncation=True, max_length=max_length)
            return {"input_ids_swap": _enc["input_ids"],
                    "attention_mask_swap": _enc["attention_mask"]}
        ds = ds.map(_tok_swap, batched=True, remove_columns=["text_swap"])
    return ds


def ds_labels(ds, n_out=None):
    """把 datasets 的 labels 列取成 [N, n_out] 的 float 数组。
    【为什么需要这个函数】datasets 取「列表型列」时可能返回嵌套 list，
       也可能返回展平后的 list（取决于底层 arrow 的存储类型）。
       直接 np.asarray 再 reshape 会踩到 "cannot reshape array of size N into shape (N,2)"，
       所以统一走这个函数，按 n_out 把形状还原回去。"""
    n_out = n_out if n_out is not None else N_OUTPUTS
    arr = np.asarray(ds["labels"], dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, n_out)
    return arr


tokenizer = load_tokenizer(MODEL_KEY, MODEL_SOURCE)
print("tokenizer:", type(tokenizer).__name__, "| pad:", tokenizer.pad_token,
      "| padding_side:", tokenizer.padding_side)

In [ ]:
# ============ 关键封装：一次运行的全部数据准备 ============
# 【为什么要有这个函数】本 notebook 要自动跑 5 次（4 个单子集 + 1 个联合），
#   每次的「哪些子集参与」不同，切分结果、验证集、评测设置清单也就不同。
#   把这些打包成一个函数，5 次运行就只是同一个函数的 5 次调用，
#   绝不会出现「第一次跑的切分方式和第二次不一样」这类难查的问题。
def prepare_run(subset_list, tokenizer):
    """按给定的子集列表准备一次运行所需的全部数据。
    返回 dict：train_df / val_df / train_ds / val_ds / eval_cols。
    【切分在子集内部做，再合并】官方的分层协议是**在子集内部**做的；
      而且四个子集的目标难度分布差异极大（德语复合词修饰语 79.4% 落在 [4,5]，
      英语小品词动词均值只有 2.46），混在一起切会得到难度分布错乱的验证集。
    【group_key 带子集名】保证 group_index 在跨子集时也不会撞号。"""
    frames_tr, frames_va = [], []
    for name in subset_list:
        sp = SUBSET_SPECS[name]
        df = load_subset(name)
        df["subset_name"] = name
        df["subset_id"] = SUBSET_ORDER.index(name)
        df["group_key"] = name + "||" + df[sp["key"]].astype(str)
        tr, va = split_by_target(df, sp)
        frames_tr.append(tr)
        frames_va.append(va)
    train_df = pd.concat(frames_tr, ignore_index=True)
    val_df = pd.concat(frames_va, ignore_index=True)
    train_ds = build_dataset(train_df, tokenizer)
    val_ds = build_dataset(val_df, tokenizer)
    # 评测设置清单：复合词 2 列、小品词动词 1 列
    eval_cols = [(n, c, j) for n in subset_list for j, c in enumerate(SUBSET_SPECS[n]["cols"])]
    return dict(subset_list=list(subset_list), train_df=train_df, val_df=val_df,
                train_ds=train_ds, val_ds=val_ds, eval_cols=eval_cols)


# 演示一次：只跑 en-nn，确认切分互斥、分箱生效
_demo_run = prepare_run(["en-nn"], tokenizer)
_tr, _va = _demo_run["train_df"], _demo_run["val_df"]
print(f"en-nn 训练集 {_tr.shape} | 验证集 {_va.shape}")
print("目标交集:", len(set(_tr["group_key"]) & set(_va["group_key"])), "（必须为 0）")
for _n, _c, _j in _demo_run["eval_cols"]:
    _col = "y0" if _j == 0 else "y1"
    print(f"  {_n}·{_c}: train 均值 {_tr[_col].mean():.3f} | val 均值 {_va[_col].mean():.3f}")
print("示例解码:", tokenizer.decode(_demo_run["train_ds"][0]["input_ids"])[:220])

In [ ]:
# ============ 动态 padding 的省算力实测 ============
# 【为什么要专门量一下】注意力开销是 O(L²)。若每条都补齐到 MAX_LENGTH=160，
#   而本批最长的只有 60 token，实际算力会被浪费掉一倍以上。
#   下面的对照把「定长补齐」与「动态补齐」的 token 总量直接算出来。
_lens = [len(x) for x in _demo_run["train_ds"]["input_ids"]]
_fixed_total = len(_lens) * MAX_LENGTH
_dynamic_total = 0
for _i in range(0, len(_lens), BATCH_SIZE):
    _dynamic_total += min(BATCH_SIZE, len(_lens) - _i) * max(_lens[_i:_i + BATCH_SIZE])
print(f"序列长度: 中位 {int(np.median(_lens))} | 均值 {int(np.mean(_lens))} | 最长 {max(_lens)}")
print(f"定长补齐到 {MAX_LENGTH} 的 token 总量: {_fixed_total:,}")
print(f"动态补齐的 token 总量            : {_dynamic_total:,}")
print(f"→ 动态补齐省下 {(1 - _dynamic_total / _fixed_total):.1%} 的 token，"
      f"注意力开销按平方计，实际省得更多。")

In [ ]:
# ============ Collator：把样本拼成 batch ============
class CompartmentCollator:
    """把若干条样本拼成一个 batch。
    【为什么必须自己写，不能直接用 DataCollatorWithPadding】
      它会把 labels 当作**序列**去 pad，用 -100 填充缺失位置。
      对分类任务那是有用的「忽略位」；对我们的**连续值回归标签**是灾难：
      被填成 -100 的位置会一起参与 MSE，把 loss 直接拉飞。
      所以这里把 labels 单独叠成 float 张量、group_index 叠成 long 张量，
      只把 input_ids / attention_mask 交给 tokenizer.pad。
    【pad_to_multiple_of=8】把序列长度补到 8 的倍数，
      矩阵乘的尾维对齐后张量核心能走更快的 kernel。"""
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        model_inputs = [{k: v for k, v in f.items()
                         if k not in ("labels", "loss_mask", "group_index", "subset_index",
                                      "ctx_share", "swap_ok")
                         and k != "input_ids2"
                         and not k.endswith("_type")
                         and not k.endswith("_swap")}
                        for f in features]
        batch = self.tokenizer.pad(model_inputs, padding=True,
                                   pad_to_multiple_of=self.pad_to_multiple_of,
                                   return_tensors="pt")
        # lm_head 模式的第二套 token（第二个读出锚点）：tokenizer.pad 不认 "input_ids2"
        # 这个名字 ⇒ 单独 pad 一次；两套序列长度可能不同，各自 pad 才不会读错最后一个真实 token。
        if "input_ids2" in features[0]:
            _b2 = self.tokenizer.pad([{"input_ids": f["input_ids2"]} for f in features],
                                     padding=True, pad_to_multiple_of=self.pad_to_multiple_of,
                                     return_tensors="pt")
            batch["input_ids2"] = _b2["input_ids"]
            batch["attention_mask2"] = _b2["attention_mask"]
        # 模块 2：类型流的 token，单独 pad（长度与主序列不同）
        if "input_ids_type" in features[0]:
            _bt = self.tokenizer.pad([{"input_ids": f["input_ids_type"]} for f in features],
                                     padding=True, pad_to_multiple_of=self.pad_to_multiple_of,
                                     return_tensors="pt")
            batch["input_ids_type"] = _bt["input_ids"]
            batch["attention_mask_type"] = _bt["attention_mask"]
        # 模块 4：成分交换视图的 token，再单独 pad（第三条序列）
        if "input_ids_swap" in features[0]:
            _bsw = self.tokenizer.pad([{"input_ids": f["input_ids_swap"]} for f in features],
                                      padding=True, pad_to_multiple_of=self.pad_to_multiple_of,
                                      return_tensors="pt")
            batch["input_ids_swap"] = _bsw["input_ids"]
            batch["attention_mask_swap"] = _bsw["attention_mask"]
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.float32)
        # loss_mask：标明每一列**是否真的有标签**。小品词动词只有第 1 列有标签，
        # 第 2 列必须被屏蔽，否则模型会被要求去拟合一个恒为 0 的假标签。
        batch["loss_mask"] = torch.tensor([f["loss_mask"] for f in features], dtype=torch.float32)
        batch["group_index"] = torch.tensor([f["group_index"] for f in features], dtype=torch.long)
        batch["subset_index"] = torch.tensor([f["subset_index"] for f in features], dtype=torch.long)
        # 模块 2：该目标的「语境有效度 share」（门控的可选监督目标；默认档不用）
        batch["ctx_share"] = torch.tensor([float(f.get("ctx_share", 0.0)) for f in features],
                                          dtype=torch.float32)
        # 模块 4：这一行有没有可用的成分交换视图（0/1）
        batch["swap_ok"] = torch.tensor([float(f.get("swap_ok", 0.0)) for f in features],
                                        dtype=torch.float32)
        return batch


collator = CompartmentCollator(tokenizer)
_demo_batch = collator([_demo_run["train_ds"][i] for i in range(4)])
for k, v in _demo_batch.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")
print("\nlabels 样例   :", _demo_batch["labels"].tolist())
print("loss_mask 样例:", _demo_batch["loss_mask"].tolist())
print("subset_index  :", _demo_batch["subset_index"].tolist())
print("→ labels 是 [B, 2] 的 float 张量，没有 -100 填充；loss_mask 标出哪一列真有标签。")

## Step4 创建模型

**这一节要解决什么**：把「编码器」和「解码器」两个完全不同的架构，
装配成**同一个形状的模型**：`(input_ids, attention_mask) → 两个 0–5 的连续分数`。

### 两种「读出方式」

两条基线各自验证一个机制：DeBERTa 用回归头，Qwen 完全不加回归头。
两者的**输入端完全一样**，只有末端不同：

```
【A】回归头（DeBERTa，head_mode = "regression"）

AutoModel 主干 ──► last_hidden_state [B, L, H]
                          │
              掩码均值池化(编码器) / 取最后一个真实 token(解码器)
                          │
                       [B, H]
                          │
                  Dropout → Linear(H, n_out)          ← 新建的回归头
                          │
                    logits [B, n_out]
                          │
                  × 5 并落在 (0,5)

【B】无回归头（Qwen，head_mode = "lm_head"）

Qwen2ForCausalLM ──► 主干 ──► last_hidden_state [B, L, H]
                                        │
                            取锚点（"\nModifier: "）最后一个 token 处
                                        │
                                    [B, H]
                                        │
                   主干**自带**的 LM 头（不新建任何层，与 embedding 共享权重）
                                        │
                              全词表 logits [B, V]
                                        │
                        只取 "0"–"5" 这 6 个 token（id 15…20）
                                        │
                             p = softmax(6 个 logit)
                                        │
                    分数 = Σ_k k·p_k  ∈ [0,5]     ← 6 个数的凸组合
```

**为什么「期望值」这个读法是对的**：标签本来就是 0–5 的**连续均值**
（实测：每个标签是 10 个整数评分的均值，粒度 0.1，唯一值 47–51 个）。
所以「标签 3.5」对应的天然分布就是「3 与 4 的概率混合」，
而不是「3 或 4」—— 模型输出一个数字 token 的分布、再取期望，
恰好与标签的生成方式**同构**。

**为什么不用 `AutoModelForSequenceClassification`**：两条理由，都很硬。

1. **任务类型不匹配**。强行用 `num_labels=6` + 交叉熵，会把「4.2 分」和「4.0 分」的差别完全抹平，
   而指标 1（组合性相关）恰恰只看**排序**——抹平排序等于直接放弃指标 1。
   （注意：lm_head 模式里出现的「6」**不是**把标签离散化成 6 类，
   而是模型本来就要生成的 6 个 token；两回事。）
2. **接口不统一**。DeBERTa 的 `DebertaV2ForSequenceClassification` 与
   Qwen 的 `Qwen2ForSequenceClassification` 在 pooling、损失、logits 形状上都不一致。
   自己写一个 20 行的头，两个模型才能**逐行同构**，
   对照实验里唯一的变量才真的是「架构」而不是「框架细节」。

**为什么回归头那条要 ×5**：标签的量纲是 0–5，模型是无约束实数输出。
用 `sigmoid × 5` 把输出**先验地**限制在合法区间，
这样即使训练早期发散，预测值也不会跑到 [-100, 100] 这种让 RMSE 直接爆掉的范围。
（lm_head 模式不需要这一步：期望值是 6 个概率的凸组合，**天然**落在 [0,5]。）

In [ ]:
# ============ 统一：把模型 config 缩到小尺寸（并同步所有派生维度）============
def shrink_config(cfg, num_layers=2, hidden_size=64, num_heads=4, tokenizer=None):
    """把 config 缩小到一个小尺寸，**同时同步所有派生维度**。
    不同架构的「注意力内部宽度」不都等于 hidden_size：
      · DeBERTa-v2 的注意力内部维度 = attention_head_size × num_attention_heads，
        只改 heads 不同步它，前向会抛 mat1/mat2 形状不匹配；
      · Qwen2 的 head_dim、DeBERTa-v2 的 embedding_size 同理必须跟着改。
    防御约定：缩尺寸只走这一个函数，任何地方不许直接改 cfg 的尺寸字段。
    """
    cfg.num_hidden_layers = num_layers
    cfg.hidden_size = hidden_size
    cfg.num_attention_heads = num_heads
    cfg.intermediate_size = hidden_size * 2
    # 下面三个是「派生宽度」，必须跟着 hidden_size 一起改
    if hasattr(cfg, "attention_head_size"):
        cfg.attention_head_size = hidden_size // num_heads
    if hasattr(cfg, "head_dim"):
        cfg.head_dim = hidden_size // num_heads
    if hasattr(cfg, "embedding_size"):
        cfg.embedding_size = hidden_size
    if hasattr(cfg, "num_key_value_heads"):
        cfg.num_key_value_heads = min(int(getattr(cfg, "num_key_value_heads", 1)), num_heads)
    if tokenizer is not None:
        # 【为什么要抬 vocab_size】tokenizer 的词表可能比 config 记录的 embedding 行数更大
        #   （SentencePiece 的 [UNK]/控制符不占 embedding 行），不抬大会在 forward 时越界。
        cfg.vocab_size = max(cfg.vocab_size, len(tokenizer))
    return cfg


# ============ 演示：主干一次前向，看看 last_hidden_state 长什么样 ============
# 【为什么先演示】很多人在这里踩坑：用 AutoModel 加载时拿到的其实是
#   「最后一层隐状态」，而不是「句子的向量」。
#   池化方式选错，模型就等于在拿一堆碎片做回归。先看清楚形状再写封装。
_demo_cfg = shrink_config(AutoConfig.from_pretrained(MODEL_SOURCE), tokenizer=tokenizer)
_demo_backbone = AutoModel.from_config(_demo_cfg)
_demo_backbone.eval()

_demo_ids = tokenizer("MWE: account book | Context: He read his [[account book]] last night.",
                      return_tensors="pt")
with torch.no_grad():
    _demo_out = _demo_backbone(input_ids=_demo_ids["input_ids"],
                               attention_mask=_demo_ids["attention_mask"])
_h = _demo_out.last_hidden_state
print("主干类型:", type(_demo_backbone).__name__)
print("config: hidden_size=%d num_attention_heads=%d attention_head_size=%s head_dim=%s" % (
    _demo_cfg.hidden_size, _demo_cfg.num_attention_heads,
    getattr(_demo_cfg, "attention_head_size", "-"), getattr(_demo_cfg, "head_dim", "-")))
print("last_hidden_state shape:", tuple(_h.shape))
# 断言把「输出宽度 == hidden_size」钉死：派生宽度字段漏同步会当场被抓住
assert _h.shape[-1] == _demo_cfg.hidden_size, (
    f"主干输出宽度 {_h.shape[-1]} ≠ hidden_size {_demo_cfg.hidden_size}；"
    "多半是 config 里有派生宽度字段（attention_head_size / head_dim / embedding_size）没同步")
print("→ [batch, 序列长度, 隐层维度]。真正的「句子向量」还需要池化，这就是下面要做的。")

In [ ]:
# ============ 封装：统一的读出模型（两种读出方式）============
class CompositionalityRegressor(nn.Module):
    """AutoModel 主干 + 池化 + 读出，两种读出方式由 HEAD_MODE 决定。

    "regression"：池化 → Linear 回归头 → ×5·sigmoid（输出先验地限制在 [0,5]）。
    "lm_head"：完全不加回归头，分数由主干自带的 LM 头读出——在锚点最后一个 token
        处取 "0"–"5" 六个 token 的概率分布，取期望值 Σ k·p_k（凸组合天然落在 [0,5]）。

    pooling："mean"（编码器，掩码均值）/"last"（解码器，最后一个真实 token）。

    lm_head 模式需要两次前向：两列分数各配一个锚点，同一锚点读两次只会得到同一个数；
    第 2 列只对有真标签的行跑（need2 掩码），联合运行时省约 1/4 前向开销。

    标签是 0–5 的连续均值（10 个整数评分的均值），「标签 3.5 = 3 与 4 的概率混合」，
    所以 lm_head 的 6 类不是离散化，而是模型本来就要生成的 6 个数字 token。"""

    def __init__(self, backbone, hidden_size, n_outputs=1, pooling="mean", dropout=0.1,
                 head_mode="regression", lm_head=None, digit_ids=None, digit_values=None,
                 proj_dim=None, tied_to_embedding=False, m1_dist_head=False, n_classes=6,
                 m2_dual_stream=False, m4_role_emb=False):
        super().__init__()
        self.backbone = backbone
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.head_mode = head_mode
        self.n_outputs = int(n_outputs)
        # 模块 1（序数-分布头）：m1_dist_head=True 时回归头输出 6 类（0–5）logits，
        # 预测值 = 期望值 Σ k·p(k)。期望值是凸组合 ⇒ 值域 [0,5] 由结构保证；
        # 标签本身是「10 个整数评分的均值」，分布形式与标签生成过程同构。
        # lm_head 模式下读出本来就是 6 类分布 ⇒ 该模块对 Qwen 是空操作。
        self.m1_dist_head = bool(m1_dist_head) and head_mode == "regression"
        self.n_classes = int(n_classes)
        # 期望值读数要用的「档位取值」：0,1,2,3,4,5（两种读出方式都用它）
        self.register_buffer("digit_values",
                             torch.as_tensor(list(digit_values or [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]),
                                             dtype=torch.float32))
        if head_mode == "regression":
            if self.m1_dist_head:
                # ★ 模块 1：Linear(H → n_outputs×6) ⇒ 每个标签列一套 6 类 logits
                self.head = nn.Linear(hidden_size, n_outputs * self.n_classes)
            else:
                self.head = nn.Linear(hidden_size, n_outputs)
            self.lm_head = None
            self.tied_to_embedding = False
        else:
            # lm_head 模式：不新建任何输出层，读出用主干自带的头。
            # 绑定（tie_word_embeddings=True）时 lm_head.weight 与 embed_tokens.weight
            # 是同一块显存，两者都进 state_dict 会被 safetensors 拒绝保存
            # ⇒ 绑定时不挂 lm_head 子模块，读出直接用 embedding 矩阵算
            #   pooled @ embed.weight.T（与 Linear(hidden, vocab, bias=False) 等价）。
            self.head = None
            self.tied_to_embedding = bool(tied_to_embedding)
            self.lm_head = None if self.tied_to_embedding else lm_head
            assert digit_ids is not None and len(digit_ids) == 6, (
                f"lm_head 模式需要 6 个数字 token 的 id，实际收到 {digit_ids}")
            self.register_buffer("digit_ids", torch.as_tensor(list(digit_ids), dtype=torch.long))
            # 读出梯度累积缓冲（[6, hidden]）：读出通路的梯度只落在 6 行上，
            # 用 detach + 反向 hook 手工算，避免自动求导为整块 [vocab, hidden]
            # 分配稠密临时缓冲（约 890 MiB）。step 之前由 flush_readout_grad 并进 .grad。
            self._pending_readout_g6 = None
        # 【★ 投影头只给对比学习用】对比学习在**投影空间**里做，不污染主干的表征空间
        #   （这是 SimCLR 的标准做法：投影头只服务于对比损失，下游任务用的是投影前的向量）。
        self.proj = nn.Linear(hidden_size, proj_dim) if proj_dim else None
        # ---- 模块 2：类型-语境双流 + 语境敏感度门 ----
        # 分数 = (1 − g)·μ̂_type + g·δ_ctx：μ̂_type 是类型流（只编码 MWE）的读出，
        # δ_ctx 是语境流（整串输入）的读出，g 是只从类型表征读出的逐目标门。
        # 凸组合保证分数恒在 [0,5]；g 只看类型表征 ⇒ 它是「这个目标有多语境敏感」
        # 这个逐目标的量，不随例句变化。
        self.m2_dual_stream = bool(m2_dual_stream)
        if self.m2_dual_stream:
            # 类型流的头故意用标量回归（5×sigmoid），不用模块 1 的分布头：
            # 若类型头也换分布头，两个模块的效果就混在一起、无法归因。
            self.type_head = nn.Linear(hidden_size, n_outputs)
            # 门：权重初始化压小、偏置 0 ⇒ 初始 g ≈ 0.5，两条流都能拿到梯度。
            self.gate_head = nn.Linear(hidden_size, 1)
            with torch.no_grad():
                self.gate_head.weight.mul_(0.01)
                self.gate_head.bias.zero_()
        # ---- 模块 4：角色嵌入（把 Mod / Head 变成「同一 MWE 的两个视角」）----
        # 共享主干 + 可学习的角色向量：第 r 列的读出输入 = pooled + role_emb[r]；
        # 参数量只增加 n_outputs × hidden（约 3K），两列共享全部内容表征，
        # 只在「从哪个角度看」上不同。
        self.m4_role_emb = bool(m4_role_emb)
        if self.m4_role_emb:
            self.role_emb = nn.Embedding(n_outputs, hidden_size)
            with torch.no_grad():
                self.role_emb.weight.normal_(0.0, 0.02)

    def _role_pooled(self, pooled, role):
        """给池化表征加上「角色向量」（模块 4）。
        【为什么是加在池化之后、读出之前】主干与池化完全共享 ⇒
          两个角色共用同一份「句子的内容」，只有「从哪个角度看」不同 ⇒
          这正是「同一 MWE 的两个视角」的结构表达。"""
        if not getattr(self, "m4_role_emb", False):
            return pooled
        return pooled + self.role_emb.weight[role]

    def readout_weight(self):
        """读出通路的权重张量（lm_head 模式用）。
        【为什么要单独暴露】绑定的情形下它就是 embedding 矩阵本身；
        开训前的梯度检查需要拿到它的梯度来判断「读出通路有没有接上」。"""
        if self.lm_head is not None:
            return self.lm_head.weight
        return self.backbone.embed_tokens.weight

    # ---- gradient checkpointing 的转发（Trainer 会直接调这两个成员）----
    # 本模型是自定义 nn.Module（不是 PreTrainedModel），没有 gradient_checkpointing_enable
    # ⇒ Trainer 开 checkpointing 时会 AttributeError，必须转发给主干。
    supports_gradient_checkpointing = True

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
        """把 Trainer's 调用转发给主干（主干本身支持 checkpointing）。"""
        if not hasattr(self.backbone, "gradient_checkpointing_enable"):
            raise RuntimeError("当前主干不支持 gradient checkpointing，无法开启")
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs)

    @property
    def is_gradient_checkpointing(self):
        return bool(getattr(self.backbone, "is_gradient_checkpointing", False))

    # ---- 池化 ----
    def _pool(self, h, attention_mask):
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            return (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        last_idx = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), last_idx]

    # ---- 读出通路的梯度只落在 6 行上 ----
    def flush_readout_grad(self):
        """把累积的「读出梯度」写进读出权重的 `.grad`（只动那 6 行）。

        绑定（tie_word_embeddings=True）时，读出权重就是 [vocab, hidden] 的 embedding 矩阵；
        若让自动求导直接回传读出梯度，反向会同时出现「查表路径 + 读出路径」两份
        稠密梯度（每份 ≈ 890 MiB）。所以：前向 detach + tensor hook 手工累积
        g6 = grad_out^T @ pooled（与 index_select 的自动求导数值完全一致），
        在 optimizer.step 之前（training_step 里）用 index_add_ 并进 W.grad，
        全程不新建稠密缓冲。
        """
        _p = getattr(self, "_pending_readout_g6", None)
        if _p is None:
            return
        W = self.readout_weight()
        with torch.no_grad():
            if W.grad is None:
                W.grad = torch.zeros_like(W)
            W.grad.index_add_(0, self.digit_ids, _p.to(W.grad.dtype))
        self._pending_readout_g6 = None

    def _digit_logits(self, pooled):
        """把隐状态过一遍主干自带的 LM 头，取出 "0"–"5" 这 6 个 token 的 logits。

        三个关键点：
        ① 先取 6 行再做 6 列矩阵乘（W6 = W[digit_ids]，out = pooled @ W6.T），
           与「先算全词表再取 6 列」数值完全等价（第 k 列 logit 只依赖 W[k]），
           但算力降到 6/151936；
        ② 强制 fp32 并显式关掉 autocast：AMP 下主干输出是 fp16，
           Qwen2.5 有 massive activation，fp16 下极易溢出成 inf；
           只写 .float() 不够——autocast 会把 matmul 降回 fp16；
        ③ 前向 detach + 反向 hook 手工累积 6 行梯度（见 flush_readout_grad），
           避免为整块绑定矩阵分配稠密临时缓冲。"""
        if self.lm_head is not None:
            W = self.lm_head.weight              # [vocab, hidden]
            b = self.lm_head.bias                # 可能为 None
        else:
            W = self.backbone.embed_tokens.weight    # [vocab, hidden]
            b = None
        # detach 掉 W 的自动求导路径：读出仍可训练（梯度由 hook 手工算），
        # 但自动求导不再为这条路径分配 [vocab, hidden] 的稠密临时缓冲。
        Wd = W.detach()
        # 强制 fp32：必须在 CUDA 上显式关掉 autocast，否则 .float() 之后
        # matmul 仍会被 autocast 降回 fp16。
        if pooled.is_cuda:
            with torch.autocast(device_type="cuda", enabled=False):
                out = pooled.float() @ Wd.index_select(0, self.digit_ids).float().t()
        else:
            out = pooled.float() @ Wd.index_select(0, self.digit_ids).float().t()
        # 反向：把「输出处的梯度」换算成 6 行的梯度并累积（与 index_select 的
        # 自动求导逐位一致：它做的也是把这 6 行的梯度 scatter 回 W）
        if out.requires_grad:
            if getattr(self, "_pending_readout_g6", None) is None:
                self._pending_readout_g6 = torch.zeros(
                    (self.digit_ids.numel(), W.shape[1]),
                    dtype=torch.float32, device=W.device)
            _pooled_d = pooled.detach().float()

            def _readout_hook(g, _pd=_pooled_d):
                with torch.no_grad():
                    self._pending_readout_g6 += g.float().t() @ _pd
            out.register_hook(_readout_hook)
        if b is not None:
            out = out + b.index_select(0, self.digit_ids).float()
        return out

    # ---- 只取隐状态，不算全词表 logits ----
    def _hidden(self, input_ids, attention_mask):
        """主干前向，只返回隐状态。

        lm_head 模式下 self.backbone 是 Qwen2ForCausalLM，直接调它会顺带算出
        全词表 logits [B, L, 151936]（前向 + 反向 + 稠密梯度共 ≈1.3 GiB 纯浪费，
        而我们只要 6 行）。直接调内层 `backbone.model` 是安全的：
        它就是 Qwen2Model（含最后的 RMSNorm），隐状态与走 lm_head 之前完全一致。
        兼容回退：logits_to_keep=1 → 原写法。"""
        _inner = getattr(self.backbone, "model", None)
        if _inner is not None:
            _out = _inner(input_ids=input_ids, attention_mask=attention_mask)
            return _out.last_hidden_state
        try:
            _out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                                 logits_to_keep=1)
            return _out.last_hidden_state
        except TypeError:
            _out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            return _out.last_hidden_state

    # ---- 模块 2：类型流 + 敏感度门 ----
    def _fuse_type_stream(self, scores, input_ids_type, attention_mask_type):
        """把「语境流读出」与「类型流读出」按门 g 融合。

        两路前向必须结构上分开：类型流只看到 MWE（学到「这个词本身有多组合」），
        语境流看到整串（学到「在这句话里有多组合」），两者是不同的量。
        门 g 只从类型表征读出 ⇒ g 逐目标恒定（「这个目标有多语境敏感」），
        不会随例句抖动、与组内 std 的机制纠缠。"""
        if not self.m2_dual_stream:
            return scores, None, None, scores
        if input_ids_type is None or attention_mask_type is None:
            raise RuntimeError(
                "模块 2（双流）已打开，但 batch 里没有 input_ids_type / attention_mask_type。"
                "请确认数据管线走了 M2_DUAL_STREAM 分支。")
        h_t = self.backbone(input_ids=input_ids_type,
                             attention_mask=attention_mask_type).last_hidden_state
        pooled_t = self._pool(h_t, attention_mask_type)
        g = torch.sigmoid(self.gate_head(pooled_t))                          # [B,1]
        if getattr(self, "m4_role_emb", False):
            # 【模块 4】类型流同样按角色读出（两列各有自己的类型级读出）
            #   【★ 注意取列】type_head 的输出已经是 [B, n_outputs] ⇒ 每个角色只取自己那一列，
            #     否则 stack 之后会变成三维（这正是本轮冒烟抓到的形状 bug）。
            mu = 5.0 * torch.sigmoid(torch.stack(
                [self.type_head(self.dropout(self._role_pooled(pooled_t, _r)))[:, _r]
                 for _r in range(self.n_outputs)], dim=1))
        else:
            mu = 5.0 * torch.sigmoid(self.type_head(self.dropout(pooled_t)))  # [B,n_out]
        return (1.0 - g) * mu + g * scores, g, mu, scores

    def forward(self, input_ids=None, attention_mask=None,
                input_ids2=None, attention_mask2=None, need2=None,
                input_ids_type=None, attention_mask_type=None,
                labels=None, group_index=None, **kwargs):
        """返回 ModelOutput(logits=..., digit_logits=..., proj=...)。
        【注意】labels / group_index 只在这里被「接收但不使用」——
           真正的损失在 Step7 的 Trainer 里算，这样才方便把多个损失项拆开做消融。
        **kwargs 是必需的：Trainer 会把数据集里的额外列一起塞进来。"""
        if self.head_mode == "regression":
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            h = out.last_hidden_state
            pooled = self._pool(h, attention_mask)
            proj = None
            if self.proj is not None:
                # L2 归一化：对比学习的相似度就是余弦相似度
                proj = F.normalize(self.proj(self.dropout(pooled)), dim=-1)
            # 模块 4：角色条件化读出——两列共用同一个头 + 各自的角色向量
            # （score_r = readout(pooled + role_emb[r])）。
            if getattr(self, "m4_role_emb", False):
                _parts = []
                for _r in range(self.n_outputs):
                    _xr = self._role_pooled(pooled, _r)
                    if self.m1_dist_head:
                        # 【★ 注意形状】头的输出是 [B, n_outputs×6] ⇒ 每个角色取自己那一段
                        #   （不能直接 view(-1, 6)，那会把 n_outputs 个角色混在一起）
                        _parts.append(self.head(self.dropout(_xr))
                                      .view(pooled.shape[0], self.n_outputs, self.n_classes)[:, _r, :])
                    else:
                        _parts.append(self.head(self.dropout(_xr)).view(pooled.shape[0], self.n_outputs)[:, _r])
                if self.m1_dist_head:
                    d6 = torch.stack(_parts, dim=1)             # [B, n_out, 6]
                    p = d6.softmax(dim=-1)
                    vals = self.digit_values.to(p.dtype)
                    scores = (p * vals).sum(dim=-1)
                    _var = (p * (vals ** 2)).sum(dim=-1) - scores ** 2
                    dstd = torch.sqrt(_var.clamp(min=0.0))
                else:
                    scores = 5.0 * torch.sigmoid(torch.stack(_parts, dim=1))
                    d6 = None
                    dstd = None
            elif self.m1_dist_head:
                # 模块 1：头输出 6 类 logits ⇒ softmax 得分布 ⇒ 期望值即预测分数；
                # 分布宽度（var = Σ k²p_k − (Σ k p_k)² 的平方根）是不确定度的显式读出，
                # 不参与训练（否则会拟合标注噪声），只作诊断。
                d6 = self.head(self.dropout(pooled)).view(-1, self.n_outputs, self.n_classes)
                p = d6.softmax(dim=-1)
                vals = self.digit_values.to(p.dtype)
                scores = (p * vals).sum(dim=-1)                       # [B, n_out] ∈ [0,5]
                _var = (p * (vals ** 2)).sum(dim=-1) - scores ** 2   # 分布方差
                dstd = torch.sqrt(_var.clamp(min=0.0))                # 分布标准差
            else:
                # 【为什么在这里就把输出压到 (0,5)】标签的量纲就是 0–5，模型是无约束实数。
                #   用 5×sigmoid 把输出**先验地**限制在合法区间，好处是：
                #   ① 训练早期发散时预测值也跑不出 [-100,100]，RMSE 不会被离谱值带崩；
                #   ② 导出时不需要任何事后裁剪，官方要求的「值域 [0,5]」由结构保证。
                #   代价：sigmoid 两端梯度会变小，标签真的取到 0 或 5 时模型只能逼近而不能取到，
                #   这在 0–5 有界回归里是可以接受的常规取舍。
                scores = 5.0 * torch.sigmoid(self.head(self.dropout(pooled)))
                d6 = None
                dstd = None
            # ---- 模块 2：与类型流融合 ----
            scores, gate, mu, ctx_scores = self._fuse_type_stream(
                scores, input_ids_type, attention_mask_type)
            _out = dict(logits=scores, proj=proj)
            if d6 is not None:
                _out["digit_logits"] = d6
                _out["dist_std"] = dstd
            if gate is not None:
                # 【★ 把融合前后的量都暴露出来】这样「分数 = (1−g)·μ̂ + g·δ_ctx」
                #   这个恒等式可以在前向自检里被**逐位验证**（而不是靠注释保证）。
                _out["gate"] = gate
                _out["type_scores"] = mu
                _out["ctx_scores"] = ctx_scores
            return ModelOutput(**_out)

        # ---------------- lm_head 模式：不加任何回归头 ----------------
        # 用 _hidden() 而不是直接调 backbone ⇒ 不算全词表 logits
        h = self._hidden(input_ids, attention_mask)
        # 读出位置 = 锚点的最后一个真实 token：锚点以空格结尾（空格是独立 token），
        # 因果解码器在这个位置的输出分布正是「下一个 token」的分布 ⇒ 数字。
        pooled0 = self._pool(h, attention_mask)          # pooling=last ⇒ 取最后一个真实 token
        # 模块 4：第 0 列（修饰语）加角色向量 0
        pooled0 = self._role_pooled(pooled0, 0)
        d0 = self._digit_logits(pooled0)                 # [B, 6]
        p0 = d0.softmax(dim=-1)
        _v = self.digit_values.to(p0.dtype)
        exp0 = (p0 * _v).sum(dim=-1)                     # [B] 期望值 ∈ [0,5]
        # 分布宽度（不确定度通道）：读出本来就是 6 类分布，把宽度也读出来供诊断。
        w0 = torch.sqrt(((p0 * (_v ** 2)).sum(dim=-1) - exp0 ** 2).clamp(min=0.0))
        exps = [exp0]
        d_list = [d0]
        w_list = [w0]
        if self.n_outputs > 1:
            B = input_ids.shape[0]
            if input_ids2 is None:
                # 模块 4：交换视图只需要「第 0 列」⇒ 允许不传第 2 列的 token
                #（此时第 2 列读出恒为 0）；但确实需要第 2 列却没给 token ⇒ 必须报错。
                if need2 is None or bool(torch.as_tensor(need2).any()):
                    raise RuntimeError(
                        "lm_head 模式需要第 2 列的 token（input_ids2）。"
                        "请确认数据管线走的是 HEAD_MODE == 'lm_head' 分支。")
                need2 = torch.zeros(B, dtype=torch.bool, device=input_ids.device)
            elif need2 is None:
                need2 = torch.ones(B, dtype=torch.bool, device=input_ids.device)
            idx = need2.nonzero(as_tuple=True)[0]
            e1 = torch.zeros(B, device=exp0.device, dtype=exp0.dtype)
            d1 = torch.zeros(B, d0.shape[1], device=d0.device, dtype=d0.dtype)
            w1 = torch.zeros(B, device=exp0.device, dtype=exp0.dtype)
            if idx.numel() > 0:
                h2 = self._hidden(input_ids2[idx], attention_mask2[idx])
                pooled1 = self._pool(h2, attention_mask2[idx])
                # 模块 4：第 1 列（中心语）加角色向量 1
                pooled1 = self._role_pooled(pooled1, 1)
                d1v = self._digit_logits(pooled1)
                p1 = d1v.softmax(dim=-1)
                e1 = e1.index_copy(0, idx, (p1 * self.digit_values.to(p1.dtype)).sum(dim=-1))
                d1 = d1.index_copy(0, idx, d1v)
                w1v = torch.sqrt(((p1 * (self.digit_values.to(p1.dtype) ** 2)).sum(dim=-1)
                                  - e1[idx] ** 2).clamp(min=0.0))
                w1 = w1.index_copy(0, idx, w1v)
            exps.append(e1)
            d_list.append(d1)
            w_list.append(w1)
        logits = torch.stack(exps, dim=1)                # [B, n_out] 期望值
        digit_logits = torch.stack(d_list, dim=1)        # [B, n_out, 6]
        dist_std = torch.stack(w_list, dim=1)             # [B, n_out] 分布宽度（不确定度）
        # ---- 模块 2：与类型流融合 ----
        logits, gate, mu, ctx_scores = self._fuse_type_stream(
            logits, input_ids_type, attention_mask_type)
        _out = dict(logits=logits, digit_logits=digit_logits, dist_std=dist_std)
        if gate is not None:
            _out["gate"] = gate
            _out["type_scores"] = mu
            _out["ctx_scores"] = ctx_scores
        return ModelOutput(**_out)


def build_tiny_backbone(model_key, source, tokenizer):
    """冒烟模式：用**真实模型的 config** 缩小成 2 层随机初始化模型。
    【为什么这样做】只改层数与宽度，不改模型结构与接口，
       于是「主干前向 → 池化 → 回归头 → Trainer → 评估」整条链路都被真实覆盖，
       而下载权重与训练时间被压到秒级；没有 GPU 的机器也能跑通全流程。
    【为什么必须走 shrink_config】缩小尺寸时若不同步 attention_head_size / head_dim /
       embedding_size 这些派生宽度，模型会「建得出来但前向必炸」——
       这正是云端实跑崩掉的那一类问题。"""
    from transformers import DebertaV2Model, Qwen2Model, XLMRobertaModel
    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)
    if model_key == "qwen":
        cfg.use_cache = False
        return Qwen2Model(cfg)
    if model_key == "xlmr":
        cfg.use_cache = False
        return XLMRobertaModel(cfg)
    return DebertaV2Model(cfg)


def build_tiny_causal_lm(source, tokenizer):
    """冒烟模式（lm_head）：**带输出头**的微型因果语言模型。
    【为什么不能用 build_tiny_backbone】无回归头模式需要主干**自带**的 LM 头
       （它与 embedding 共享权重），而 AutoModel 造出来的 Qwen2Model **没有** lm_head。
    【★ 为什么随机初始化也无所谓】冒烟只验证「链路能跑通」，不验证精度；
       真正权重由云端正式运行时加载。"""
    from transformers import Qwen2ForCausalLM
    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)
    cfg.use_cache = False
    return Qwen2ForCausalLM(cfg)


# transformers 5 用 dtype=，4.x 用 torch_dtype= —— 云端版本不可控，必须探测
_AUTO_PRETRAINED_PARAMS = set(inspect.signature(AutoModel.from_pretrained).parameters)
_DTYPE_KW = "dtype" if "dtype" in _AUTO_PRETRAINED_PARAMS else "torch_dtype"
print("from_pretrained 的精度参数名:", _DTYPE_KW)


def load_backbone():
    """加载主干，带注意力实现的自动回退与显式 fp32。
    显式 fp32：transformers 5 会照 checkpoint 记录的 dtype 加载（有的是 fp16），
      不报错地以 fp16 进入微调会在 CPU/MPS 上溢出成 NaN，必须显式指定；
      参数名 5.x 是 dtype=、4.x 是 torch_dtype=（签名探测，见 Step1）。
    注意力回退：只吞「注意力实现不支持」这一类 ValueError，
      其它原样抛出——否则真 bug 会被当成环境问题静默降级。"""
    mcfg = MODEL_REGISTRY[MODEL_KEY]
    if SMOKE:
        return build_tiny_backbone(MODEL_KEY, MODEL_SOURCE, tokenizer)

    backbone, last_err = None, None
    for impl in mcfg["attn_impls"]:
        try:
            backbone = AutoModel.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},
                                                 attn_implementation=impl)
            print(f"主干加载成功: {mcfg['model_id']} | attn_implementation={impl}")
            break
        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise
            last_err = e
            print(f"attn_implementation={impl} 不支持该架构，自动回退")
    if backbone is None:
        raise last_err

    backbone.config.use_cache = False   # 训练时不需要 KV cache
    return backbone


def load_causal_lm():
    """lm_head 模式：加载带输出头的因果语言模型。
    必须是 CausalLM 而不是 AutoModel：无回归头模式要用主干自己的 LM 头读出
    「下一个 token 是数字」的概率，AutoModel 造出来的 Qwen2Model 没有 lm_head。
    显式 fp32 / 注意力回退与 load_backbone 同源。"""
    mcfg = MODEL_REGISTRY[MODEL_KEY]
    if SMOKE:
        return build_tiny_causal_lm(MODEL_SOURCE, tokenizer)

    clm, last_err = None, None
    for impl in mcfg["attn_impls"]:
        try:
            clm = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},
                                                       attn_implementation=impl)
            print(f"因果语言模型加载成功: {mcfg['model_id']} | attn_implementation={impl}")
            break
        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise
            last_err = e
            print(f"attn_implementation={impl} 不支持该架构，自动回退")
    if clm is None:
        raise last_err
    clm.config.use_cache = False
    return clm


def resolve_digit_ids(tokenizer):
    """找出 "0"–"5" 这 6 个 token 的 id，并当场验证它们是单 token。
    整套读出机制建立在「模型在锚点处生成的下一个 token 就是数字本身」上：
    若分词器把 " 0" 合并成一个 token（BPE 常见 "Ġ0"），读到的分布就错了且不会报错
    ⇒ 必须断言「数字 = 单 token」「空格与数字不合并」「锚点以空格结尾」。"""
    ids = []
    for d in range(6):
        _s = str(d)
        _enc = tokenizer(_s, add_special_tokens=False).input_ids
        assert len(_enc) == 1, (
            f"数字 {_s!r} 不是单 token（编码成 {_enc}）："
            "无回归头模式的读出机制依赖「数字 = 一个 token」，否则分布读错了且不会报错。")
        ids.append(int(_enc[0]))
    # 验证「空格 + 数字」不会被合并成一个 token
    for d in range(6):
        _enc = tokenizer(" " + str(d), add_special_tokens=False).input_ids
        assert len(_enc) == 2 and int(_enc[1]) == ids[d], (
            f"' {d}' 的编码 {_enc} 说明空格与数字被合并了（期望 [空格, 数字]）："
            "锚点读出的下一个 token 就不再是数字本身。")
    # 验证锚点确实以「空格」结尾（下一个 token 才是数字）
    for c in READOUT_CUES:
        _enc = tokenizer(c, add_special_tokens=False).input_ids
        assert tokenizer.decode([_enc[-1]]) == " ", (
            f"锚点 {c!r} 的最后一个 token 解码后不是空格（是 {tokenizer.decode([_enc[-1]])!r}）："
            "锚点必须以空格结尾，模型生成的下一个 token 才是数字。")
    print(f"数字 token id = {ids} | 锚点 = {READOUT_CUES}（均以空格结尾 ✓）")
    return ids


_DIGIT_IDS_CACHE = None


def get_digit_ids(tokenizer):
    """取数字 token id（只算一次，之后复用）。
    【为什么缓存】build_model 每次运行都会被调用 5～10 次（5 次运行 + 探测 + 重建），
       每次都重新断言一遍只是噪音；但**断言必须保留在第一次**（那时才发现问题）。"""
    global _DIGIT_IDS_CACHE
    if _DIGIT_IDS_CACHE is None:
        _DIGIT_IDS_CACHE = resolve_digit_ids(tokenizer)
    return _DIGIT_IDS_CACHE


# 权重格式：deberta-v2-xlarge 仓库只有 pytorch_model.bin，Qwen2.5 是 safetensors；
# 两者都走同一条 from_pretrained 路径，无需特殊处理。

In [ ]:
# ============ 训练档位：只训练顶部若干层（显存不够时的最后一档）============
FIT_SCOPE_FRAC = 1.0   # 1.0 = 全参微调；< 1.0 = 只训练顶部这一比例的 transformer 层


def _find_layer_list(backbone):
    """找到主干里的 transformer 层列表（DeBERTa 与 Qwen 的存放路径不同）。"""
    for path in ("encoder.layer", "layers", "layer", "decoder.layers", "encoder.decoder.layer"):
        obj = backbone
        for part in path.split("."):
            if not hasattr(obj, part):
                obj = None
                break
            obj = getattr(obj, part)
        if obj is not None and isinstance(obj, (nn.ModuleList, list)) and len(obj) > 0:
            return obj
    return None


def apply_fit_scope(model, scope_frac=None):
    """按档位冻结底部的 transformer 层，只训练顶部若干层。
    【为什么需要这一档】全参微调的固定显存开销 =
        权重 4 B/param（**全部**参数都要占显存） + 梯度与优化器状态 12 B/param（只算可训部分）。
        只训顶部一半的层 → 16 B/param 降到 4 + 12×0.5 = 10 B/param。
    【为什么必须做成单一入口】冻结与否直接决定「哪些参数有梯度」，
        如果有的运行冻了、有的没冻，两次运行之间就不可比了。
        所以只能由 build_model() 统一调用，别的地方不许私自改 requires_grad。"""
    frac = FIT_SCOPE_FRAC if scope_frac is None else scope_frac
    if frac >= 1.0:
        return model
    layers = _find_layer_list(model.backbone)
    if layers is None:
        print(f"⚠️ 找不到主干的层列表，本档位（只训顶部 {frac:.0%} 层）无法生效 → 退回全参微调")
        return model
    n_total = len(layers)
    n_keep = max(1, int(round(n_total * frac)))
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    for layer in layers[n_total - n_keep:]:
        for p in layer.parameters():
            p.requires_grad_(True)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"档位生效：只训练顶部 {n_keep}/{n_total} 层 | 可训练参数 {n_tr / 1e6:.1f}M / "
          f"{n_all / 1e6:.1f}M ({n_tr / n_all:.1%})")
    return model


def ensure_readout_trainable(model):
    """保证「读出通路」永远可训（lm_head 模式专用）。
    绑定时读出权重就是 embedding 本身：apply_fit_scope 冻结 backbone 会连它一起冻住，
    读出通路学不到任何东西，且不报错、只让指标停在随机水平 ⇒ 必须显式解冻并打印。"""
    if getattr(model, "lm_head", None) is None and getattr(model, "head", None) is not None:
        return model      # 回归头模式：读出权重就是 Linear，不会被冻结
    if getattr(model, "head", None) is not None and getattr(model, "lm_head", None) is None:
        return model      # 回归头模式
    if model.head_mode != "lm_head":
        return model
    w = model.readout_weight()
    w.requires_grad_(True)
    _n = w.numel()
    assert _n > 0, "读出权重被冻结了：读出通路学不到东西（无回归头模式会退化成随机读出）"
    print(f"读出通路已确认可训练：读出权重 {_n / 1e6:.2f}M"
          f"（{'与 embedding 共享同一张量 ⇒ embedding 同时可训' if model.tied_to_embedding else '独立的 LM 头'}）")
    return model

In [ ]:
# ============ 装配并体检 ============
def build_model():
    """构造一个全新的读出模型（随机初始化）。
    每次运行都必须从干净的随机初始化开始（冒烟与显存探针都会污染权重）。
    两条基线的读出方式不同，但装配入口只有这一个：
    regression = 主干 + 线性回归头；lm_head = 主干 + 自带的 LM 头。"""
    _use_proj = any(float(v.get("softcl", 0.0)) > 0 for v in SUBSET_LOSS_PROFILE.values())
    if HEAD_MODE == "lm_head":
        clm = load_causal_lm()
        _digit_ids = get_digit_ids(tokenizer)
        # 绑定检测用 data_ptr 比较，而不是只信 config.tie_word_embeddings：
        # 「配置说绑定」与「权重真的共享同一块显存」是两件事。
        _tied = False
        try:
            _tied = (clm.lm_head.weight.data_ptr() == clm.model.embed_tokens.weight.data_ptr())
        except Exception:
            _tied = bool(getattr(clm.config, "tie_word_embeddings", False))
        print(f"读出权重绑定检测：lm_head.weight 与 embed_tokens.weight "
              f"{'共享同一块显存（⇒ 读出直接用 embedding 矩阵，避免 safetensors 共享张量报错）' if _tied else '不共享（⇒ 直接挂 lm_head 子模块）'}")
        m = CompositionalityRegressor(
            backbone=clm.model,                  # 主干（Qwen2Model）
            hidden_size=clm.config.hidden_size,
            n_outputs=N_OUTPUTS,
            pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
            head_mode="lm_head",
            lm_head=clm.lm_head,                 # ★ 主干自带的输出头（与 embedding 共享权重）
            digit_ids=_digit_ids,
            digit_values=DIGIT_VALUES,
            proj_dim=None,
            tied_to_embedding=_tied,
            # lm_head 模式下读出本来就是 6 类分布 ⇒ 这里传 False，不新建任何输出层。
            m1_dist_head=False,
            # 模块 2：双流开关
            m2_dual_stream=M2_DUAL_STREAM,
            # 模块 4：角色嵌入开关
            m4_role_emb=M4_ROLE_EMB,
        ).to(DEVICE)
    else:
        bb = load_backbone()
        m = CompositionalityRegressor(
            backbone=bb,
            hidden_size=bb.config.hidden_size,
            n_outputs=N_OUTPUTS,
            pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
            head_mode="regression",
            # 投影头只在真的要用 softcl 时才建，否则多出一份没人用的参数
            proj_dim=(SOFTCL_PROJ_DIM if _use_proj else None),
            # 模块 1：回归头输出 6 类 logits + 期望值读数
            m1_dist_head=M1_DIST_HEAD,
            # 模块 2：双流开关
            m2_dual_stream=M2_DUAL_STREAM,
            # 模块 4：角色嵌入开关
            m4_role_emb=M4_ROLE_EMB,
        ).to(DEVICE)   # 【坑】显式 .to(DEVICE)：transformers 5 会把模型放到默认加速器上
    apply_fit_scope(m)      # 【单一入口】按当前档位决定冻结范围
    ensure_readout_trainable(m)   # 【单一入口】读出通路必须可训
    return m


model = build_model()
n_all = sum(p.numel() for p in model.parameters())
if HEAD_MODE == "regression":
    n_head = sum(p.numel() for p in model.head.parameters())
    if getattr(model, "m1_dist_head", False):
        _head_txt = (f"序数-分布头 {n_head / 1e3:.1f}K（Linear(H→{N_OUTPUTS}×{M1_CLASSES})，"
                     f"读数 = Σ k·p(k)）")
    else:
        _head_txt = f"回归头 {n_head / 1e3:.1f}K ({n_head / n_all:.4%})"
else:
    _n_lm = sum(p.numel() for p in model.readout_weight())
    _head_txt = f"**无回归头**（用主干自带的 LM 头 {_n_lm / 1e6:.1f}M" \
                + ("，与 embedding 共享权重" if model.tied_to_embedding else "）")
print(f"主干       : {MODEL_REGISTRY[MODEL_KEY]['model_id']}（SMOKE={SMOKE}）")
print(f"读出方式   : {HEAD_MODE} | {_head_txt}")
print(f"结构模块: 模块1(序数-分布头)={M1_DIST_HEAD} | "
      f"模块2(类型-语境双流+敏感度门)={M2_DUAL_STREAM} | "
      f"模块4(角色嵌入+交换一致性)={M4_ROLE_EMB}")
if M4_ROLE_EMB:
    _n_role = sum(p.numel() for p in model.role_emb.parameters())
    print(f"角色嵌入   : {model.role_emb.weight.shape[0]} 个角色 × "
          f"{model.role_emb.weight.shape[1]} 维（{_n_role / 1e3:.1f}K 参数，"
          f"占总参数 {_n_role / n_all:.4%}）")
print(f"池化方式   : {MODEL_REGISTRY[MODEL_KEY]['pooling']}")
print(f"输出维度   : {N_OUTPUTS}（统一 2 列）")
print(f"总参数     : {n_all / 1e6:.1f}M | {_head_txt}")
if model.proj is not None:
    print(f"投影头     : {SOFTCL_PROJ_DIM} 维（仅供 softcl 对比学习使用）")
print(f"设备       : {DEVICE}")

# 一次前向自检：确认形状、确认没有 NaN
model.eval()
with torch.no_grad():
    _b = collator([_demo_run["train_ds"][i] for i in range(2)])
    _out = model(input_ids=_b["input_ids"].to(DEVICE),
                 attention_mask=_b["attention_mask"].to(DEVICE),
                 input_ids2=_b["input_ids2"].to(DEVICE) if "input_ids2" in _b else None,
                 attention_mask2=_b["attention_mask2"].to(DEVICE) if "attention_mask2" in _b else None,
                 input_ids_type=_b["input_ids_type"].to(DEVICE) if "input_ids_type" in _b else None,
                 attention_mask_type=_b["attention_mask_type"].to(DEVICE) if "attention_mask_type" in _b else None,
                 need2=(_b["loss_mask"][:, 1] > 0).to(DEVICE) if "loss_mask" in _b else None)
print("前向 logits shape:", tuple(_out.logits.shape))
print("是否含 NaN:", bool(torch.isnan(_out.logits).any().item()))
# 模块 1 的结构自检：期望值读数落在 [0,5]、6 类 logits 形状正确、
# 期望值确实等于 Σ k·p(k)（否则读数与分布对不上，分布宽度诊断就是假的）。
if getattr(model, "m1_dist_head", False) or HEAD_MODE == "lm_head":
    _dl = _out.digit_logits
    _p = _dl.softmax(dim=-1)
    _exp = (_p * model.digit_values.to(_p.dtype)).sum(dim=-1)
    assert _dl.shape[-1] == M1_CLASSES, f"分布头类别数必须是 {M1_CLASSES}，实际 {_dl.shape[-1]}"
    # 【★ 注意校验对象】有双流时，最终分数是**融合后**的 (1−g)·μ̂ + g·δ_ctx，
    #   它**不等于** ctx 分布的期望值 ⇒ 必须分别校验「ctx 读数 = 期望值」与「融合恒等式」。
    _ctx_ref = getattr(_out, "ctx_scores", None)
    _ctx_ref = _out.logits if _ctx_ref is None else _ctx_ref
    assert torch.allclose(_exp, _ctx_ref, atol=1e-5), (
        "语境流读数与分布不一致：分数不是 Σ k·p(k) 读出来的")
    assert float(_ctx_ref.min()) >= 0.0 and float(_ctx_ref.max()) <= 5.0, (
        "语境流读数必须落在 [0,5]")
    _ws = getattr(_out, "dist_std", None)
    print(f"分布头自检 ✓ 类别数 {_dl.shape[-1]} | 期望值 == Σ k·p(k) | "
          f"分布宽度范围 [{float(_ws.min()):.3f}, {float(_ws.max()):.3f}]"
          if _ws is not None else "分布头自检 ✓")
# 模块 2 的结构自检：门 g ∈ [0,1]、融合后分数在 [0,5]、融合恒等式逐位成立。
if M2_DUAL_STREAM:
    _g = getattr(_out, "gate", None)
    _mu = getattr(_out, "type_scores", None)
    _cx = getattr(_out, "ctx_scores", None)
    assert _g is not None, "模块 2 已打开，但前向没有返回 gate ⇒ 融合没接上"
    assert _mu is not None and _cx is not None, "模块 2：前向没有返回 type_scores / ctx_scores"
    assert float(_g.min()) >= 0.0 and float(_g.max()) <= 1.0, (
        f"门 g 必须落在 [0,1]，实际 [{float(_g.min()):.3f}, {float(_g.max()):.3f}]")
    assert float(_out.logits.min()) >= 0.0 and float(_out.logits.max()) <= 5.0, (
        "双流融合后的分数必须落在 [0,5]（凸组合保证，不靠裁剪）")
    _fused = (1.0 - _g) * _mu + _g * _cx
    assert torch.allclose(_fused, _out.logits, atol=1e-5), (
        "融合恒等式不成立：分数 ≠ (1−g)·μ̂_type + g·δ_ctx")
    print(f"模块 2 自检 ✓ 门 g 形状 {tuple(_g.shape)} | "
          f"取值 [{float(_g.min()):.3f}, {float(_g.max()):.3f}] | "
          f"融合后分数 ∈ [{float(_out.logits.min()):.3f}, {float(_out.logits.max()):.3f}] | "
          f"融合恒等式逐位成立")
if HEAD_MODE == "lm_head":
    # 【★ 无回归头模式的自检】读出的必须是**合法的分数**而不是别的数字
    print("lm_head 读出样例:", _out.logits.detach().cpu().tolist())
    print("数字 token 分布 shape:", tuple(_out.digit_logits.shape),
          "（每列 6 个数字 token 的 logits）")
    assert float(_out.logits.min()) >= 0.0 and float(_out.logits.max()) <= 5.0, (
        "期望值读出必须落在 [0,5]：它是 6 个数字的概率加权平均，不可能越界")

## Step5 创建评估函数

**这一节要解决什么**：把官方的三项指标**逐字实现**出来，并让本地实现和
未来用 Trial 的 `scores.txt` 对得上。

| 指标 | 官方口径 | 一句话含义 |
|---|---|---|
| 指标 1 | 组合性相关 ρ | 全部例句上 Spearman(预测, 真值)——整体排序对不对 |
| 指标 2 | 语境方差相关 ρ | 每个目标跨例句算 std，再在**目标之间** Spearman(预测 std, 人工 std)——模型有没有真的「因句而异」 |
| 指标 3 | RMSE | 全部例句上的均方根误差——数值校准准不准 |

**指标 2 为什么是硬门槛**：如果模型对同一目标的所有例句给出同一个分数，
那它的组内 std 恒为 0，相关系数的输入数组是常数 → **相关系数未定义（nan）**。
所以指标 2 不是「加分项」，而是**及格线**：一个常数输出系统直接出局。

**指标 1 与指标 2 存在张力**：指标 1 奖励跨目标的数值校准，
指标 2 奖励目标内部的离散程度。过度压缩输出方差 → 指标 2 掉分；
过度放大方差 / 加噪声 → 指标 1 的排序被破坏、RMSE 变差。
本 notebook 不做后处理调参（那属于刷分），而是让模型真正对语境敏感。

**这一节末尾有一组断言自检**：用「完美预测 / 常数预测 / 反向预测」三种人造系统
把指标函数验一遍。指标实现一旦写错，训练完才发现就太晚了。

In [ ]:
# ============ 演示：为什么 type-level（常数输出）系统会直接出局 ============
# 【为什么先演示】这是本任务最有辨识度的地方，值得用真实数据讲清楚。
_demo_v = _demo_run["val_df"]
_demo_key = SUBSET_SPECS["en-nn"]["key"]
_type_level_pred = _demo_v.groupby(_demo_key)["ModAvg"].transform("mean")   # 每个目标一个常数
_gold = _demo_v["ModAvg"].to_numpy()
_g = _demo_v[_demo_key].to_numpy()

_df = pd.DataFrame({"g": _g, "p": _type_level_pred.to_numpy(), "y": _gold})
_ps = _df.groupby("g")["p"].std(ddof=1)      # 组内预测 std：恒为 0
_gs = _df.groupby("g")["y"].std(ddof=1)      # 组内人工 std：有值
_ok = _ps.notna() & _gs.notna()
print(f"type-level 基线：指标1 Spearman = {stats.spearmanr(_type_level_pred, _gold)[0]:.3f}"
      f" | 指标3 RMSE = {np.sqrt(((_type_level_pred.to_numpy() - _gold) ** 2).mean()):.3f}")
print(f"组内预测 std 的唯一取值个数 = {_ps[_ok].nunique()}（0 表示每个目标内完全相同）")
print("→ 指标 2 的输入是常数数组，相关系数**未定义（nan）**：该系统在指标 2 上直接出局。")
print("→ 这就是为什么本任务必须「逐例句预测」，不能先聚合成目标级分数再回归。")

In [ ]:
# ============ 封装：三项指标 ============
def strip_eval_prefix(metrics):
    """Trainer.evaluate() 会给所有返回的指标加上 'eval_' 前缀，
    这里统一剥掉，好让它们和 compute_metrics 里的原始键名对齐。"""
    return {k[len("eval_"):] if k.startswith("eval_") else k: v for k, v in metrics.items()}


def _three_metrics(p, g, grp):
    """在一个「子集 × 标签列」的切片上算官方三项指标。
    指标1 组合性相关  ：全部例句上 Spearman(预测, 真值)
    指标2 语境方差相关：每个目标跨例句算 std，再在目标之间 Spearman(预测 std, 人工 std)
    指标3 RMSE        ：全部例句上的均方根误差
    【指标 2 的三个边界条件，缺一不可】
      · 组内至少 2 条例句，std 才有定义；
      · 人工 std 与预测 std 都必须非空；
      · 预测 std 必须**不全是同一个值**，否则相关系数未定义（返回 nan 而不是 0）。
    【为什么用 Spearman 而不是 Pearson】官方协议用秩相关：
       组合性评分是有界的离散均值，秩相关对个别离群点更稳健；
       而指标 2 比较的是「离散程度的大小」，本身就不是正态分布，秩相关更合适。"""
    # 常数输入的相关系数本来就没定义（指标 2 的 nan 就来自这里），
    # scipy 会发一条 ConstantInputWarning，这里局部静音，避免刷屏。
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConstantInputWarning)
        m1 = float(stats.spearmanr(p, g)[0])
        frame = pd.DataFrame({"g": grp, "p": p, "y": g})
        ps = frame.groupby("g")["p"].std(ddof=1)
        gs = frame.groupby("g")["y"].std(ddof=1)
        ok = ps.notna() & gs.notna()
        if ok.sum() > 2 and ps[ok].nunique() > 1:
            m2 = float(stats.spearmanr(ps[ok], gs[ok])[0])
        else:
            m2 = float("nan")
    rmse = float(np.sqrt(((p - g) ** 2).mean()))
    return m1, m2, rmse, int((ps[ok] == 0).sum())


def compute_compartment_metrics(pred, gold, group_index, subset_ids, eval_cols):
    """按「子集 × 标签列」分别算三项指标，并给出各设置值的平均（「合计」）。
    【为什么必须按子集分开算】四个子集的语言、任务、标签难度完全不同
       （德语复合词修饰语 79.4% 落在 [4,5]，英语小品词动词均值只有 2.46），
       混在一起算一个总 ρ 会被大子集主导，既看不出哪个设置出了问题，
       也无法与官方「6 个评测设置」逐项对照。
    【「合计」怎么算】取各评测设置指标的**平均**。
       【为什么不能把各设置的行拼起来算】官方是 6 个评测设置分别打分，没有「总分」。
       如果把不同标签列的行拼成一组算，同一目标的「修饰语列」和「中心语列」会被
       并进同一个组，组内 std 被人为撑大，指标 2 就不再是官方口径
       （实跑时正是这样把 type-level oracle 的 nan 算成了 0.96）。
       所以合计一律取各设置值的平均，nan 的设置自动跳过。"""
    pred = np.asarray(pred, dtype=float).reshape(len(pred), -1)
    gold = np.asarray(gold, dtype=float).reshape(len(gold), -1)
    group = np.asarray(group_index)
    subset_ids = np.asarray(subset_ids)
    out = {}
    _m1s, _m2s, _rmses, _ndegs = [], [], [], []
    for name, cname, j in eval_cols:
        m = subset_ids == name
        if m.sum() < 3:
            continue
        m1, m2, rmse, ndeg = _three_metrics(pred[m, j], gold[m, j], group[m])
        out[f"{name}|{cname}_指标1_ρ"] = m1
        out[f"{name}|{cname}_指标2_ρ"] = m2
        out[f"{name}|{cname}_指标3_RMSE"] = rmse
        out[f"{name}|{cname}_组内std退化目标数"] = ndeg
        out[f"{name}|{cname}_样例数"] = int(m.sum())
        _m1s.append(m1); _m2s.append(m2); _rmses.append(rmse); _ndegs.append(ndeg)

    def _nanmean(vals):
        """对 nan 安全的平均：nan 的设置不参与平均，全 nan 时返回 nan。"""
        v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
        return float(v.mean()) if len(v) else float("nan")

    out["合计|_指标1_ρ"] = _nanmean(_m1s)
    out["合计|_指标2_ρ"] = _nanmean(_m2s)
    out["合计|_指标3_RMSE"] = _nanmean(_rmses)
    out["合计|_组内std退化目标数"] = int(np.sum(_ndegs)) if _ndegs else 0
    out["合计|_评测设置数"] = len(_m1s)
    out["合计|_样例数"] = int(len(pred))
    out["合计|_目标数"] = int(len(np.unique(group)))
    # ---- 综合分（best checkpoint 准则）：z 分数空间等权求和 ----
    # 三项各自除以运行间噪声 sd（Z_M1/Z_M2/Z_M3），在「可检测性」上等权；
    # 指标 2 为 nan → 按 −1 计（对同一目标给同一分 = 出局语义）⇒ 必然选不中。
    _m1 = out["合计|_指标1_ρ"]
    _m2 = out["合计|_指标2_ρ"]
    _m3 = out["合计|_指标3_RMSE"]
    _c1 = 0.0 if not np.isfinite(_m1) else float(_m1)
    _c2 = float(_m2) if np.isfinite(_m2) else -1.0
    _c3 = float(_m3) if np.isfinite(_m3) else float("nan")
    out[BEST_METRIC_KEY] = (_c1 / Z_M1) + (_c2 / Z_M2) - (_c3 / Z_M3 if np.isfinite(_c3) else 0.0)
    return out


def make_compute_metrics(eval_group_index, eval_subset_index, eval_cols, subset_names):
    """生成 Trainer 的 compute_metrics 钩子。
    【为什么闭包里要带 group_index 和 subset_index】指标 2 需要知道
      「哪些例句属于同一个目标」「每条例句来自哪个子集」。
      Trainer.predict 按顺序吐出预测，验证集不打乱，顺序与闭包里的数组一致。"""
    groups = np.asarray(eval_group_index)
    # 【为什么这里要把编号映射成子集名】指标函数是按**子集名**分组的，
    #   传进来的却是 subset_index（整数），直接比较会一个都匹配不上，
    #   结果是所有按子集的指标全变成 nan（这个坑在实跑时被抓到过）。
    # 【★ 越界检查】subset_index 是**全局**编号（0..3），
    #   所以 subset_names 必须是**全局**的 SUBSET_ORDER；
    #   若误传「本次运行参与的子集名」，单子集运行就会越界 → IndexError。
    _max_id = int(np.max(eval_subset_index)) if len(eval_subset_index) else -1
    assert _max_id < len(subset_names), (
        f"subset_index 最大值 {_max_id} 超出 subset_names 长度 {len(subset_names)}："
        "subset_index 是全局编号，subset_names 必须传全局的 SUBSET_ORDER")
    subsets = np.asarray([subset_names[i] for i in eval_subset_index])

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        preds = np.asarray(preds)
        labels = np.asarray(labels)
        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)
        if labels.ndim == 1:
            labels = labels.reshape(-1, 1)
        return compute_compartment_metrics(preds, labels, groups, subsets, eval_cols)

    return compute_metrics

In [ ]:
# ============ 指标函数的断言自检：三组人造系统，错了就当场崩 ============
# 【为什么必须写成断言而不是 print】指标实现是整套管线的「测量仪器」。
#   仪器错了，训练得再好也只是在优化一个错误的目标。
#   三组人造系统的期望值是数学上确定的，可以直接断言。
_gold = ds_labels(_demo_run["val_ds"])
_grp = np.asarray(_demo_run["val_ds"]["group_index"])
_sub = np.asarray(["en-nn"] * len(_grp))
_cols = _demo_run["eval_cols"]

# ① 完美预测：指标1 = 1、指标2 = 1、RMSE = 0
_m = compute_compartment_metrics(_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    _k3 = f"{_c[0]}|{_c[1]}_指标3_RMSE"
    assert abs(_m[_k1] - 1.0) < 1e-9, f"完美预测的指标1 应为 1，实际 {_m[_k1]}"
    assert abs(_m[_k2] - 1.0) < 1e-9, f"完美预测的指标2 应为 1，实际 {_m[_k2]}"
    assert _m[_k3] < 1e-9, f"完美预测的 RMSE 应为 0，实际 {_m[_k3]}"
print("① 完美预测      : 指标1=1.000 指标2=1.000 RMSE=0.000  ✓")

# ② 常数预测：相关系数的输入是常数数组 → 指标1 与指标2 都必须是 nan（不是 0）
_m = compute_compartment_metrics(np.full_like(_gold, 3.0), _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k1]), f"常数预测的指标1 应为 nan，实际 {_m[_k1]}"
    assert np.isnan(_m[_k2]), f"常数预测的指标2 应为 nan，实际 {_m[_k2]}"
print("② 常数预测      : 指标1=nan 指标2=nan（未定义）              ✓")

# ③ 反向预测：指标1 = -1、RMSE 与真值对称
_m = compute_compartment_metrics(-_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    assert abs(_m[_k1] + 1.0) < 1e-9, f"反向预测的指标1 应为 -1，实际 {_m[_k1]}"
print("③ 反向预测      : 指标1=-1.000                              ✓")

# ④ 人工标准差恒为 0 的切片 → 指标2 必须 nan（被解释变量无方差，相关系数未定义）
_flat = np.full_like(_gold, 2.5)
_m = compute_compartment_metrics(_gold, _flat, _grp, _sub, _cols)
for _c in _cols:
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k2]), f"人工 std 恒为 0 时指标2 应为 nan，实际 {_m[_k2]}"
print("④ 人工 std 恒为 0: 指标2=nan（被解释变量无方差）           ✓")
print("→ 指标实现通过四组断言，接下来所有结果都由它产出。")

## Step6 配置训练参数

**这一节要解决什么**：把超参组装成一个**函数**，而不是散落的字面量。

**为什么必须是函数**：transformers 的 `Trainer` 会**就地改写** `TrainingArguments` 对象
（`per_device_train_batch_size`、`max_steps`、`label_names`、`_n_gpu` 都会被动过）。
如果两个 Trainer 共用同一个 args 对象，第二个 Trainer 拿到的就是「已经被训练过」的脏 args，
它的 `num_batches` 与实际 dataloader 长度对不上，直接抛
`ValueError: zip() argument 2 is longer than argument 1`。
所以「超参相同」的正确实现是：**每次调用 `build_training_args()` 重新构造一个对象**，
字段默认值相同即等价，而不是复用同一个变量。

**超参取值与理由**：

| 超参 | 值 | 为什么 |
|---|---|---|
| `learning_rate` | 2e-5 | 微调大模型的常规档位。两个模型**用同一个值**，对照实验里 lr 不成为混淆变量 |
| `per_device_train_batch_size` | 8 | 序列长 160、模型 900M～1.5B；显存不够时用 `gradient_accumulation_steps` 把有效 batch 补回去（数学等价、几乎无损，第一选择） |
| `num_train_epochs` | 3 | 本项目每个目标只有 5–10 条例句，全量数据仅 1.5k–3.5k 条；轮数太多会过拟合到「记住句子」 |
| `warmup_steps` | 总步数 × 10% | transformers 5 **删除了 `warmup_ratio`**，只保留绝对步数，所以这里自己算 |
| `lr_scheduler_type` | linear | 线性衰减是回归任务的稳定默认；余弦/常数留到消融时再比 |
| `weight_decay` | 0.01 | 常规正则量级 |
| `eval_strategy` / `save_strategy` | epoch | transformers 5 把 `evaluation_strategy` 改名为 `eval_strategy` |
| `load_best_model_at_end` | True | 必须与 `save_strategy="epoch"` 同时满足，否则初始化阶段直接抛 ValueError |

In [ ]:
# ============ 封装：训练参数 ============
# 【为什么要做版本兼容】Kaggle 上 transformers 的版本不受我们控制：
#   5.x 用 eval_strategy，4.x 用 evaluation_strategy；
#   5.x 的 Trainer 用 processing_class 传 tokenizer，4.x 用 tokenizer；
#   5.x 的 from_pretrained 用 dtype=，4.x 用 torch_dtype=（Step4 已探测）。
#   这里用签名探测自动选，避免在云端因为版本差异直接崩掉。
_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
_EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
_TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)
_TOKENIZER_KW = "processing_class" if "processing_class" in _TRAINER_PARAMS else "tokenizer"
_HAS_SAVE_ONLY_MODEL = "save_only_model" in _TA_PARAMS
print("transformers 兼容层:", {"eval_key": _EVAL_STRATEGY_KEY, "tokenizer_kw": _TOKENIZER_KW,
                                "save_only_model": _HAS_SAVE_ONLY_MODEL})


# ============ 训练「档位」体系：按显存自动降级；默认写死 t3 ============
def _bnb_available():
    """bitsandbytes 是否可用（8bit 优化器的前提）。"""
    try:
        import bitsandbytes  # noqa: F401
        return True
    except Exception:
        return False


_HAS_BNB = _bnb_available()


def _use_8bit_allowed():
    """8bit 优化器档位是否可用。
    【★ 判定只依赖「环境」，不依赖「模型」】同一个环境里解析出的 optim 必然相同
       ⇒ 不会出现「一条基线 fp32、另一条 8bit」的无声分叉。
       "auto" 的语义就是「环境里有 bitsandbytes 就允许 8bit 档」——
       同一环境里解析结果必然相同 ⇒ 口径可复现。"""
    mode = USE_8BIT_OPT
    if isinstance(mode, str):
        low = mode.strip().lower()
        if low in ("true", "1", "yes", "8bit"):
            return True
        if low in ("false", "0", "no", "fp32"):
            return False
        return _HAS_BNB
    return bool(mode)


# 【档位表】从「最标准」排到「最省显存」；第一个装得下的就是本次采用的档位。
#
# 【显存模型】每参数的显存开销 =
#     权重 fp32 4 B/param
#   + （开混合精度时）fp16 权重副本 2 B/param（反向传播要用它算 grad_input）
#   + 梯度 fp32 × 可训比例 4 B/param
#   + 优化器状态 × 可训比例（AdamW 8 / 8bit AdamW 2 / Adafactor ≈0.2）
#
# 【「装得下」的判据用 bs=1 的最小激活（FIT_ACT_MIN_GB）】档位负责降固定开销，
#   激活是 batch 的事、交给探测阶段降 batch；用 bs=8 判会误杀「固定开销装得下、
#   只是 batch 要调小」的档位。
#
# 【前提：单卡模式】公式默认进程只看见 1 张卡；多卡时 Trainer 会自动套
#   nn.DataParallel，其反向会额外复制一整份梯度（显存翻倍）。
#   Step1 用 CUDA_VISIBLE_DEVICES 限制单卡，后面再钉 args._n_gpu = 1 兜底。
FIT_ACT_MIN_GB = 0.3     # bs=1 + gradient checkpointing 时的激活显存（含碎片余量）

FIT_TIERS = [
    {"name": "t0", "label": "全参微调 · fp32 AdamW · AMP", "optim": "adamw_torch",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t1", "label": "全参微调 · 8bit AdamW · AMP", "optim": "adamw_bnb_8bit",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t2", "label": "全参微调 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t3", "label": "全参微调 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t4", "label": "顶部 50% 层 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t5", "label": "顶部 50% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t6", "label": "顶部 25% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.25, "ckpt": True, "act_gb": 1.5},
]

_OPTIM_STATE_BYTES = {"adamw_torch": 8.0, "adamw_bnb_8bit": 2.0, "adafactor": 0.2}


def tier_fixed_bytes(tier):
    """每参数的**固定**显存开销（bytes），与 batch 大小无关。
    【为什么固定开销是关键】权重 4 B/param + 梯度 4 B/param 是硬的；
    剩下的全是优化器状态与 AMP 的 fp16 副本。
    所以「降 batch / 加梯度累积」能省的只是激活，**救不了这一块**。"""
    w = 4.0                                   # fp32 权重（全部参数都占显存）
    amp = 2.0 if tier["amp"] else 0.0         # AMP 的 fp16 权重副本（反向要用）
    st = _OPTIM_STATE_BYTES[tier["optim"]]    # 优化器状态
    return w + amp + tier["scope"] * (4.0 + st)


def tier_estimate_gb(n_params, tier, act_gb=None):
    """返回 (固定开销 GiB, 加上激活后的 GiB)。"""
    fixed = n_params * tier_fixed_bytes(tier) / (1024 ** 3)
    act = tier["act_gb"] if act_gb is None else act_gb
    return fixed, fixed + act


def _tier_available(tier):
    """该档位在当前环境是否可用（只与环境有关，与模型无关）。"""
    if tier["optim"] == "adamw_bnb_8bit":
        return _use_8bit_allowed() and _HAS_BNB
    return True


def _vram_total_gb():
    """当前**正在用**的那张卡的容量。
    【为什么不用 device 0】本 notebook 可以通过 GPU_INDEX 指定用哪张卡
    （两条基线同时跑时各占一张），显存预算必须按**实际占用的那张卡**算。"""
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / (1024 ** 3)


def resolve_fit_tier(vram_gb, ref_params):
    """选出「装得下」的最靠前档位，并打印全部档位的估算。

    auto 模式用参考参数量（两条基线中较大者）统一选档，保证两条落在同一档；
    默认 FIT_MODE="t3" 写死，这条自动路径只是兜底。
    估算只是候选：激活、碎片、CUDA context 都没算进去，
    真正的判定由 Step8 的「真跑 2 步」完成，OOM 自动降下一档。"""
    if vram_gb is None:
        return FIT_TIERS[0], []      # 非 CUDA 环境：不做任何降级，保持标准全参微调
    budget = vram_gb * FIT_SAFETY
    rows, chosen = [], None
    for t in FIT_TIERS:
        if not _tier_available(t):
            rows.append((t["name"], t["label"], None, None, None,
                         "不可用（需要 bitsandbytes）" if t["optim"] == "adamw_bnb_8bit" else "不可用"))
            continue
        # 判据用「最小激活」：档位负责降固定开销，激活交给探测阶段降 batch
        fixed, need_min = tier_estimate_gb(ref_params, t, act_gb=FIT_ACT_MIN_GB)
        _fixed8, need8 = tier_estimate_gb(ref_params, t)
        ok = need_min <= budget
        rows.append((t["name"], t["label"], fixed, need_min, need8,
                     "✓ 装得下" if ok else f"✗ 超出预算 {budget:.1f} GiB"))
        if ok and chosen is None:
            chosen = t
    # 显式指定档位时优先服从用户
    if FIT_MODE != "auto":
        forced = next((t for t in FIT_TIERS if t["name"] == str(FIT_MODE).lower()), None)
        if forced is not None and _tier_available(forced):
            chosen = forced
    if chosen is None:
        # 一档都装不下：取最后一档（探测阶段会给出精确诊断并报错，而不是悄悄跑）
        avail = [t for t in FIT_TIERS if _tier_available(t)]
        chosen = avail[-1] if avail else FIT_TIERS[0]
    return chosen, rows


FIT_TIER_NAME = FIT_TIERS[0]["name"]
OPTIM = FIT_TIERS[0]["optim"]
USE_GRAD_CKPT = False
FIT_SCOPE_FRAC = 1.0


def apply_tier(tier):
    """把档位写进全局变量：5 次运行必须用同一套设置。
    【★ 混合精度也是档位的一部分】开 AMP 会多占 2 B/param（fp16 权重副本），
       但训练快得多；关 AMP 省下这 2 B/param，代价是纯 fp32 训练慢（T4 没有 TF32）。
       这一项由档位统一决定 —— 5 次运行之间不能一个开 AMP 一个不开（否则曲线不可比）。"""
    global OPTIM, USE_GRAD_CKPT, FIT_SCOPE_FRAC, FIT_TIER_NAME, USE_BF16, USE_FP16
    OPTIM = tier["optim"]
    USE_GRAD_CKPT = tier["ckpt"]
    FIT_SCOPE_FRAC = tier["scope"]
    FIT_TIER_NAME = tier["name"]
    if tier["amp"]:
        USE_BF16, USE_FP16 = HW_BF16, HW_FP16
    else:
        USE_BF16, USE_FP16 = False, False


def check_vram_budget(n_params, model_name, optim=None):
    """对比各档位的估算需求与本机实际显存；不够就**当场**给出可执行的建议。
    返回 True=有档位装得下 / False=全部装不下 / None=非 CUDA 环境无法判断。"""
    if not torch.cuda.is_available():
        print("非 CUDA 环境：跳过显存预算检查（按标准全参微调跑）")
        return None
    n_gpu = torch.cuda.device_count()
    for _i in range(n_gpu):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor}")
    total_gb = _vram_total_gb()
    print(f"显存预算：{model_name} | {n_params / 1e6:.0f}M 参数（实际） | "
          f"选档参考 {FIT_REF_PARAMS / 1e6:.0f}M（仅 auto 模式用；当前 FIT_MODE={FIT_MODE}） | "
          f"单卡容量 {total_gb:.1f} GiB | 安全系数 {FIT_SAFETY:.0%}")
    tier, rows = resolve_fit_tier(total_gb, FIT_REF_PARAMS)
    print("  档位表（固定开销与 batch 无关，降 batch 救不了）：")
    print(f"    {'档':4s}{'含义':34s}{'固定':>9s}{'bs=1 需':>10s}{'bs=8 需':>10s}  判定")
    for name, label, fixed, need_min, need8, verdict in rows:
        if fixed is None:
            print(f"    {name:4s}{label:34s}{'-':>9s}{'-':>10s}{'-':>10s}  {verdict}")
        else:
            print(f"    {name:4s}{label:34s}{fixed:8.1f}G{need_min:9.1f}G{need8:9.1f}G  {verdict}")
    fixed_act, need_act = tier_estimate_gb(n_params, tier)
    print(f"  → 采用档位 {tier['name']}（{tier['label']}）；"
          f"本模型固定开销 {fixed_act:.1f} GiB"
          f"（+ 激活：bs=1 约 {tier['act_gb'] * 0.2:.1f} / bs=8 约 {tier['act_gb']:.1f} GiB，"
          f"实际能跑多大由探测决定）")
    if n_gpu > 1:
        print(f"  ⚠️ 本机有 {n_gpu} 张卡，但**单进程默认只用 GPU0**。多卡要用对方式才有效：")
        print("     · DDP（数据并行）：每张卡各存**一整份**模型 → **不降低单卡显存需求**，只是把数据分半")
        print("     · ZeRO-3 / FSDP（参数分片）：把参数+梯度+优化器状态切到多卡 → "
              "这才能真正用满多卡显存（需装 deepspeed）")
        print("     · 最省事：两条基线**各占一张卡并行跑**（两个独立实验，互不干扰）")
    _fixed_m, need_min = tier_estimate_gb(n_params, tier, act_gb=FIT_ACT_MIN_GB)
    if need_min <= total_gb:
        return True
    print("=" * 78)
    print(f"⚠️ 警告：档位 {tier['name']} 即使按 micro-batch=1 也要约 {need_min:.1f} GiB，"
          f"本卡只有 {total_gb:.1f} GiB。")
    print("   Step8 会用真实模型逐档试探；若仍 OOM 会自动再降一档，"
          "并在全部档位都失败时给出精确诊断（而不是一句 out of memory）。")
    print("=" * 78)
    return False


def ds_int_col(ds, name):
    """把数据集里的整型列取成**一维** int64 数组。
    【为什么需要】`datasets` 取「列表型列」时可能返回嵌套 list，也可能返回展平后的 list
       （取决于底层 arrow 的存储类型）。group_index 若被读成 (N,1)，
       后面 `np.max` / `index_add_` 的形状都会对不上，所以这里统一压平。
    """
    arr = np.asarray(ds[name])
    if arr.ndim > 1:
        arr = arr.reshape(-1)
    return arr.astype(np.int64)


def _assert_args_consistent(args, n_train):
    """把「配置常量」与「TrainingArguments 实际生效的口径」对齐，开训前执行。
    口径清单若只打印配置变量，就可能出现「清单说 8、实际跑 16」的静默漂移
    ⇒ 这里断言 batch / 累积 / warmup 与推导值逐项一致，漂移当场报错。

    transformers 5 的步数语义（与 4.x 不同）：
      `train_batch_size = per_device × n_gpu`（不含梯度累积）；
      `num_update_steps_per_epoch = ceil(len(dataloader) / grad_accum)`；
      `max_steps = ceil(epochs × num_update_steps_per_epoch)`。
    """
    _bs = int(args.per_device_train_batch_size)
    _ga = int(args.gradient_accumulation_steps)
    _ep = int(args.num_train_epochs)
    assert _bs == BATCH_SIZE, (
        f"per_device_train_batch_size={_bs} ≠ 配置 {BATCH_SIZE}：口径漂移，拒绝开训")
    assert _ga == GRAD_ACCUM, (
        f"gradient_accumulation_steps={_ga} ≠ 配置 {GRAD_ACCUM}：口径漂移，拒绝开训")
    _loader = math.ceil(n_train / _bs)                       # DataLoader 每轮的批数
    _max_steps = math.ceil(_loader / _ga) * _ep              # Trainer 实际的总更新步数
    _expected_warmup = max(1, int(math.ceil(n_train / (_bs * _ga)) * _ep * WARMUP_RATIO))
    assert int(args.warmup_steps) == _expected_warmup, (
        f"warmup_steps={args.warmup_steps} ≠ 按同一口径算出的 {_expected_warmup}：口径漂移，拒绝开训")
    _wf = float(args.warmup_steps) / max(1, _max_steps)
    # 容差：warmup_steps 是 int(ratio×T)，截断最多差 1 步；
    # 步数很少时（冒烟模式）1 步占比就很大，所以容差随步数自适应。
    _tol = max(0.02, 2.0 / max(1, _max_steps))
    assert abs(_wf - WARMUP_RATIO) <= _tol, (
        f"warmup 实际占比 {_wf:.3f} 与设定 {WARMUP_RATIO} 相差过大"
        f"（warmup_steps={args.warmup_steps} / max_steps={_max_steps}）：口径漂移，拒绝开训")
    print(f"   口径自检 ✓ 有效 batch = {_bs}×{_ga} = {_bs * _ga} | "
          f"每轮批数 {_loader} | 总更新步 {_max_steps} | "
          f"warmup {args.warmup_steps} 步（占 {_wf:.1%}）")


def build_training_args(output_dir, n_train, epochs=None, **overrides):
    """构造一份 TrainingArguments。每次调用都新建对象：
    Trainer 会就地改写传入的 args，共用对象会让第二个 Trainer 拿到脏状态。
    warmup_steps 自己算（transformers 5 删除了 warmup_ratio）；
    n_train 必须传入：5 次运行数据量不同，步数按各自数据量算。
    epochs 可按运行覆盖（联合运行用 JOINT_EPOCHS），warmup 跟着重算。"""
    _epochs = int(epochs) if epochs else int(EPOCHS)
    steps_per_epoch = math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM))
    total_steps = steps_per_epoch * _epochs
    base = dict(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=_epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=max(1, int(total_steps * WARMUP_RATIO)),   # transformers 5 只有绝对步数
        lr_scheduler_type="linear",
        # logging_steps 只影响日志密度：每 1/8 轮记一个点（3 轮 → ~24 点），
        # 不改变梯度、更新次数或权重。
        logging_steps=max(1, steps_per_epoch // 8),
        save_strategy="epoch",
        save_total_limit=1,
        # save_only_model：默认 checkpoint 连优化器状态一起存（900M 模型 ~14GB/个），
        # 只存权重 → ~3.6GB；参数是 transformers 4.56+ 才有，按签名探测。
        load_best_model_at_end=True,
        # best checkpoint 用综合分（z 分数空间），不能只看指标 1——
        # 指标 2 是唯一的出局线，只看指标 1 会选出一个指标 2 已变差的 checkpoint。
        metric_for_best_model=BEST_METRIC_KEY,
        greater_is_better=True,
        remove_unused_columns=False,   # 我们要保留 labels / group_index，交给自定义 collator
        dataloader_pin_memory=(DEVICE == "cuda"),   # MPS 上会警告「pin_memory 不支持」
        dataloader_num_workers=NUM_WORKERS,
        dataloader_persistent_workers=DATALOADER_PERSISTENT,   # 必须与 num_workers 一起降
        dataloader_multiprocessing_context=None if NUM_WORKERS == 0 else "fork",
        dataloader_prefetch_factor=None if NUM_WORKERS == 0 else 4,
        seed=int(overrides.pop("seed", SEED)),
        data_seed=SEED,
        report_to=[],                  # 不接 wandb/tensorboard
        disable_tqdm=True,
        bf16=USE_BF16,
        fp16=USE_FP16,
        optim=OPTIM,
        gradient_checkpointing=USE_GRAD_CKPT,   # 显存不够时开：激活降到约 1/N
    )
    base[_EVAL_STRATEGY_KEY] = "epoch"      # 5.x = eval_strategy，4.x = evaluation_strategy
    if _HAS_SAVE_ONLY_MODEL:
        base["save_only_model"] = True
    base.update(overrides)
    # use_reentrant=False：reentrant=True 的 checkpoint 要求输入张量 requires_grad，
    # 冻结底层（只训顶部层）时输入不再 requires_grad ⇒ 梯度会静默变 None。
    # 放在 overrides 之后：探测阶段的临时覆盖也要拿到这个保护。
    if base.get("gradient_checkpointing") and "gradient_checkpointing_kwargs" in _TA_PARAMS:
        base["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    args = TrainingArguments(**base)
    # 强制单卡：进程可见 GPU ≥ 2 时 Trainer 会自动套 nn.DataParallel，
    # 其反向额外复制一整份梯度（显存翻倍）。Step1 已用 CUDA_VISIBLE_DEVICES
    # 限制单卡，这里再钉 args._n_gpu = 1 作为第二道保险。
    try:
        args._n_gpu = 1
    except Exception:      # 不同版本可能没有这个属性，忽略即可
        pass
    return args

## Step7 创建训练器

**这一节要解决什么**：写出损失函数，并把它接进 `Trainer`。

### 三个官方指标各自都有一个对应的损失项

| 官方指标 | 对应的损失项 | 项的形式 | 为什么这样对应 |
|---|---|---|---|
| **指标 1** 组合性相关 ρ（主指标） | `L_huber`（绝对值锚）+ `L_order`（组内成对排序） | 绝对误差 / 组内成对符号 | ρ 是秩相关：跨目标的相对大小由 Huber 承担，组内的相对顺序由只取符号的排序项承担（对独立噪声不敏感） |
| **指标 2** 语境方差相关 ρ（次要 / 非 nan 红线） | `L_guard`（反收缩 hinge）+ `L_xrank`（跨目标 std 的序） | 下界 hinge / 序比较 | 指标 1 与 3 的最优解都是「收缩到条件均值」，会系统性压平组内离散度 ⇒ 必须有反向力量把离散度顶住；`L_xrank` 是 Spearman 的可微形式 |
| **指标 3** RMSE | `L_mse`（显式二次项） | 平方误差 | Huber 在大误差处走线性段（抗噪），而 RMSE 是纯二次 ⇒ 给它一个小权重的专属项 |

### 两个机制项（各挂在一个基线上，互斥开启）

| 基线 | 新项 | 形式 | 作用范围 | 为什么 |
|---|---|---|---|---|
| **DeBERTa** | `L_softcl`（软标签对比学习） | `(cos(z_i,z_j) − w_ij)²`，`w_ij = cos(π·|Δy|/5)` | 仅 pv | 用标签距离做成对监督，比纯符号（order）与逐样本绝对误差（huber，被噪声主导）更稳；nn 的组内差异 93–129% 是标注噪声 ⇒ 只在 pv 开 |
| **Qwen** | `L_ce`（数字 token 软目标交叉熵） | `−Σ_k q_k·log softmax(logits)_k`，`q_k ∝ exp(−(k−y)²/2σ²)` | 四个子集 | 无回归头时 LM 头的分布是唯一输出通路：期望值层（huber）管「分数对不对」，分布层（本项）管「分布形状对不对」 |

`L_softcl` 用有界的相似度回归而不是 InfoNCE：按锚点归一化会抹掉确信度、
把离群样本拉向簇中心；batch 内 softmax 对压低正例概率的惩罚无界。
目标 `w_ij` 必须带符号（否则分不出「正交」与「重合」）。
`L_ce` 的最小值是目标分布的熵 H(q) 而不是 0（软目标交叉熵的数学性质）。

### 为什么按子集差异化（`SUBSET_LOSS_PROFILE`）

指标 2 的现实天花板由蒙特卡洛给出（`analysis/mc_metric2_ceiling.py`）：

| 设置 | m2_perfect | m2_realistic |
|---|---|---|
| en-nn/ModAvg | 0.176 | 0.052 |
| en-nn/HeadAvg | 0.235 | 0.081 |
| de-nn/ModAvg | 0.002 | 0.001 |
| de-nn/HeadAvg | 0.001 | 0.002 |
| en-pv/Avg | 0.573 | 0.413 |
| de-pv/Avg | 0.437 | 0.257 |

gold 组内 std 的 40–74% 由「目标级歧义度」决定而非语境效应，
split-half 可复现性只有 21–39% ⇒ nn 上追指标 2 基本无望（只防塌陷），
pv 上天花板可达（值得用更紧的 κ + 跨目标 std 序）；
指标 2 的红线含义是「非 nan」，不是「数值高」。

### 为什么删掉「组内 std 等值匹配」

人工的目标级 std 里掺着标注噪声（膨胀 2.7–13.6%），等值匹配 = 要求模型复现
不可复现的噪声 ⇒ 方向错。更根本的：grouped sampler 让一个 batch 通常只有
1–2 个目标，「组内 std 失配」在 batch 内既稀疏又高噪声 ⇒ 用 `L_xrank`
的「跨 batch 累积统计 + 序比较」替代。

### 为什么权重要「按占比自动标定」

手猜权重太小没效果、太大压过主项。`SubsetLossBalancer` 把权重定义成
「该项应占总损失的百分之几」，用各项实时量级（EMA）在线反解实际权重；
每个子集、每个标签列各有自己的 EMA，避免联合运行时互相污染量级估计。

### 为什么实现要「少同步」

布尔掩码索引 / `.item()` 每组都会插入 GPU→CPU 同步，打断流水；
用 `index_add_`（输出长度固定）+ 张量化加权求和 ⇒ 热路径只剩 2 次同步。

### 打开组内项之前必须有的护栏

小品词动词第 2 列是占位常数 0（loss_mask=0），人工组内 std 恒为 0；
不跳过它，组内项会把该列离散度压成 0，梯度经共享主干干扰第 1 列
⇒ 损失项一律接收 `valid`，无效列整体跳过。

**为什么只覆写 `compute_loss` 这一个方法**：前向、反向、梯度累积、日志、
评估、保存全由框架负责，我们只替换损失这一环。

In [ ]:
# ============ 封装：损失引擎（按子集差异化 + 按占比自动标定）============
N_SUB = len(SUBSET_ORDER)


def profile_vectors(device, dtype=torch.float32):
    """把 SUBSET_LOSS_PROFILE 展开成「按子集编号」的向量（长度 N_SUB）。

    【为什么要展开成向量】损失项要在**一个 batch 内**同时处理多个子集（联合运行），
    而每个子集的「占比 / κ」都不同 ⇒ 只能以张量形式逐子集并行计算，
    这样才不会为了「分子集」把 batch 切开（切开必然引入 GPU→CPU 同步）。
    """
    def vec(key):
        return torch.tensor([float(SUBSET_LOSS_PROFILE[n].get(key, 0.0)) for n in SUBSET_ORDER],
                            device=device, dtype=dtype)
    # 各项默认 0（该基线用不到时 .get(key, 0.0) 兜底）：
    # ce / softcl 机制项、gate 门控监督项、tierank 跨目标分数序、swap 交换一致性。
    return dict(order=vec("order"), mse=vec("mse"), guard=vec("guard"),
                xrank=vec("xrank"), kappa=vec("kappa"),
                ce=vec("ce"), softcl=vec("softcl"), gate=vec("gate"),
                swap=vec("swap"), tierank=vec("tierank"))


def _agg_by_subset(values, index, n_sub):
    """把逐元素的值按「所属子集」聚合（index 是每个元素的子集编号）。

    【为什么用 index_add_ 而不是布尔掩码索引】布尔掩码索引的输出长度在 GPU 上未知，
    每次调用都要一次同步；index_add_ 的输出长度固定 ⇒ 全程不同步。
    """
    return torch.zeros(n_sub, device=values.device, dtype=values.dtype).index_add_(0, index, values)


def order_parts(pred, gold, group_index, sidx, tau, valid, order_vec, n_sub):
    """组内成对排序（指标 1 的组内部分）。返回按子集聚合的 (分子, 分母)。

    【为什么用 softplus 而不是 hinge】softplus(-s·Δ/τ) 是 margin 排序损失的光滑版，
      Δ=0 时梯度最大，|Δ| 很大时梯度趋近 0 —— 既推动「排对」，又不会加压过头。
    【为什么只在组内配对】跨目标的「谁更组合」不是本任务要学的东西，
      不同目标的标注者尺度也可能不同。
    【为什么对噪声稳健】排序只用符号：独立噪声不改变期望梯度方向，只衰减幅度
      ⇒ 是「去噪后的信号」，补上绝对值回归被噪声主导的部分。
    【valid】这一列有没有真标签（小品词动词第 2 列是占位的 0），
      无标签列不能参与排序。
    【占比权重 order_vec】占比为 0 的子集既不产生梯度也不进分母（该项不存在）。
    """
    B = pred.shape[0]
    iu = torch.triu(torch.ones(B, B, dtype=torch.bool, device=pred.device), diagonal=1)
    mf = iu & (group_index[:, None] == group_index[None, :])
    if valid is not None:
        ok = valid > 0
        mf = mf & ok[:, None] & ok[None, :]
    diff_p = pred[:, None] - pred[None, :]
    diff_y = gold[:, None] - gold[None, :]
    sign = torch.sign(diff_y)
    mf = mf & (sign != 0)          # 真实值并列的对不提供排序信息
    mff = mf.to(pred.dtype)
    w = order_vec[sidx]             # [B] 每条例句所属子集的 order 占比
    num = F.softplus(-sign * diff_p / tau) * mff * w[:, None]
    den = mff * w[:, None]
    idx = sidx[:, None].expand(B, B).reshape(-1)
    return _agg_by_subset(num.reshape(-1), idx, n_sub), _agg_by_subset(den.reshape(-1), idx, n_sub)


def mse_parts(pred, gold, valid, sidx, mse_vec, n_sub):
    """指标 3（RMSE）的直接对应项。返回按子集聚合的 (分子, 分母)。

    【★ 为什么 Huber 之外还要单独放一项 MSE】
      Huber(δ=1) 在 |e|>1 走线性段（抗噪），而官方指标 3 是**纯二次**的 RMSE。
      两者的最优解不同：Huber 偏向「中位数型」稳健估计，MSE 偏向「条件均值」。
      指标 1 是**秩**相关（单调变换不变）⇒ 两者都能用；
      但指标 3 只认二次 ⇒ 给它一个显式的、小权重的对应项，
      同时不让它大到把预测压成常数（那会伤指标 2）。
    """
    w = mse_vec[sidx]
    if valid is not None:
        w = w * valid.to(pred.dtype)
    num = (pred - gold) ** 2 * w
    return _agg_by_subset(num, sidx, n_sub), _agg_by_subset(w, sidx, n_sub)


def guard_parts(std_pred, std_gold, cnt, group_subset, kappa_vec, n_sub):
    """反收缩 hinge：只惩罚「组内 std 掉到参考值的 κ 倍以下」。指标 2 的主力项。

    指标 1/3 的最优解都是收缩到条件均值，会系统性压平组内离散度、伤指标 2
    ⇒ 必须有一项反向力量把离散度顶在参考值之上。
    κ 按子集给：人工 std 被标注噪声膨胀 2.7–13.6% ⇒ 正常 std ≈ 人工 × 0.86–0.97；
    nn κ=0.15 只挡真塌陷（天花板 0.00–0.08），pv κ=0.50 真正顶住离散度（天花板可达）。
    护栏：cnt<2 的组没有 std、占位列人工 std 恒为 0，都必须排除。
    """
    kg = kappa_vec[group_subset]              # 每个目标组的 κ
    ok = (cnt >= 2) & (kg > 0)                 # κ=0 的子集 ⇒ 该项关闭
    okf = ok.to(std_pred.dtype)
    gap = (kg * std_gold - std_pred).clamp(min=0.0)
    num = gap * gap * okf
    return _agg_by_subset(num, group_subset, n_sub), _agg_by_subset(okf, group_subset, n_sub)


def ce_parts(digit_logits, gold, valid, sidx, ce_vec, n_sub, sigma=None):
    """数字 token 上的软目标交叉熵（Qwen 无回归头模式）。

    要求 LM 头给出的「下一个 token 是 0–5」的分布本身落在真实标签附近：
        目标分布 q_k ∝ exp(−(k − y)² / (2σ²))，再归一化
        损失 = −Σ_k q_k · log softmax(logits)_k
    标签是连续均值 ⇒「软」目标（3.5 = 3 与 4 的混合）。它和期望值回归是两层：
    huber 管分数对不对，本项管分布形状对不对（防止用尖锐双峰凑期望值）。
    σ=0.8：差 1 分算近、差 2 分以上算远。护栏：无真标签的列整体跳过。"""
    if digit_logits.dim() == 2:            # [B, K] → 补一个列维（单列）
        digit_logits = digit_logits.unsqueeze(1)
    if gold.dim() == 1:                   # [B] → [B, 1]
        gold = gold.unsqueeze(1)
    if sigma is None:                     # 运行时再取默认值（避免定义期就要求全局常量）
        sigma = CE_SOFT_SIGMA
    B, C, K = digit_logits.shape
    k = torch.arange(K, device=digit_logits.device, dtype=digit_logits.dtype).view(1, 1, K)
    y = gold.unsqueeze(-1)                                   # [B, C, 1]
    q = torch.exp(-0.5 * ((k - y) ** 2) / (sigma ** 2))
    q = q / q.sum(dim=-1, keepdim=True).clamp(min=1e-9)
    logp = F.log_softmax(digit_logits, dim=-1)
    ce = -(q * logp).sum(dim=-1)                              # [B, C]
    w = ce_vec[sidx].unsqueeze(1)          # [B,1] → 广播到两列
    if valid is not None:
        v = valid.to(ce.dtype)
        if v.dim() == 1:
            v = v.unsqueeze(1)          # [B] → [B,1]，与 w 的形状对齐
        w = w * v
    # 【按「列」聚合】每条例句有两列 ⇒ 把 (B, C) 摊平后按子集合并
    idx = sidx.unsqueeze(1).expand(B, C).reshape(-1)
    num = _agg_by_subset((ce * w).reshape(-1), idx, n_sub)
    den = _agg_by_subset(w.reshape(-1), idx, n_sub)
    return num, den


def softcl_parts(z, sample_label, sidx, label_span, softcl_vec, n_sub):
    """软标签对比学习（DeBERTa 机制实验）的损失项。

    让表征相似度贴合标签相似度：
        目标相似度   w_ij = cos(π·|y_i − y_j| / 标签量程) ∈ [−1, 1]
        预测相似度   s_ij = cos(z_i, z_j) ∈ [−1, 1]
        损失         (s_ij − w_ij)²
    即「表征空间的距离结构必须复刻标签的距离结构」。

    用相似度回归而不是 InfoNCE：按锚点归一化会抹掉确信度、把离群样本拉向簇中心；
    batch 内 softmax 对压低正例概率的惩罚无界。目标必须带符号（cos 形式），
    否则平方误差分不出「正交」与「重合」。
    相比 order（只用符号）与 huber（被标注噪声主导），
    它用标签距离做成对监督，对噪声更稳且保留「差多少」。
    只在同一子集内配对（跨语言/任务标签尺度不可比）；
    分母为 0 的子集整体退出。"""
    B = z.shape[0]
    eye = torch.eye(B, dtype=torch.bool, device=z.device)
    same_sub = sidx[:, None] == sidx[None, :]
    m = (~eye) & same_sub
    diff = (sample_label[:, None] - sample_label[None, :]).abs()
    w = torch.cos(math.pi * diff / float(label_span))   # 目标相似度 ∈ [-1, 1]
    sim = z @ z.t()                                      # 余弦相似度 ∈ [-1, 1]（z 已 L2 归一化）
    err = (sim - w) ** 2
    wf = softcl_vec[sidx].unsqueeze(1)                   # [B,1]：该样本所属子集的占比
    fm = m.to(z.dtype)
    num = _agg_by_subset((err * fm * wf).reshape(-1),
                         sidx.unsqueeze(1).expand(B, B).reshape(-1), n_sub)
    den = _agg_by_subset((fm * wf).reshape(-1),
                         sidx.unsqueeze(1).expand(B, B).reshape(-1), n_sub)
    return num, den


def gate_parts(gate, target, valid, sidx, gate_vec, n_sub):
    """语境敏感度门 g 的监督项（模块 2 的可选档）。

    让 g 逼近该目标的「语境有效度 share」（组内观测方差去掉标注噪声后的比例，
    见 cells_a.compute_ctx_share）。默认不开（权重 0）：歧义度与组内 std 的相关
    主要由噪声驱动（用它监督方向是反的），且带监督会让「g 是否分化」这个诊断
    变成同义反复 ⇒ 默认自由门 + 诊断，本项留作可选档。"""
    g = gate.reshape(-1).to(torch.float32)
    t = target.reshape(-1).to(torch.float32)
    w = gate_vec[sidx]
    if valid is not None:
        w = w * valid.to(g.dtype)
    num = ((g - t) ** 2) * w
    return _agg_by_subset(num, sidx, n_sub), _agg_by_subset(w, sidx, n_sub)


def xrank_parts(std_pred, std_gold, cur_ids, group_subset, xrank_vec, margin, tau, hist_mask, n_sub):
    """跨目标「组内 std 的序」比较 —— 指标 2（Spearman over targets）的直接可微代理。

    grouped sampler 让一个 batch 通常只有 1–2 个目标，batch 内凑不出跨目标对
    ⇒ 把当前目标与「历史累积统计里见过的所有目标」逐一比较：
    梯度只流过当前目标（历史 detach）。
    只在同一子集内部比较（跨语言/任务 std 尺度不可比）；
    只取「人工 std 差异 > margin」的对（差异太小的对大小关系接近随机，只会注入噪声）；
    softplus 是「比大小」的光滑替代，|Δ| 大时梯度趋 0、不加压过头。
    """
    N = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)
    D = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)
    G = std_pred.numel()
    ar = torch.arange(G, device=std_pred.device)
    for g in cur_ids:
        sg = std_gold[g]
        sp = std_pred[g]
        sub_g = group_subset[g]
        allowed = (group_subset == sub_g) & hist_mask & (ar != g) & ((std_gold - sg).abs() > margin)
        af = allowed.to(std_pred.dtype)
        sign = torch.sign(sg - std_gold)
        pen = F.softplus(-sign * (sp - std_pred) / tau)
        w = xrank_vec[sub_g].to(std_pred.dtype)      # 该子集的 xrank 占比（0 ⇒ 关闭）
        idx = sub_g.repeat(G)
        N = N.index_add_(0, idx, pen * af * w)
        D = D.index_add_(0, idx, af * w)
    return N, D


def tierank_parts(mean_p, mean_g, cur_ids, group_subset, tierank_vec, margin, tau, hist_ok, n_sub):
    """跨目标「分数序」—— 目标间分数大小关系的可微代理。

    标签方差 68–88% 在目标之间，order 只管组内序、xrank 只管跨目标 std 序
    ⇒ 本项补上「目标间分数大小关系」这一层（指标 1 的 type-level 主战场）。
    实现与 xrank 同构：梯度只过当前目标、与历史（detach）比；
    均值从 GroupStats 矩还原（float64，EMA 衰减对均值尺度不变）；
    均值差 ≤ margin 的对不参与（大小关系接近随机）；同子集内比。
    """
    N = torch.zeros(n_sub, device=mean_p.device, dtype=mean_p.dtype)
    D = torch.zeros(n_sub, device=mean_p.device, dtype=mean_p.dtype)
    G = mean_p.numel()
    ar = torch.arange(G, device=mean_p.device)
    for g in cur_ids:
        sub_g = group_subset[g]
        allowed = (group_subset == sub_g) & hist_ok & (ar != g) & ((mean_g - mean_g[g]).abs() > margin)
        af = allowed.to(mean_p.dtype)
        sign = torch.sign(mean_g[g] - mean_g)     # 真值序：+1 ⇒ 目标 g 应高于 h
        pen = F.softplus(-sign * (mean_p[g] - mean_p) / tau)
        w = tierank_vec[sub_g].to(mean_p.dtype)   # 该子集的 tierank 占比（0 ⇒ 关闭）
        idx = sub_g.repeat(G)
        N = N.index_add_(0, idx, pen * af * w)
        D = D.index_add_(0, idx, af * w)
    return N, D

In [ ]:
# ============ 演示：各损失项分别在做什么（且按子集差异化）============
# 多项损失最容易出的错是「以为在优化 A，其实在优化 B」；
# 用 6 条样本的玩具 batch（en-nn 2 条 + en-pv 4 条、两个目标）逐项打印数值。
toyl = torch.tensor([[1.0], [4.0], [2.0], [5.0], [1.0], [3.0]])   # 真实值
toyp = torch.tensor([[1.5], [3.0], [2.2], [4.0], [1.3], [1.7]])   # 一个「还行」的预测
toyg = torch.tensor([0, 0, 1, 1, 2, 2])                         # 三个目标组（一维）
toys = torch.tensor([0, 0, 2, 2, 2, 2])                           # 前两条属 en-nn，其余属 en-pv
toyv = torch.ones(6)

_prof = profile_vectors(torch.device("cpu"), torch.float32)
print("各子集的损失配置（占比 / κ）：")
for _s in range(len(SUBSET_ORDER)):
    _p = SUBSET_LOSS_PROFILE[SUBSET_ORDER[_s]]
    print(f"  {SUBSET_ORDER[_s]:6s} order={_p['order']:.2f}  mse={_p['mse']:.2f}  "
          f"guard={_p['guard']:.2f}  xrank={_p['xrank']:.2f}  kappa={_p['kappa']:.2f}")

_N, _D = order_parts(toyp[:, 0], toyl[:, 0], toyg, toys, ORDER_TAU, toyv, _prof["order"], 4)
_l_order = _N / _D.clamp(min=1.0)
_N, _D = mse_parts(toyp[:, 0], toyl[:, 0], toyv, toys, _prof["mse"], 4)
_l_mse = _N / _D.clamp(min=1.0)
# guard / xrank 需要组级统计：玩具里每组 2 条，手算各自的组内 std
_gold_g = [[1.0, 4.0], [2.0, 5.0], [1.0, 3.0]]
_pred_g = [[1.5, 3.0], [2.2, 4.0], [1.3, 1.7]]
_std_g = torch.tensor([float(torch.std(torch.tensor(g), unbiased=True)) for g in _gold_g])
_std_p = torch.tensor([float(torch.std(torch.tensor(g), unbiased=True)) for g in _pred_g])
_cnt = torch.tensor([2.0, 2.0, 2.0])
_gsub = torch.tensor([0, 2, 2])          # 组 0 → en-nn，组 1/2 → en-pv
_N, _D = guard_parts(_std_p, _std_g, _cnt, _gsub, _prof["kappa"], 4)
_l_guard = _N / _D.clamp(min=1.0)
_N, _D = xrank_parts(_std_p, _std_g, torch.tensor([0, 1, 2]), _gsub, _prof["xrank"],
                      XRANK_MARGIN, XRANK_TAU, torch.ones(3, dtype=torch.bool), 4)
_l_xrank = _N / _D.clamp(min=1.0)

print()
print(f"L_huber = {F.huber_loss(toyp, toyl, delta=HUBER_DELTA).item():.4f}   "
      f"绝对值误差（指标 1 的锚 / 指标 3 的主体）")
print(f"L_order  (en-nn / en-pv) = {_l_order[0].item():.4f} / {_l_order[2].item():.4f}   "
      f"组内相对顺序（目标0真实 1<4，预测 1.5>3.0 → 被罚）")
print(f"L_mse    (en-nn / en-pv) = {_l_mse[0].item():.4f} / {_l_mse[2].item():.4f}   "
      f"指标 3（RMSE）的显式二次项")
print(f"L_guard  (en-nn / en-pv) = {_l_guard[0].item():.4f} / {_l_guard[2].item():.4f}   "
      f"反收缩下界（en-nn κ=0.15 → 下界 0.32 < 预测 1.06，不触发；"
      f"en-pv κ=0.50 → 下界 0.71 > 预测 0.28 → 被罚）")
print(f"L_xrank  (en-nn / en-pv) = {_l_xrank[0].item():.4f} / {_l_xrank[2].item():.4f}   "
      f"跨目标 std 的序（en-nn 的 xrank 占比=0 ⇒ 该子集关闭；"
      f"en-pv 在组 1/2 之间比较 ⇒ 有值）")
print()
print("→ 绝对值项是锚；排序项纠正组内顺序；MSE 项对齐 RMSE；")
print("   guard 项反向对抗「组内方差被压平」；xrank 项只在 pv 上开（nn 的天花板太低）。")

# ---- 两个机制损失项的演示（好分布 vs 坏分布）----
print()
print("--- 机制损失项（ce / softcl）---")
_k = torch.arange(6, dtype=torch.float32)
# 【ce 演示】造两组「数字分布」：一组正好落在标签上，一组整体偏了 2 分
_dl_good = -((_k.view(1, -1) - toyl.view(-1, 1)) ** 2) / (2 * CE_SOFT_SIGMA ** 2)
_dl_bad = -((_k.view(1, -1) - (toyl.view(-1, 1) + 2.0)) ** 2) / (2 * CE_SOFT_SIGMA ** 2)
_N, _D = ce_parts(_dl_good.unsqueeze(1), toyl, toyv, toys, _prof["ce"], 4)
_ce_good = (_N / _D.clamp(min=1.0))[2].item()
_N, _D = ce_parts(_dl_bad.unsqueeze(1), toyl, toyv, toys, _prof["ce"], 4)
_ce_bad = (_N / _D.clamp(min=1.0))[2].item()
print(f"L_ce    (en-pv) = {_ce_good:.4f}（分布正好落在标签上） / "
      f"{_ce_bad:.4f}（分布整体偏 2 分）   数字 token 上的软目标交叉熵")
# 【softcl 演示】造两组表征：一组按标签角度排布（标签近的表征也近），一组与标签无关
_ang = toyl[:, 0] / 5.0 * torch.pi
_z_good = F.normalize(torch.stack([torch.cos(_ang), torch.sin(_ang)], dim=1), dim=1)
_z_bad = F.normalize(torch.stack([torch.cos(toys.float()), torch.sin(toys.float())], dim=1), dim=1)
_N, _D = softcl_parts(_z_good, toyl[:, 0], toys, SOFTCL_LABEL_SPAN, _prof["softcl"], 4)
_l_sc_good = (_N / _D.clamp(min=1.0))[2].item()
_N, _D = softcl_parts(_z_bad, toyl[:, 0], toys, SOFTCL_LABEL_SPAN, _prof["softcl"], 4)
_l_sc_bad = (_N / _D.clamp(min=1.0))[2].item()
print(f"L_softcl(en-pv) = {_l_sc_good:.4f}（表征按标签排布） / "
      f"{_l_sc_bad:.4f}（表征与标签无关）   软标签相似度回归")
print("→ ce 管「分数分布的形状」，softcl 管「表征之间的相似度」；")
print("   softcl 的目标很直白：cos(z_i, z_j) 应当等于 cos(π·|y_i−y_j|/量程)；")
print("   它只在 pv 上开（nn 的组内差异九成以上是标注噪声，在那里做等于拟合噪声）。")

# ---- L_tierank 演示（真值组均值 (2.5,3.5,2.0)；「排错」把最高真值的组1压到组2 之下 ⇒ 被罚；
#      组0↔组1/2 是 nn↔pv 被排除 ⇒ 只有 pv 的组1↔组2 参与）----
_tg_mean = torch.tensor([2.5, 3.5, 2.0])
_tp_ok = torch.tensor([2.25, 3.1, 1.5])
_tp_bad = torch.tensor([2.25, 1.5, 3.1])
for _nm, _tp in (("排对", _tp_ok), ("排错", _tp_bad)):
    _N, _D = tierank_parts(_tp, _tg_mean, torch.tensor([0, 1, 2]), _gsub,
                           _prof["tierank"], TIERANK_MARGIN, TIERANK_TAU,
                           torch.ones(3, dtype=torch.bool), 4)
    _ltr = (_N / _D.clamp(min=1.0))
    if _nm == "排对":
        _tr_ok = _ltr[2].item()
    else:
        _tr_bad = _ltr[2].item()
print(f"L_tierank(en-pv) = {_tr_ok:.4f}（预测序与真值序一致） / "
      f"{_tr_bad:.4f}（组1 真值最高却被排到组2 之下 ⇒ 被罚）   跨目标分数序")
print("→ tierank 管「目标与目标之间的分数大小关系」（68–88% 的方差在目标间，"
      "order 只管组内、管不到这一层）。")

In [ ]:
def stats_dtype_for(device):
    """组统计矩的累加精度：能用 float64 就用 float64（MPS 不支持，退回 float32）。

    组内方差用「平方和 − 和的平方」算，是灾难性抵消：fp32 下误差可达 1.6%、
    组内差异极小时甚至变号 ⇒ sqrt 导数在 0 附近剧烈跳动，加上 CUDA 原子累加
    顺序随机，会产生 inf/NaN 梯度；float64 下误差降到 ~1e-9 量级。
    按设备能力探测：MPS 不支持 float64，直接写死会让本机冒烟崩溃。
    """
    try:
        torch.zeros(1, dtype=torch.float64, device=device)
        return torch.float64
    except Exception:
        return torch.float32


class GroupStats:
    """跨 batch 累积的目标级统计量（把组内 std 的估计噪声降下来）。

    一个 batch 通常只有 1–2 个目标、每组约 8 条，组内 std 的单次估计相对误差
    ≈ 27%（k=8）⇒ 每个目标的「计数/和/平方和」用 EMA 跨 batch 累积，
    历史部分 detach（梯度不穿历史），估计噪声随累积下降。
    累加精度由 stats_dtype_for 决定（矩相减必须 float64，见该函数说明）。"""

    def __init__(self, n_groups, device, dtype=None, decay=0.8):
        self.n = int(n_groups)
        self.decay = float(decay)
        self.device = device
        if dtype is None:
            dtype = stats_dtype_for(device)
        self.dtype = dtype
        self.p_cnt = torch.zeros(self.n, device=device, dtype=dtype)
        self.p_s1 = torch.zeros(self.n, device=device, dtype=dtype)
        self.p_s2 = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_cnt = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_s1 = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_s2 = torch.zeros(self.n, device=device, dtype=dtype)

    def _batch_stats(self, x, gidx, valid):
        """把 (cnt, Σx, Σx²) 按目标组累加。x 必须先升精度再算平方：
        若先在 float32 里算平方再升精度，舍入误差已经固化。"""
        xd = x.to(self.dtype)
        vd = valid.to(self.dtype)
        cnt = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        s1 = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        s2 = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        cnt.index_add_(0, gidx, vd)
        s1.index_add_(0, gidx, vd * xd)
        s2.index_add_(0, gidx, vd * xd * xd)
        return cnt, s1, s2

    def stats(self, pred, gold, gidx, valid):
        """返回 (cnt, s1, s2, gcnt, gs1, gs2) = 历史(detach) + 当前 batch。"""
        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)
        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)
        return (self.p_cnt + pc, self.p_s1 + ps1, self.p_s2 + ps2,
                self.g_cnt + gc, self.g_s1 + gs1, self.g_s2 + gs2)

    def commit(self, pred, gold, gidx, valid):
        """把本 batch 的贡献并入历史（**detach**，不留计算图）。"""
        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)
        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)
        d = self.decay
        self.p_cnt = d * self.p_cnt + pc.detach()
        self.p_s1 = d * self.p_s1 + ps1.detach()
        self.p_s2 = d * self.p_s2 + ps2.detach()
        self.g_cnt = d * self.g_cnt + gc.detach()
        self.g_s1 = d * self.g_s1 + gs1.detach()
        self.g_s2 = d * self.g_s2 + gs2.detach()


def std_from_moments(cnt, s1, s2, out_dtype=None):
    """由（计数, 和, 平方和）还原组内 std（ddof=1，与官方指标 2 口径一致）。

    四道安全网，保证梯度永不 inf/NaN：
    float64 输入（抵消误差可忽略）/ sqrt 加 eps（var=0 时导数有界）/
    cnt≥2 掩码（单例组的 std 数学上未定义，不能当「离散度最小」参与比较）/
    出口 nan_to_num 兜底。
    """
    n = cnt.clamp(min=1.0)
    mean = s1 / n
    var = (s2 / n - mean * mean) * (cnt / (cnt - 1.0).clamp(min=1.0))
    var = var.clamp(min=0.0)
    safe = torch.sqrt(var + 1e-12)
    out = safe * (cnt >= 2).to(var.dtype)
    out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    if out_dtype is not None:
        out = out.to(out_dtype)
    return out


class SubsetLossBalancer:
    """按「子集 × 标签列」分别在线标定辅助项的权重（全程不同步）。

    把权重定义成「该项应占总损失的百分之几」，用各项实时量级（EMA）反解：
        w_t = (frac_t / (1 − frac_t)) · EMA(main) / (EMA(term_t) + eps)
    无论两项量级怎么变，占比都稳定在设定值附近。
    按子集分开标定：nn 与 pv 的量级/目标占比都不同，共用 EMA 会互相污染。
    全程不调用 .item()（每次同步都会打断 GPU 流水）。
    「缺席即冻结」：某项缺席的步里 EMA 不衰减，否则下次出现时权重会爆炸。
    """

    MAX_WEIGHT = 5.0     # 权重上限：防止某项 EMA 趋近 0 时权重爆炸

    def __init__(self, n_sub, mix, beta=BALANCE_EMA_BETA, device="cpu"):
        self.n = int(n_sub)
        self.beta = float(beta)
        self.mix = {k: v.to(device) for k, v in mix.items()}    # [n_sub] 目标占比
        self.main_ema = torch.zeros(self.n, device=device)
        self.term_ema = {k: torch.zeros(self.n, device=device) for k in mix}
        self.weights = {}
        # 权重自身的全程 EMA：最后一步的权重可能恰好是缺席步（打印出 0），
        # 快照取全程平均才代表整场训练的标定结果。
        self.w_ema = {k: torch.zeros(self.n, device=device) for k in mix}

    def update_main(self, main_vec, active):
        """更新主项（按子集）的 EMA。active 是 0/1 向量：该子集本步是否出现。"""
        new = self.beta * self.main_ema + (1.0 - self.beta) * main_vec * active
        self.main_ema = torch.where(active > 0, new, self.main_ema)
        return self.main_ema

    def weight(self, name, term_vec, active):
        """返回该项的权重（[n_sub] 张量：不同步、不带梯度）；本步缺席的子集权重为 0。"""
        frac = self.mix[name]
        ema = self.term_ema[name]
        new_ema = self.beta * ema + (1.0 - self.beta) * term_vec * active
        ema = torch.where(active > 0, new_ema, ema)
        self.term_ema[name] = ema
        w = (frac / (1.0 - frac).clamp(min=1e-3)) * self.main_ema / (ema + 1e-6)
        w = w.clamp(max=self.MAX_WEIGHT).detach()
        w = torch.where(active > 0, w, torch.zeros_like(w))
        self.weights[name] = w
        # 权重自身的全程 EMA（缺席即冻结，与 term_ema 同一套规则）
        _we = self.w_ema[name]
        _we = torch.where(active > 0, self.beta * _we + (1.0 - self.beta) * w, _we)
        self.w_ema[name] = _we
        return w


class CompartmentTrainer(Trainer):
    """只覆写 compute_loss 的 Trainer。

    compute_loss 的签名必须带 num_items_in_batch（transformers 5 新增，
    子类不写会参数不匹配）；return_outputs 返回 {"logits": ...} 而不是
    ModelOutput（后者会被展开成不等长张量，评估抛 inhomogeneous shape）。"""

    def __init__(self, *args, grouped_batching=True, n_groups=None, group_subset=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.grouped_batching = grouped_batching
        self.n_groups = n_groups
        # 每个标签列一份组统计缓存（pv 的第 2 列全程无效 ⇒ 它的缓存一直是空的）
        # dtype 由 stats_dtype_for 按目标设备的能力决定（CUDA/CPU→float64；MPS→float32）
        self._gs = [GroupStats(n_groups or 1, torch.device("cpu"))
                    if n_groups else None for _ in range(N_OUTPUTS)]
        self._gs_ready = False
        # 每个目标组属于哪个子集：损失按子集差异化时要用
        self._group_subset = group_subset        # np.ndarray [n_groups]
        self._gsub_t = None
        self._prof = None
        self._bals = None
        # 训练期健康检查：每 N 步验一次 loss 是否有限
        self._step = 0
        self._finite_check_every = int(LOSS_FINITE_CHECK_EVERY)

    def _ensure_prof(self, device, dtype):
        """把组统计缓存、损失配置、标定器搬到训练设备上（第一次调用时）。"""
        if self._gs_ready:
            return
        # 累加精度按目标设备的能力决定（CUDA/CPU → float64；MPS → float32）
        _sdtype = stats_dtype_for(device)
        for st in self._gs:
            if st is not None:
                st.dtype = _sdtype
                st.p_cnt = st.p_cnt.to(device=device, dtype=_sdtype)
                st.p_s1 = st.p_s1.to(device=device, dtype=_sdtype)
                st.p_s2 = st.p_s2.to(device=device, dtype=_sdtype)
                st.g_cnt = st.g_cnt.to(device=device, dtype=_sdtype)
                st.g_s1 = st.g_s1.to(device=device, dtype=_sdtype)
                st.g_s2 = st.g_s2.to(device=device, dtype=_sdtype)
        self._prof = profile_vectors(device, dtype)
        mix = {k: self._prof[k] for k in ("order", "mse", "guard", "xrank", "ce", "tierank")}
        # 每个标签列一个标定器（列与列的量级不同，混在一起会算不准占比）
        self._bals = [SubsetLossBalancer(N_SUB, mix, BALANCE_EMA_BETA, device)
                      for _ in range(N_OUTPUTS)]
        # softcl 是逐样本的（不按列分开）⇒ 独立标定器，主项尺度取「第 0 列的主项」
        self._bal_softcl = SubsetLossBalancer(N_SUB, {"softcl": self._prof["softcl"]},
                                               BALANCE_EMA_BETA, device)
        # 门控项也是逐样本的 ⇒ 独立标定器
        self._bal_gate = SubsetLossBalancer(N_SUB, {"gate": self._prof["gate"]},
                                            BALANCE_EMA_BETA, device)
        # 交换一致性同样是逐样本的 ⇒ 独立标定器
        self._bal_swap = SubsetLossBalancer(N_SUB, {"swap": self._prof["swap"]},
                                            BALANCE_EMA_BETA, device)
        self._main_vec_col0 = None
        if self._group_subset is not None:
            self._gsub_t = torch.as_tensor(self._group_subset, device=device, dtype=torch.long)
        self._gs_ready = True

    # ---- 训练步结束后把「读出梯度」并进读出权重的 .grad ----
    def training_step(self, model, inputs, num_items_in_batch=None):
        """标准 training_step 之后调用模型的 flush_readout_grad()。
        lm_head 模式的读出梯度是手工算的（hook 累积到 6 行），必须在
        optimizer.step 之前并进 W.grad，否则优化器看不到读出梯度。
        放在 training_step（每个 micro-batch 调用一次）里，累加语义与自动求导一致。"""
        loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
        _m = model if hasattr(model, "flush_readout_grad") else self.model
        if hasattr(_m, "flush_readout_grad"):
            _m.flush_readout_grad()
        return loss

    def _get_train_sampler(self, train_dataset=None):
        """让同一目标的例句尽量落在同一个 batch（组内损失项的前提：
        随机打散时组内配对几乎永远为空）。副作用：一个 batch 通常只有 1–2 个目标
        ⇒ 跨目标比较要靠 xrank/tierank 的跨 batch 累积统计。"""
        if not self.grouped_batching:
            return super()._get_train_sampler(train_dataset)
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return TargetGroupedSampler(ds["group_index"], seed=self.args.seed)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]                    # [B, 2]
        mask = inputs.get("loss_mask")               # [B, 2] 哪些列真有标签
        gidx = inputs.get("group_index")             # [B] 目标组编号
        sidx = inputs.get("subset_index")            # [B] 每条例句属于哪个子集
        # lm_head 模式：need2 = 第 2 列有真标签的行，只有这些行跑第二次前向
        _need2 = None
        if mask is not None:
            _need2 = (mask[:, 1] > 0)
        outputs = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        input_ids2=inputs.get("input_ids2"),
                        attention_mask2=inputs.get("attention_mask2"),
                        # 模块 2：类型流的 token（只含 MWE）
                        input_ids_type=inputs.get("input_ids_type"),
                        attention_mask_type=inputs.get("attention_mask_type"),
                        need2=_need2)
        logits = outputs.logits                       # [B, 2]
        # 【为什么要掩码】小品词动词只有 1 个标签，第 2 列是占位的假标签。
        #   不屏蔽的话，模型会被要求去拟合一列恒为 0 的值，
        #   白白浪费一半输出维度，还会让这个位置的方差混进指标 2 的统计口径。
        per = F.huber_loss(logits, labels, delta=HUBER_DELTA, reduction="none")
        if mask is not None:
            main = (per * mask).sum() / mask.sum().clamp(min=1.0)
        else:
            main = per.mean()
        loss = main
        if gidx is None or sidx is None or not self.n_groups:
            return (loss, {"logits": logits}) if return_outputs else loss
        self._ensure_prof(logits.device, logits.dtype)
        gsub = self._gsub_t
        # 【★ 热路径只有 2 次同步】① torch.unique 取当前 batch 里的组 ② any() 取历史是否非空
        cur_ids = torch.unique(gidx)
        hist_any = torch.stack([(st.g_cnt > 0).any() for st in self._gs]).any()
        for j in range(logits.shape[1]):
            # 把「这一列有没有真标签」传进损失项：pv 第 2 列是占位常数 0，
            # 必须整体跳过，否则组内项会把它压成常数并经共享主干干扰第 1 列。
            _valid = (mask[:, j] if mask is not None else
                      torch.ones_like(logits[:, j]))
            _valid = _valid.to(logits.dtype)
            # ---- 主项按子集聚合（标定器的分母）----
            _mnum = _agg_by_subset(per[:, j] * _valid, sidx, N_SUB)
            _mden = _agg_by_subset(_valid, sidx, N_SUB)
            _main_vec = _mnum / _mden.clamp(min=1.0)
            _act = (_mden > 0).to(logits.dtype)
            bal = self._bals[j]
            bal.update_main(_main_vec, _act)
            if j == 0:
                self._main_vec_col0 = (_main_vec, _act)
            # 各子集在总损失里的份额：把「每子集」的贡献折算回全局尺度
            _share = _mden / _mden.sum().clamp(min=1.0)
            # ---- 组级统计（跨 batch 累积，历史 detach；矩用 float64 累加）----
            st = self._gs[j]
            cnt, s1, s2, gcnt, gs1, gs2 = st.stats(logits[:, j], labels[:, j], gidx, _valid)
            std_p = std_from_moments(cnt, s1, s2, out_dtype=logits.dtype)
            std_g = std_from_moments(gcnt, gs1, gs2, out_dtype=logits.dtype)
            for name in ("order", "mse", "guard", "xrank", "ce", "tierank"):
                _frac = self._prof[name]
                if name == "order":
                    _N, _D = order_parts(logits[:, j], labels[:, j], gidx, sidx,
                                         ORDER_TAU, _valid, _frac, N_SUB)
                elif name == "mse":
                    _N, _D = mse_parts(logits[:, j], labels[:, j], _valid, sidx, _frac, N_SUB)
                elif name == "guard":
                    _N, _D = guard_parts(std_p, std_g, cnt, gsub, self._prof["kappa"], N_SUB)
                elif name == "ce":
                    # 数字 token 上的软目标交叉熵（只在 lm_head 模式存在）
                    _dl = getattr(outputs, "digit_logits", None)
                    if _dl is None:
                        continue          # 回归头模式没有数字分布 ⇒ 本项关闭
                    _N, _D = ce_parts(_dl[:, j, :], labels[:, j], _valid, sidx, _frac, N_SUB)
                elif name == "tierank":
                    # 跨目标分数序：均值从矩还原（float64 累加，衰减对均值尺度不变）。
                    if cur_ids.numel() == 0 or not bool(hist_any):
                        continue
                    _mean_p = (s1 / cnt.clamp(min=1.0)).to(logits.dtype)
                    _mean_g = (gs1 / gcnt.clamp(min=1.0)).to(logits.dtype)
                    _hist_ok = (cnt > 0) & (gcnt > 0)
                    _N, _D = tierank_parts(_mean_p, _mean_g, cur_ids, gsub, _frac,
                                           TIERANK_MARGIN, TIERANK_TAU, _hist_ok, N_SUB)
                else:   # xrank
                    if cur_ids.numel() == 0 or not bool(hist_any):
                        continue
                    # 历史里只比较「组内例句数 ≥ 2」的目标：
                    # 单例组的 std 未定义（被置 0），参与比较会制造虚假的大小关系。
                    _hist_ok = (st.g_cnt > 0) & (gcnt >= 2)
                    _N, _D = xrank_parts(std_p, std_g, cur_ids, gsub, _frac,
                                         XRANK_MARGIN, XRANK_TAU, _hist_ok, N_SUB)
                _tv = _N / _D.clamp(min=1.0)
                _tact = (_D > 0).to(logits.dtype)
                if AUTO_BALANCE:
                    _w = bal.weight(name, _tv, _tact)
                else:
                    _w = torch.ones(N_SUB, device=logits.device)
                # 占比是「上限」而不是「目标」：guard 是 hinge，平时恒 0（权重顶上限），
                # 一旦触发若不截断会瞬间主导整步损失 ⇒ 贡献截断在该项应占的份额以内。
                _cap = (_frac / (1.0 - _frac).clamp(min=1e-3)) * _main_vec * _share
                _contrib = torch.minimum(_w * _tv * _tact, _cap)
                loss = loss + _contrib.sum()
            # ---- 把本 batch 的贡献并入历史（供下一批比较用）----
            st.commit(logits[:, j], labels[:, j], gidx, _valid)
        # ---- 软标签对比学习 L_softcl（逐样本，不按列分开 ⇒ 放在列循环之外）----
        # 对比学习作用在句子整体表征上，与列无关；量级参照用第 0 列的主项（绝对值锚）。
        _z = getattr(outputs, "proj", None)
        if _z is not None and self._bal_softcl is not None:
            _frac_sc = self._prof["softcl"]
            if float(_frac_sc.max()) > 0:
                # 样本级标签：两列有效标签的平均（pv 只有第 1 列 ⇒ 就是它）
                if mask is not None:
                    _mlab = (labels * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
                else:
                    _mlab = labels.mean(dim=1)
                _N, _D = softcl_parts(_z, _mlab, sidx, SOFTCL_LABEL_SPAN,
                                      _frac_sc, N_SUB)
                _tv = _N / _D.clamp(min=1.0)
                _tact = (_D > 0).to(logits.dtype)
                if self._main_vec_col0 is not None:
                    _mv0, _act0 = self._main_vec_col0
                    self._bal_softcl.update_main(_mv0, _act0)
                    if AUTO_BALANCE:
                        _w = self._bal_softcl.weight("softcl", _tv, _tact)
                    else:
                        _w = torch.ones(N_SUB, device=logits.device)
                    _share0 = _act0 / _act0.sum().clamp(min=1.0)
                    _cap = (_frac_sc / (1.0 - _frac_sc).clamp(min=1e-3)) * _mv0 * _share0
                    loss = loss + torch.minimum(_w * _tv * _tact, _cap).sum()
        # ---- 模块 2：门值 g 的监督（可选档，默认权重 0 ⇒ 不产生梯度）----
        # g 是逐样本的量（同一目标共享同一个 g），放列循环里会被算两遍 ⇒ 放在外面。
        _gate = getattr(outputs, "gate", None)
        if _gate is not None and self._bal_gate is not None:
            _frac_g = self._prof["gate"]
            if float(_frac_g.max()) > 0:
                _share = inputs.get("ctx_share")
                if _share is not None:
                    _N, _D = gate_parts(_gate, _share, None, sidx, _frac_g, N_SUB)
                    _tv = _N / _D.clamp(min=1.0)
                    _tact = (_D > 0).to(logits.dtype)
                    if self._main_vec_col0 is not None:
                        _mv0, _act0 = self._main_vec_col0
                        self._bal_gate.update_main(_mv0, _act0)
                        if AUTO_BALANCE:
                            _w = self._bal_gate.weight("gate", _tv, _tact)
                        else:
                            _w = torch.ones(N_SUB, device=logits.device)
                        _share0 = _act0 / _act0.sum().clamp(min=1.0)
                        _cap = (_frac_g / (1.0 - _frac_g).clamp(min=1e-3)) * _mv0 * _share0
                        loss = loss + torch.minimum(_w * _tv * _tact, _cap).sum()
        # ---- 模块 4：成分交换一致性 ----
        # 对「成分交换视图」要求：交换后第 0 个成分的分 ≈ 原来第 1 个成分的分
        # （结构性监督，把 Mod/Head 钉成「同一 MWE 的两个视角」）。
        # 只在 en-nn 真的生效（de-nn 单词形无法换序、pv 只有一列 ⇒ 分母为 0 退出）。
        # 交换视图需要第三次前向 ⇒ 每 batch 最多取 M4_SWAP_MAX_ROWS 行（随机抽）。
        _frac_sw = self._prof["swap"]
        if M4_ROLE_EMB and float(_frac_sw.max()) > 0:
            _sw_ok = inputs.get("swap_ok")
            _sw_ids = inputs.get("input_ids_swap")
            if _sw_ok is not None and _sw_ids is not None:
                _idx = (_sw_ok > 0).nonzero(as_tuple=True)[0]
                if _idx.numel() > M4_SWAP_MAX_ROWS:
                    _perm = torch.randperm(_idx.numel(), device=_idx.device)[:M4_SWAP_MAX_ROWS]
                    _idx = _idx[_perm]
                if _idx.numel() > 0:
                    _sw_kw = dict(input_ids=_sw_ids[_idx],
                                   attention_mask=inputs["attention_mask_swap"][_idx])
                    # 类型流刻意沿用原视图：类型不因成分换序而改变 ⇒ 本项只检验
                    # 「角色读取」的一致性；交换视图只需要第 0 列 ⇒ need2 全 False。
                    _sw_kw["need2"] = torch.zeros(_idx.numel(), dtype=torch.bool,
                                                  device=_sw_ids.device)
                    for _k in ("input_ids2", "attention_mask2",
                               "input_ids_type", "attention_mask_type"):
                        if _k in inputs:
                            _sw_kw[_k] = inputs[_k][_idx]
                    _sw_out = model(**_sw_kw)
                    _err = (_sw_out.logits[:, 0] - logits[_idx, 1]) ** 2      # [k]
                    _num = _agg_by_subset(_err, sidx[_idx], N_SUB)
                    _den = _agg_by_subset(torch.ones_like(_err), sidx[_idx], N_SUB)
                    _tv = _num / _den.clamp(min=1.0)
                    _tact = (_den > 0).to(logits.dtype)
                    if self._main_vec_col0 is not None:
                        _mv0, _act0 = self._main_vec_col0
                        self._bal_swap.update_main(_mv0, _act0)
                        if AUTO_BALANCE:
                            _w = self._bal_swap.weight("swap", _tv, _tact)
                        else:
                            _w = torch.ones(N_SUB, device=logits.device)
                        _share0 = _act0 / _act0.sum().clamp(min=1.0)
                        _cap = (_frac_sw / (1.0 - _frac_sw).clamp(min=1e-3)) * _mv0 * _share0
                        loss = loss + torch.minimum(_w * _tv * _tact, _cap).sum()
        # ---- 训练期健康检查（每 N 步验一次 loss 是否有限）----
        # 无 AMP 档位没有 GradScaler 兜底，NaN 梯度会被直接写进权重且训练不报错；
        # 每 N 步检查一次，非有限立刻报错，把「几小时后才发现」变成「第 N 步就停」。
        self._step += 1
        if self._finite_check_every > 0 and self._step % self._finite_check_every == 0:
            if not bool(torch.isfinite(loss.detach())):
                raise RuntimeError(
                    f"训练第 {self._step} 步的 loss 变成 NaN/Inf（当前档位无 AMP ⇒ 没有 GradScaler "
                    "兜底，坏梯度会直接写进权重）。请检查：该步的组内 std 是否退化"
                    "（analysis/diagnose_std_gradient.py 可复现 fp32 抵消误差）、"
                    "以及是否有目标只带 1 条例句（en-nn 实测存在 1 个这样的目标）。")
        return (loss, {"logits": logits}) if return_outputs else loss

    def loss_weight_snapshot(self):
        """训练结束后取一次「自适应标定出的实际权重」（允许同步，只在训练后调用）。
        取权重全程 EMA（最后一步可能恰好是缺席步，打印出 0 不代表该项被关）。
        每一项都记一份：「配置里写了但实际没生效」的项也能看出来。"""
        out = {}
        for j, bal in enumerate(self._bals or []):
            for k, v in bal.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[col{j}·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        if self._bal_softcl is not None:
            for k, v in self._bal_softcl.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[sample·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        if self._bal_gate is not None:
            for k, v in self._bal_gate.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[sample·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        if self._bal_swap is not None:
            for k, v in self._bal_swap.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[sample·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        return out


class TargetGroupedSampler(Sampler):
    """按目标分组的采样器：先打乱目标顺序，再把同一目标的例句连着吐出。
    DataLoader 会按顺序把连续下标切成一个 batch，于是同一目标的例句自然聚在一起。"""
    def __init__(self, group_index, seed=SEED, shuffle=True):
        self.groups = np.asarray(group_index)
        self.seed = seed
        self.shuffle = shuffle
        self.epoch = 0
        self._indices = self._build()

    def _build(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        uniq = np.unique(self.groups)
        if self.shuffle:
            rng.shuffle(uniq)
        order = []
        for g in uniq:
            idx = np.where(self.groups == g)[0]
            if self.shuffle:
                rng.shuffle(idx)
            order.extend(idx.tolist())
        return order

    def __iter__(self):
        return iter(self._indices)

    def __len__(self):
        return len(self._indices)

    def set_epoch(self, epoch):
        """Trainer 每轮会调用它，让每个 epoch 的目标顺序不同。"""
        self.epoch = epoch
        self._indices = self._build()


print("CompartmentTrainer / TargetGroupedSampler 已定义")

## Step8 模型训练（自动跑完全部运行）

**这一节不需要任何手动切换**：它按 Step1 的 `RUN_PLAN` 自动依次执行
「四个子集各单独训练一次」+「四个子集联合训练一次」，共 5 次。

**每一次运行都做四件事**（顺序不能换）：

1. **准备数据**：按本次参与的子集切分、构造 Dataset（`prepare_run`）。
2. **训练前评估随机初始化模型** —— 给 Step9 的「参考对照」区提供一条基线。
   少了它，「训练后 ρ=0.6」无法回答「它到底学到了什么」。
3. **正式训练**。
4. **训练后评估**：与随机初始化**完全相同的口径**（同一份验证集、同一个
   `compute_metrics`），唯一区别是模型权重。口径不对齐的对照没有意义。

**第一轮额外做三项开训前检查**（只做一次，因为它们与数据规模无关）：

- **显存探测**：逐档真跑 2 步，量峰值显存，取能跑通的最大 micro-batch，
  再用梯度累积把有效 batch 补回 8。
- **梯度流验证**：手动 `backward()` 一次，打印「有多少参数真的拿到了梯度」。
  能抓到的典型 bug 是「主干被冻结 / 回归头没接上」——这些**不会报错**，
  只会让模型训半天纹丝不动。
- **冒烟测试**：`max_steps=5` 跑通「前向 → 损失 → 反向 → 参数更新 → 评估」
  整条链路。直接开长训，一旦中途报错，要么白等几十分钟，要么分不清是 bug 还是超时。

**为什么每次运行都要重建模型**：冒烟测试那 5 步、显存探测那 2 步探针都会改动权重。
正式训练必须从**干净的随机初始化**开始，否则基线不干净、不可复现。

In [ ]:
# ============ 8.1 组装 Trainer ============
def should_check_consistency(overrides, check_consistency=None):
    """这个 Trainer 要不要跑「口径一致性断言」？

    口径断言按 epochs 推 warmup_steps，而体检/冒烟 trainer 会故意把
    num_train_epochs 改成 1 ⇒ 断言必然拿错口径报「漂移」。
    规则：只要 overrides 动了 max_steps / num_train_epochs 就自动关闭断言；
    正式训练路径永远保留。判定收敛在这一个函数里，不靠调用点自觉。
    """
    if check_consistency is not None:
        return bool(check_consistency)
    return not (("max_steps" in overrides) or ("num_train_epochs" in overrides))


def make_trainer(m, out_dir, data, epochs=None, check_consistency=None, **overrides):
    """组装一个 Trainer。
    每次都重新构造 args（Trainer 会就地改写 TrainingArguments，共用对象会在
    第二个 train() 时抛 zip 长度错误）；data 里的 train_ds / val_ds / eval_cols
    必须与本次运行严格对应。epochs 可按运行覆盖（joint 用 JOINT_EPOCHS）。
    口径断言的开关由 should_check_consistency 自动判定：体检/冒烟 trainer
    （动了 max_steps / num_train_epochs）自动关闭，正式训练永远保留。"""
    # 自动判定：动了训练步数/轮数的 trainer 一律不做口径断言（见上面的函数）
    _probed = ("max_steps" in overrides) or ("num_train_epochs" in overrides)
    check_consistency = should_check_consistency(overrides, check_consistency)
    if _probed and check_consistency is not True:
        print(f"   （体检用途的 Trainer：覆盖了 {sorted(set(overrides) & {'max_steps', 'num_train_epochs'})} "
              "⇒ 自动关闭口径断言；正式训练路径仍保留）")
    _args = build_training_args(out_dir, len(data["train_ds"]), epochs=epochs, **overrides)
    # 守住「单卡」：n_gpu > 1 时 Trainer 自动套 nn.DataParallel（反向复制一整份梯度，
    # 显存翻倍）。断言守住：环境变量万一失效，当场报错而不是静默被套上。
    _n_gpu = int(getattr(_args, "n_gpu", 1) or 1)
    assert _n_gpu == 1, (
        f"args.n_gpu={_n_gpu} > 1：Trainer 会自动套 nn.DataParallel，"
        "其反向传播会复制一整份梯度导致 OOM。请用 COMPARTMENT_GPU=<卡号> 只暴露一张卡。")

    # 开训前把「配置常量」与「Trainer 实际生效的口径」对齐（清单只打印配置变量
    # 会出现静默漂移）；显存探针例外——它故意覆盖 batch 参数，自动豁免。
    if check_consistency:
        _assert_args_consistent(_args, n_train=len(data["train_ds"]))

    # 【★ 组内统计需要知道「总共有多少个组」】index_add_ 的输出长度必须固定，
    #   否则就会退化成变长操作（带回同步）。这里从训练集里取 group_index 的最大值。
    _n_groups = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1
    # 每个目标组属于哪个子集（损失按子集差异化时要用）；
    # 组与子集一一对应，直接从训练集的 (group_index, subset_index) 两列建映射。
    _gs_map = np.zeros(_n_groups, dtype=np.int64)
    _gs_map[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")
    assert _gs_map.max() < len(SUBSET_ORDER), (
        f"目标组的子集编号最大 {_gs_map.max()}，超出 SUBSET_ORDER 长度 {len(SUBSET_ORDER)}")

    return CompartmentTrainer(
        model=m,
        args=_args,
        train_dataset=data["train_ds"],
        eval_dataset=data["val_ds"],
        data_collator=collator,
        # 字典展开：transformers 5 的参数叫 processing_class，4.x 叫 tokenizer（已探测）
        **{_TOKENIZER_KW: tokenizer},
        compute_metrics=make_compute_metrics(
            data["val_ds"]["group_index"], data["val_ds"]["subset_index"],
            data["eval_cols"], SUBSET_ORDER),
        # 必须传全局的 SUBSET_ORDER：subset_index 是全局编号，
        # 传「本次运行的子集清单」会在单子集运行时越界。
        grouped_batching=GROUPED_BATCHING,
        n_groups=_n_groups,
        group_subset=_gs_map,
    )


# ============ 8.2 显存探测：按「档位」逐级试探，OOM 就自动降一档 ============
# 先选档位、再探 batch：档位负责降固定开销，batch 负责降激活
#（全参微调的固定开销与 batch 无关，只降 batch 试不出来）。
# 实现要点：用同一个模型去探（deepcopy 1.5B 要多 6GB 显存），探完重建干净模型；
# 每档之间彻底释放；判 OOM 只看「out of memory」，其它异常原样抛出；
# 非 CUDA 环境直接跳过。
class FitOOMError(RuntimeError):
    """显存不够（而不是别的 bug）。
    单独的异常类型：把「显存不够」和「代码有 bug」严格分开，
    真 bug 一律原样抛出，绝不能被当成显存不足静默降档。"""


class FitNumericError(RuntimeError):
    """档位级的数值不稳（该档位不可用），但不是代码 bug。

    特征是「loss 正常、grad_norm 非有限」（如 AMP 的 fp16 溢出）⇒ 换档位能解决，
    归入降档链；真正指向代码逻辑的报错（KeyError / AttributeError / 断言失败）
    一律原样抛出，绝不降档掩盖。"""


def _is_oom(exc):
    return "out of memory" in str(exc).lower()


def release_gpu_memory(tag=""):
    """释放显存，并打印「已分配 / 已保留」快照。
    每个关键节点都要显式释放：旧模型的权重/梯度/优化器状态挂着会逐次累积，
    最后在正式训练里以「只差几十 MiB」的形式炸掉；快照让那一步占着显存可查。"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _a = torch.cuda.memory_allocated() / (1024 ** 3)
        _r = torch.cuda.memory_reserved() / (1024 ** 3)
        _c = _vram_total_gb() or 0.0
        print(f"[显存·{tag or '释放'}] 已分配 {_a:.2f} GiB | 已保留 {_r:.2f} GiB"
              f" | 卡 {_c:.2f} GiB")
    else:
        print(f"[显存·{tag or '释放'}] 非 CUDA 环境，跳过")


def _probe_reserve_gb():
    """探测阶段给后续步骤留出的显存余量（GiB）。
    探针的判据是「峰值 + 余量 ≤ 卡容量」而不是「不 OOM」：preflight 还会
    多持一份计算图、一份稠密梯度、一次 Trainer。
    余量 0.8 GiB 只覆盖碎片与 CUDA context——探针与正式训练同模型结构、同梯度累积。
    关键判断：余量不足 ≠ 这档不行。峰值由固定开销主导、与 batch 几乎无关，
    降 micro-batch 降不下峰值；把「余量不足」当降档理由会让连 bs=1 都被否
    ⇒ 自动降到只训部分层的档位 = 改掉了训练范围 ⇒ 保留档位、打印告警、继续用。"""
    try:
        return float(FIT_PROBE_RESERVE_GB)
    except Exception:
        return 0.8


def probe_fit_tier(model, data, tier, candidates=(8, 4, 2, 1), steps=2):
    """在给定档位内逐个 batch 真跑，返回 (bs, peak_gb, sps)。
    返回 None 只表示「所有 batch 档位都真的 OOM」⇒ 调用方据此降档。
    判据是「跑通且尽量留有余量」：优先选「峰值 + 余量 ≤ 卡容量」的 batch；
    找不到就选能跑通的最大 batch 并打印告警，保留档位（只有全 OOM 才降档）。"""
    if not torch.cuda.is_available():
        print("非 CUDA 环境（CPU/MPS）：跳过显存探测，沿用配置中的 batch size")
        return None
    _card = _vram_total_gb() or 0.0
    _reserve = _probe_reserve_gb()
    _fallback = None          # (bs, peak, sps)：跑得通但没满足余量判据的最好候选
    for bs in candidates:
        _tr = None
        try:
            _tr = make_trainer(model, OUTPUT_ROOT / "_probe", data,
                               per_device_train_batch_size=bs,
                               # 探针与正式训练用相同的梯度累积：
                               # accum>1 时 grads 在第二个 micro-batch 期间已占着显存，
                               # 探针若用 accum=1，量到的峰值就不等于正式训练的峰值。
                               gradient_accumulation_steps=GRAD_ACCUM,
                               max_steps=steps,
                               save_strategy="no",
                               load_best_model_at_end=False,
                               logging_steps=steps,
                               # 探针故意改 batch/累积 → 关掉口径一致性断言，
                               # 否则 8 失败转 4 时会抛 AssertionError 而非优雅降档。
                               check_consistency=False,   # 体检用途：故意改 batch/累积
                               optim=tier["optim"],
                               gradient_checkpointing=tier["ckpt"],
                               dataloader_num_workers=0, dataloader_persistent_workers=False,
                               dataloader_multiprocessing_context=None,
                               dataloader_prefetch_factor=None,
                               **{_EVAL_STRATEGY_KEY: "no"})
            torch.cuda.reset_peak_memory_stats()
            _t0 = time.time()
            _tr.train()
            _dt = time.time() - _t0
            peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
            # 探针阶段的梯度有限性检查（这一步真的训练了 2 步）：
            # grad_norm 非有限 = 档位的数值精度问题（如 AMP fp16 溢出）而非代码 bug
            # ⇒ 抛 FitNumericError（可降档），不把整个 notebook 送走。
            _gn = [h.get("grad_norm") for h in _tr.state.log_history if h.get("grad_norm") is not None]
            _bad_gn = []
            for _g in _gn:
                try:
                    _gv = float(_g)
                except (TypeError, ValueError):
                    continue
                if not math.isfinite(_gv):
                    _bad_gn.append(_gv)
            if _bad_gn:
                raise FitNumericError(
                    f"探针阶段的梯度已经是 NaN/Inf（grad_norm={_bad_gn[:3]}），"
                    f"但 loss 与数据都正常 ⇒ 这是**档位的数值精度问题**"
                    f"（{'AMP/fp16 溢出' if tier.get('amp') else 'fp32 精度问题'}）而不是代码 bug。"
                    "处理方式：换下一个档位重试。"
                    "若全部档位都这样，再查：① 组内 std 的 fp32 抵消误差"
                    "（analysis/diagnose_std_gradient.py）；② 只有 1 条例句的目标"
                    "（en-nn 实测存在 1 个）。")
            # 【★ 顺便量出每样本耗时】用来投影「5 次运行」的总训练时长，
            #   好在真正开跑之前就知道会不会撞上 Kaggle 的 12 小时上限。
            sps = _dt / max(1, steps * bs)
            # 留余量的判据：峰值 + 余量 ≤ 卡容量
            _need = peak + _reserve
            _fits = (_need <= _card)
            print(f"  {tier['name']} micro-batch={bs}: 通过，峰值显存 {peak:.2f} GiB，"
                  f"每样本 {sps * 1000:.2f} ms"
                  f"（+余量 {_reserve:.1f} GiB = {_need:.2f} GiB / 卡 {_card:.2f} GiB"
                  f" ⇒ {'余量充足 ✓' if _fits else '余量不足 ✗（但仍保留本档位）'}）")
            if _fits:
                return bs, peak, sps
            # 余量不足不是降档的理由 ⇒ 记下来当备选，继续试更小的 batch；
            # 备选取「能跑通的最大 batch」：峰值由固定开销主导，降 batch 换不来多少余量，
            # 却要赔上好几倍速度；真 OOM 有正式训练的「自动重试」兜底。
            if _fallback is None:
                _fallback = (bs, peak, sps)
        except FitNumericError:
            raise            # 数值不稳：交给降档链处理（本档位直接放弃）
        except RuntimeError as e:
            if not _is_oom(e):
                raise
            # 【★ 把真实的 OOM 信息打出来】只打印一句「OOM」没法定位：
            #   「Tried to allocate X MiB」与「还剩多少」直接对应上原因。
            _lines = [l for l in str(e).strip().splitlines() if l.strip()]
            _head = _lines[0][:150] if _lines else ""
            _detail = next((l for l in _lines if "Tried to allocate" in l or "GiB is free" in l), "")
            print(f"  {tier['name']} micro-batch={bs}: OOM | {_head}")
            if _detail:
                print(f"      {_detail.strip()[:220]}")
        finally:
            if _tr is not None:
                del _tr
            gc.collect()
            torch.cuda.empty_cache()
    # 【★ 走到这里 = 没有任何 batch 完全 OOM，但都没满足余量判据】
    if _fallback is not None:
        _b, _p, _s = _fallback
        print(f"  ⚠️ 档位 {tier['name']}：没有哪个 batch 能同时满足「余量 {_reserve:.1f} GiB」，"
              f"但 micro-batch={_b} 跑得通（峰值 {_p:.2f} GiB / 卡 {_card:.2f} GiB）")
        print("     ⇒ **保留本档位**（余量不足不是降档的理由：Qwen 的峰值由固定开销主导，"
              "降 micro-batch 降不下峰值；降档会让两条基线不同档 ⇒ 对照结论作废）")
        print(f"     ⇒ 采用**能跑通的最大 batch**={_b}（峰值由固定开销主导，降 batch 换不来多少余量，"
              "却要赔上好几倍速度；真 OOM 了有「正式训练自动重试」兜底）")
        print(f"     ⇒ 后续的 preflight 已做三项降峰值处理（开 checkpointing / 样本数减半 / "
              f"释放计算图），并使用探测出的 micro-batch={_b}")
        return _fallback
    return None            # 所有 batch 都真 OOM ⇒ 调用方降档


# ============ 训练时长投影：开跑前就知道会不会撞上 12 小时上限 ============
def _total_train_rows():
    """四个子集的总行数（只数行，很便宜）→ 训练时长投影的输入。"""
    tot = 0
    for _name in SUBSET_ORDER:
        _p = DATA_ROOT / f"{_name}-train.tsv"
        with open(_p, "r", encoding="utf-8") as _f:
            tot += max(0, sum(1 for _ in _f) - 1)      # 去掉表头
    return tot


def project_train_time(sec_per_sample):
    """按**本运行计划**投影总训练时长（逐运行查运行级轮数表）。
    每个运行取其训练集的 80% × 该运行的轮数；1.3 系数给评估/预测/出图留余量
    （该系数用 v4.4 的「投影 7.8h vs 实际 7.91h」校准过）。"""
    n_rows = _total_train_rows()
    per_epoch = 0.8 * n_rows
    _eps = sum(RUN_EPOCHS_OVERRIDE.get(
        (f"{m}_{s}" if s else m),
        JOINT_EPOCHS if m.startswith("joint") else EPOCHS)
        for m, s in RUN_PLAN)
    total = per_epoch * _eps
    hours = total * sec_per_sample * 1.3 / 3600
    return n_rows, total, hours


def prepare_for_training(build_fn, data):
    """选档 → 探测 → 开训前检查；任何一步 OOM 就自动降一档重来。

    优化器不许自动切换（切了数值就不可比且没人察觉）；档位可以降，
    但 FIT_MODE 显式指定时锁定档位、禁止降档——降档会改变训练范围
    （只训部分层），而机制实验要求全参微调；这一档跑不通就直接报错。
    preflight 也纳入降档链：它会真的建优化器状态，是必然 OOM 的第一现场。"""
    vram = _vram_total_gb()
    tier0, _rows = resolve_fit_tier(vram, FIT_REF_PARAMS)
    start = FIT_TIERS.index(tier0)
    # 显式指定档位 ⇒ 锁定，不许降档。
    # 锁的前提是「这个档位真的被选中了」：若 FIT_MODE 写的档位在当前环境不可用，
    # resolve_fit_tier 会退回自动选择 ⇒ 此时不能锁（否则一进循环就 break）。
    _locked = ""
    if FIT_MODE != "auto" and str(FIT_MODE).lower() == tier0["name"]:
        _locked = tier0["name"]
    if _locked:
        print(f"★ 档位已锁定为 {_locked}（FIT_MODE 显式指定）⇒ **不自动降档**")
        print("   （降档 = 冻住一半主干 = 换了实验；本版机制要求全参微调。"
              "若这一档真跑不通，请先查「有效 batch / micro-batch / 读出结构」是否与设计一致，"
              "再考虑改档位，并把改动写进口径清单）")
    last_err = ""
    for i in range(start, len(FIT_TIERS)):
        t = FIT_TIERS[i]
        if _locked and t["name"] != _locked:
            break                       # 锁档：到点就停，不往下走
        if not _tier_available(t):
            print(f"跳过档位 {t['name']}：当前环境不可用")
            continue
        apply_tier(t)
        print("=" * 78)
        print(f"试探档位 {t['name']}：{t['label']} | optim={t['optim']} | "
              f"可训层比例={t['scope']:.0%} | 混合精度={'开' if t['amp'] else '关'} | "
              f"gradient_checkpointing={t['ckpt']} | "
              f"固定开销 {tier_estimate_gb(FIT_REF_PARAMS, t)[0]:.1f} GiB（按参考 {FIT_REF_PARAMS / 1e6:.0f}M 算）")
        print("=" * 78)
        model = None
        try:
            model = build_fn()      # 按当前档位重建（冻结范围可能随档位变化）
            if PROBE_MEMORY and torch.cuda.is_available():
                res = probe_fit_tier(model, data, t)
                if res is None:
                    last_err = f"档位 {t['name']}：所有 batch 档位都 OOM 或余量不足"
                    print(f"→ {last_err}，自动降一档重试")
                    continue
                bs, peak, sps = res
            else:
                print("非 CUDA 环境或 PROBE_MEMORY=False：跳过显存探测，沿用配置中的 batch size")
                bs, peak, sps = BATCH_SIZE, float("nan"), float("nan")
            # 探针之后必须重建一个「干净」模型：探针真的训练了 2 步，权重已被改过
            # （若那 2 步产生 NaN 梯度，不重建的话 preflight 拿到的是脏模型，
            # 报错会被带偏）。preflight 必须在干净权重上做检查。
            del model
            model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            model = build_fn()
            try:
                run_preflight_checks(model, data, OUTPUT_ROOT / "_preflight", micro_bs=bs)
            except FitOOMError as e:
                last_err = f"档位 {t['name']}：开训前检查阶段仍 OOM（{str(e)[:160]}）"
                print(f"→ {last_err}，自动降一档重试")
                print("   ⚠️ 若最终落在与另一条基线**不同**的档位 ⇒ 训练范围不同，结论不可混谈！")
                print("      ⇒ 本版机制要求**全参微调**（t3）；先查有效 batch 与 micro-batch 是否符合设计，"
                      "再决定要不要改档。")
                continue
            return t, bs, peak, sps
        except FitNumericError as e:
            # 档位级数值不稳 ⇒ 换下一档重试（与 OOM 同级），而不是弄死整本 notebook。
            last_err = f"档位 {t['name']}：数值不稳（{str(e)[:200]}）"
            print(f"→ {last_err}，自动降一档重试")
            continue
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    # 全部档位都失败：给出**精确诊断**，而不是一句「out of memory」
    n_params = FIT_REF_PARAMS
    _lines = [f"档位 {t['name']}：{t['label']} → 固定开销 "
              f"{tier_estimate_gb(n_params, t)[0]:.1f} GiB" for t in FIT_TIERS]
    _lock_txt = (f"\n  ⚠️ 档位已锁定为 {FIT_MODE}（FIT_MODE 显式指定）⇒ 不会自动降档。\n"
                 f"    ★ 若要换档，请记住：本版机制要求**全参微调**（t3）；"
                 f"换档后训练范围改变，这一版的数字就不能和别的档位的结果放在一起说。"
                 if _locked else "")
    raise RuntimeError(
        "所有训练档位都跑不通。\n"
        f"  本卡容量          : {vram:.1f} GiB\n"
        f"  参考参数量        : {n_params / 1e6:.0f}M（两条基线中较大者）\n"
        f"  最后一次失败原因  : {last_err}\n" + _lock_txt + "\n"
        "  各档位固定开销    : " + " | ".join(_lines) + "\n"
        "  可选方案：\n"
        "    ① 选更大显存的实例（如 L4 24GB / A100 40GB）—— 档位会自动回到 t0/t1\n"
        "    ② 换更小的同族模型（deberta-v3-base / Qwen2.5-0.5B-Instruct），管线一行不用改\n"
        "    ③ ZeRO-3 / FSDP 分片（deepspeed）\n"
        "    ④ 手动指定档位：把 FIT_MODE 写成 't0'~'t6'"
        "（本版要求 t3 = 全参微调；换档会改变训练范围，结果不可跨档比较）")


print("Trainer 工厂与档位探测就绪")

In [ ]:
# ============ 8.3 训练前的两项检查（只做一次）============
def run_preflight_checks(model, data, out_dir, micro_bs=None):
    """梯度流验证 + 冒烟测试（外层只干一件事：把 OOM 翻译成 FitOOMError）。
    micro_bs 必须传探测出的 batch：手工前向/反向按它做，不能写死 8。"""
    try:
        _preflight_body(model, data, out_dir, micro_bs=micro_bs)
    except RuntimeError as e:
        if _is_oom(e):
            raise FitOOMError(str(e)[:300]) from e
        raise


def _preflight_body(model, data, out_dir, micro_bs=None):
    """梯度流验证 + 冒烟测试。
    梯度流验证抓「主干被冻结 / 读出没接上」这类不报错的白训；
    冒烟测试抓 API 层面的错误（长训中途报错，前面全是沉没成本）。
    这两步是整条链路最吃显存的地方（真的建优化器状态）。

    显存上的四个关键点：
      ① 手工前向/反向必须显式开 gradient checkpointing——
         checkpointing 是 Trainer 在自己构造时开启的，preflight 拿到的是
         探针之后重建的新模型，从未被 Trainer 碰过 ⇒ 不开则激活全量驻留（~1 GiB）；
      ② 手工前向样本数要小（梯度流验证只需证明梯度接上了）；
      ③ 真实损失反向用探测出的 batch（超出探测验证过的范围就是必然 OOM）；
      ④ 手工前向的计算图要在真实损失反向之前释放（两份激活不能同时在显存里）。"""
    # ---- 把当前档位的 gradient checkpointing 同步到模型 ----
    if USE_GRAD_CKPT and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        print(f"preflight：已按当前档位开启 gradient checkpointing"
              f"（档位 {FIT_TIER_NAME}，关掉它手工反向会多占 ~1 GiB 激活）")
    _mb = int(micro_bs) if micro_bs else int(BATCH_SIZE)
    _mb = max(1, _mb)

    # ---- 梯度流验证 ----
    model.train()
    _n_fwd = max(1, min(2, _mb))
    _gb = collator([data["train_ds"][i] for i in range(_n_fwd)])
    _gb = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in _gb.items()}
    model.zero_grad(set_to_none=True)
    _go = model(input_ids=_gb["input_ids"], attention_mask=_gb["attention_mask"],
                input_ids2=_gb.get("input_ids2"), attention_mask2=_gb.get("attention_mask2"),
                # 模块 2：类型流 token（双流打开时必须一起传，否则融合会报错）
                input_ids_type=_gb.get("input_ids_type"),
                attention_mask_type=_gb.get("attention_mask_type"),
                need2=((_gb["loss_mask"][:, 1] > 0) if "loss_mask" in _gb else None))
    _gl = F.huber_loss(_go.logits, _gb["labels"], delta=HUBER_DELTA)
    _gl.backward()
    # lm_head 模式读出梯度是手工累积的 ⇒ 反向后必须 flush 一次，
    # 否则「读出权重梯度」检查会读到 0 而误报「读出没接上」。
    if hasattr(model, "flush_readout_grad"):
        model.flush_readout_grad()
    _named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    _with_grad = [(n, p) for n, p in _named if p.grad is not None and float(p.grad.abs().sum()) > 0]
    # lm_head 模式没有 model.head，读出通路是主干自带的 LM 头（与 embedding 共享权重）
    if getattr(model, "head", None) is not None:
        _head_g = float(model.head.weight.grad.abs().sum()) if model.head.weight.grad is not None else 0.0
        _head_name = "回归头权重梯度绝对值和"
    else:
        _rw = model.readout_weight()
        _head_g = float(_rw.grad.abs().sum()) if _rw.grad is not None else 0.0
        _head_name = "LM 头权重梯度绝对值和"
    _bb_p = next(model.backbone.parameters())
    _bb_g = float(_bb_p.grad.abs().sum()) if _bb_p.grad is not None else 0.0
    print(f"loss                   : {_gl.item():.4f}   （NaN? {bool(torch.isnan(_gl))}）")
    print(f"可训练参数             : {len(_named)}")
    print(f"拿到非零梯度的参数     : {len(_with_grad)}  ({len(_with_grad) / max(1, len(_named)):.1%})")
    print(f"{_head_name:22s}: {_head_g:.6g}   （必须 > 0，否则读出通路没接上）")
    print(f"主干首参数梯度绝对值和 : {_bb_g:.6g}   （必须 > 0，否则主干被冻结）")
    assert _head_g > 0, "读出通路没有梯度：回归头/lm_head 没接进计算图"
    assert _bb_g > 0, "主干没有梯度：主干被冻结了"
    assert not torch.isnan(_gl), "loss 出现 NaN：多半是 fp16 加载或标签里有脏值"

    # ---- 模块 2：双流是否真的接上（类型头 / 门头必须有梯度）----
    # 双流是静默失效型结构：类型流没进计算图时分数与指标都正常，只是模块没生效。
    if M2_DUAL_STREAM:
        _tg = float(model.type_head.weight.grad.abs().sum()) if model.type_head.weight.grad is not None else 0.0
        _gg = float(model.gate_head.weight.grad.abs().sum()) if model.gate_head.weight.grad is not None else 0.0
        print(f"类型头权重梯度绝对值和 : {_tg:.6g}   （必须 > 0，否则类型流没接上）")
        print(f"门头权重梯度绝对值和   : {_gg:.6g}   （必须 > 0，否则 g 是常数 ⇒ 退化成单流）")
        assert _tg > 0, "类型流没有梯度：双流没接上（类型头拿不到梯度）"
        assert _gg > 0, "门没有梯度：g 是常数 ⇒ 模块 2 退化成单流"

    # ---- 各损失项的数值体检（按子集聚合的新接口）----
    # 开训前查：group_index / subset_index 的 dtype 或形状不对时，
    # 报错若发生在训练中途，前面全是沉没成本。
    try:
        _gidx = _gb.get("group_index")
        _sidx = _gb.get("subset_index")
        if _gidx is not None and _sidx is not None:
            _v = _gb["loss_mask"][:, 0] if "loss_mask" in _gb else None
            _p0 = _go.logits[:, 0].detach()
            _y0 = _gb["labels"][:, 0].detach()
            _vf = _v if _v is not None else torch.ones_like(_p0)
            _n_g = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1
            _gsub_np = np.zeros(_n_g, dtype=np.int64)
            _gsub_np[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")
            _gsub = torch.tensor(_gsub_np, device=_p0.device, dtype=torch.long)
            _prof = profile_vectors(_p0.device, _p0.dtype)
            _n, _d = order_parts(_p0, _y0, _gidx, _sidx, ORDER_TAU, _v, _prof["order"], len(SUBSET_ORDER))
            _r = _n / _d.clamp(min=1.0)
            _n, _d = mse_parts(_p0, _y0, _vf, _sidx, _prof["mse"], len(SUBSET_ORDER))
            _ms = _n / _d.clamp(min=1.0)
            _gs0 = GroupStats(_n_g, _p0.device)      # dtype 由设备能力决定
            _c, _s1, _s2, _gc, _gs1, _gs2 = _gs0.stats(_p0, _y0, _gidx, _vf)
            _sp = std_from_moments(_c, _s1, _s2, out_dtype=_p0.dtype)
            _sg = std_from_moments(_gc, _gs1, _gs2, out_dtype=_p0.dtype)
            _n, _d = guard_parts(_sp, _sg, _c, _gsub, _prof["kappa"], len(SUBSET_ORDER))
            _gd = _n / _d.clamp(min=1.0)
            _n, _d = xrank_parts(_sp, _sg, torch.unique(_gidx), _gsub, _prof["xrank"],
                                 XRANK_MARGIN, XRANK_TAU,
                                 torch.ones(_n_g, dtype=torch.bool, device=_p0.device),
                                 len(SUBSET_ORDER))
            _xr = _n / _d.clamp(min=1.0)
            # 新损失项也做同样的体检（ce / softcl）
            _ce_txt = ""
            _dl = getattr(_go, "digit_logits", None)
            if _dl is not None:
                # 【★ 注意维度】ce_parts 收的是 [B, C, 6]（B 样本 × C 列 × 6 个数字），
                #   这里取第 0 列时**必须保留列维**（[:, 0:1, :]），否则会变成 [B, 6]。
                _n, _d = ce_parts(_dl[:, 0:1, :], _gb["labels"][:, 0:1], _vf, _sidx,
                                  _prof["ce"], len(SUBSET_ORDER))
                _ce = _n / _d.clamp(min=1.0)
                assert torch.isfinite(_ce).all(), "L_ce 出现 NaN/Inf：检查数字分布与标签的形状"
                _ce_txt = f"  L_ce={_ce.tolist()}"
            _sc_txt = ""
            _z = getattr(_go, "proj", None)
            if _z is not None:
                _mlab = (_gb["labels"] * _gb["loss_mask"]).sum(1) / _gb["loss_mask"].sum(1).clamp(min=1.0)
                _n, _d = softcl_parts(_z, _mlab, _sidx, SOFTCL_LABEL_SPAN,
                                      _prof["softcl"], len(SUBSET_ORDER))
                _sc = _n / _d.clamp(min=1.0)
                assert torch.isfinite(_sc).all(), "L_softcl 出现 NaN/Inf：检查投影向量是否全零"
                _sc_txt = f"  L_softcl={_sc.tolist()}"
            print(f"损失项体检（按子集聚合）: L_order={_r.tolist()}  L_mse={_ms.tolist()}")
            print(f"                        L_guard={_gd.tolist()}  L_xrank={_xr.tolist()}"
                  f"{_ce_txt}{_sc_txt}")
            assert torch.isfinite(_r).all() and torch.isfinite(_ms).all() \
                and torch.isfinite(_gd).all() and torch.isfinite(_xr).all(), \
                "损失项出现 NaN/Inf：先查 group_index / subset_index 的 dtype 与形状"
    except AssertionError:
        raise
    except Exception as _e:
        raise AssertionError(f"损失项体检失败（{type(_e).__name__}: {_e}）：检查 group_index / subset_index / loss_mask 的形状与 dtype") from _e
    model.zero_grad(set_to_none=True)
    # 释放手工前向的计算图与中间张量：_go 持有第一次前向的整条计算图，
    # 与后面的「真实损失反向」两份激活同时在显存里会 OOM。
    del _go
    _go = None
    _gb = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- 走真实损失的反向，逐参数检查梯度是否有限 ----
    # 必须走完整的 compute_loss：只测 Huber 的反向覆盖不了全部损失分项，
    # NaN 会从没被覆盖的分项漏过去。逐参数检查才能指出「哪些参数」坏了。
    # 体检 trainer 用探测出的 batch（不能超出探测验证过的范围）且累积=1。
    _gc_trainer = make_trainer(model, out_dir / "_gradcheck", data,
                                max_steps=1, num_train_epochs=1,
                                per_device_train_batch_size=_mb,
                                gradient_accumulation_steps=1,
                                save_strategy="no", load_best_model_at_end=False,
                                logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})
    _n_real = max(1, min(int(_mb), len(data["train_ds"])))
    _probe_batch = collator([data["train_ds"][i] for i in range(_n_real)])
    _probe_batch = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in _probe_batch.items()}
    model.zero_grad(set_to_none=True)
    _real_loss = _gc_trainer.compute_loss(model, _probe_batch)
    _real_loss.backward()
    # 读出通路的梯度是手工累积的 ⇒ 反向后必须 flush 一次（否则不进 .grad）
    if hasattr(model, "flush_readout_grad"):
        model.flush_readout_grad()
    _bad = []
    _gmax = 0.0
    for _n, _p in model.named_parameters():
        if _p.grad is None:
            continue
        if torch.isnan(_p.grad).any() or torch.isinf(_p.grad).any():
            _bad.append(_n)
        _gmax = max(_gmax, float(_p.grad.abs().max()))
    print(f"真实损失反向          : loss={float(_real_loss):.4f} | 最大梯度={_gmax:.3e} | "
          f"非有限梯度的参数数={len(_bad)}")
    assert torch.isfinite(_real_loss), f"真实损失不是有限值（{float(_real_loss)}）"
    assert not _bad, (
        f"真实损失的反向产生了非有限梯度，涉及 {len(_bad)} 个参数（前 5: {_bad[:5]}）。"
        "这正是云端崩溃的根因类别：坏梯度被写进权重 ⇒ 之后 loss 全是 NaN。"
        "排查顺序：① 上面各分项的数值；② analysis/diagnose_std_gradient.py 的抵消误差；"
        "③ 是否有目标只带 1 条例句（en-nn 实测存在 1 个这样的目标）。")
    model.zero_grad(set_to_none=True)
    del _gc_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("→ 梯度流验证通过（含真实损失的四项反向）。")

    # ---- 冒烟测试 ----
    # 体检用途的 trainer（5 步、1 轮）⇒ 口径断言自动关闭；
    # 必须用探测出的 micro-batch：冒烟训练真的建优化器状态，是 preflight 里
    # 显存需求最高的一环，用回全局 batch 等于白探。
    _st = make_trainer(model, out_dir / "_smoke", data,
                       max_steps=5, num_train_epochs=1,
                       per_device_train_batch_size=_mb,
                       gradient_accumulation_steps=1,
                       save_strategy="no", load_best_model_at_end=False,
                       logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})
    _so = _st.train()
    del _st
    gc.collect()
    assert np.isfinite(_so.training_loss), "冒烟训练 loss 非有限值"
    print("冒烟训练 loss 轨迹:", _so.training_loss)
    print("→ 冒烟测试通过（每个 Trainer 都是独立构造 args，不复用对象）。")

In [ ]:
# ============ 8.35 诊断量采集：把「模块内部长什么样」变成可读出的数字 ============
# 模块 1 的分布宽度是否反映歧义度、模块 2 的门 g 是否在四个子集上分化，
# 都不会体现在三项官方指标里 ⇒ 必须单独采集。
# Trainer.predict 只吐 logits，拿不到这些 ⇒ 自己用 no_grad 纯推理跑一遍。
def collect_model_diagnostics(model, data, batch_size=None):
    """在验证集上跑一遍推理，收集模型的内部读出（分布宽度 / 门值 g）。
    返回 dict：preds [N, n_out]、widths [N, n_out]（可能为 None）、gates [N]（可能为 None）。"""
    _bs = batch_size or BATCH_SIZE
    _ds = data["val_ds"]
    _model = model
    if hasattr(_model, "module"):          # DataParallel 包装（正常路径不会发生）
        _model = _model.module
    _model.eval()
    _P, _W, _G = [], [], []
    with torch.no_grad():
        for _i in range(0, len(_ds), _bs):
            _feat = [_ds[_j] for _j in range(_i, min(_i + _bs, len(_ds)))]
            _batch = collator(_feat)
            _kw = {}
            for _k in ("input_ids2", "attention_mask2",
                       "input_ids_type", "attention_mask_type"):
                if _k in _batch:
                    _kw[_k] = _batch[_k].to(DEVICE)
            if "loss_mask" in _batch:
                _kw["need2"] = (_batch["loss_mask"][:, 1] > 0).to(DEVICE)
            _out = _model(input_ids=_batch["input_ids"].to(DEVICE),
                          attention_mask=_batch["attention_mask"].to(DEVICE),
                          **_kw)
            _P.append(_out.logits.detach().float().cpu().numpy())
            _ws = getattr(_out, "dist_std", None)
            if _ws is not None:
                _W.append(_ws.detach().float().cpu().numpy())
            _g = getattr(_out, "gate", None)
            if _g is not None:
                _G.append(_g.detach().float().cpu().numpy().reshape(-1))
    _res = dict(preds=np.concatenate(_P, axis=0) if _P else None,
                widths=np.concatenate(_W, axis=0) if _W else None,
                gates=np.concatenate(_G, axis=0) if _G else None)
    return _res

In [ ]:
# ============ 8.4 训练主循环：按 RUN_PLAN 自动跑完所有运行 ============
# 写成循环而不是手动切换：跑漏/切错的错误在结果表里看不出来；
# 循环保证 5 次运行的口径、产物路径、汇总方式完全一致。
ALL_RUNS = []      # 每次运行的结果都存这里，Step9 统一汇总

for _i, (_mode, _subset) in enumerate(RUN_PLAN):
    _subsets = SUBSET_ORDER if _mode.startswith("joint") else [_subset]
    _run_name = f"{_mode}_{_subset}" if _subset else _mode
    _out_dir = OUTPUT_ROOT / MODEL_KEY / _run_name
    # V9：运行级种子覆盖（表驱动；只影响权重初始化与采样顺序）
    _run_seed = RUN_SEED_OVERRIDE.get(_run_name, 42)
    if _run_seed != SEED:
        SEED = _run_seed
        set_seed(SEED)
        print(f"★ V9 运行级种子覆盖：{_run_seed}（权重初始化与采样顺序；数据切分保持 seed=42）")
    print("=" * 78)
    print(f"运行 {_i + 1}/{len(RUN_PLAN)}：{_run_name}  |  参与子集 {_subsets}")
    print("=" * 78)

    # ---- ② 准备数据 ----
    _data = prepare_run(_subsets, tokenizer)
    print(f"训练 {len(_data['train_ds'])} 条 | 验证 {len(_data['val_ds'])} 条 | "
          f"评测设置 {len(_data['eval_cols'])} 个")

    # ---- 本次运行的轮数（joint 用 JOINT_EPOCHS，single 用 EPOCHS）----
    _run_epochs = RUN_EPOCHS_OVERRIDE.get(
        _run_name, JOINT_EPOCHS if _mode.startswith("joint") else EPOCHS)
    print(f"本次运行轮数 = {_run_epochs}"
          f"（{'联合运行，提交用' if str(_mode).startswith('joint') else '单子集诊断'}）")

    # ---- ③ 建模型 + 显存预算检查（只负责把账算清楚并选定档位）----
    model = build_model()
    n_params = sum(p.numel() for p in model.parameters())
    check_vram_budget(n_params, MODEL_REGISTRY[MODEL_KEY]["model_id"])
    # 这里绝不做「显存不够就自动切优化器」：优化器一变，前后数值不可比；
    # 两条基线是独立的机制实验，各自锁定自己的档位与有效 batch。

    # ---- ④ 第一轮：选档 + 显存探测 + 开训前检查（做完重建干净模型）----
    if _i == 0:
        model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # 【★ 自动降档】显存不够就在下一档重试，直到某一档能真跑通；
        #   全部档位都失败时给出精确诊断（而不是一句 out of memory）。
        _tier, _bs, _peak, _sps = prepare_for_training(build_model, _data)
        apply_tier(_tier)
        BATCH_SIZE = _bs
        # 【为什么改梯度累积而不是直接改有效 batch】有效 batch 变了，
        #   梯度尺度、warmup 步数、每轮更新次数全变，本基线的结果就不再可比。
        #   分解方式可以不同，乘积必须相等。
        GRAD_ACCUM = max(1, int(round(TARGET_EFFECTIVE_BATCH / _bs)))
        print(f"→ 采用档位 {_tier['name']}（{_tier['label']}）| micro-batch={BATCH_SIZE} | "
              f"梯度累积={GRAD_ACCUM} | 有效 batch={BATCH_SIZE * GRAD_ACCUM}"
              f"（目标 {TARGET_EFFECTIVE_BATCH}） | 峰值显存约 {_peak:.2f} GiB")
        print("   （档位写死为 t3 = 全参微调，不会悄悄漂到只训部分层的档位）")
        # ---- 训练时长投影（Kaggle 单次运行 12 小时上限）----
        if _sps == _sps:          # 非 NaN，说明探测成功、量到了每样本耗时
            # V9：探针吞吐含探针自身开销，对多运行计划偏保守（V8 云端实测同一模型
            # 同一档：探针外推 1.9 h vs 实际训练 34.2 min/3 轮 = 87 ms/样本）⇒ 预算
            # 判定用「探针值与 V8 云端实测值的较小者」，两个外推都打印（透明可核）。
            _sps_cal = min(_sps, V9_CLOUD_SPS)
            _n_rows, _tot, _hours_probe = project_train_time(_sps)
            _, _, _hours_cal = project_train_time(_sps_cal)
            print(f"训练时长投影（探针口径，偏保守）：本计划共约 {_tot:,} 个样本·轮"
                  f" → 约 {_hours_probe:.1f} 小时")
            print(f"训练时长投影（V8 云端实测吞吐 {V9_CLOUD_SPS * 1000:.0f} ms/样本）"
                  f"：约 {_hours_cal:.1f} 小时（Kaggle 上限 12 小时，"
                  f"预算 {TIME_BUDGET_H} 小时；预算按此口径判定）")
            if _hours_cal > TIME_BUDGET_H:
                print("=" * 78)
                print(f"⚠️ 投影时长 {_hours_cal:.1f} 小时 > 预算 {TIME_BUDGET_H} 小时！")
                print("   V9 降级阶梯（按顺序执行，执行过哪档记进口径文件）：")
                print("   (1) V9_SWEEP 去掉 seed 44 的两档（2 seed × 4/5 轮）")
                print("   (2) V9_SWEEP 只留 4 轮档（3 seed × 4 轮）")
                print("   (3) 换更快的卡（不改任何口径项）")
                print("=" * 78)

    # ---- 正式训练带「OOM 自动重试」----
    # 探针通过不代表正式训练一定装得下（正式路径多了评估、保存等额外占用），
    # 所以正式训练要能自救：OOM 时降 micro-batch、梯度累积补回有效 batch
    #（有效 batch 是口径项不动，乘积相同即等价），重建干净模型后重试。
    _max_oom_retry = 3
    _attempt = 0
    _rand_metrics = None
    _rand_eval_loss = float("nan")
    while True:
        release_gpu_memory("正式训练前")
        model = build_model()          # 干净模型：正式训练从随机初始化开始
        trainer = make_trainer(model, _out_dir, _data, epochs=_run_epochs,
                               seed=_run_seed)

        # ---- ⑤ 训练前：随机初始化模型的基线指标（参考对照用）----
        # 【只在第一次尝试时算】重试时不必重复（模型是同一个权重、同一个种子 ⇒ 结果相同）。
        if _rand_metrics is None:
            _t0 = time.time()
            _rand_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
            print(f"随机初始化模型基线（{time.time() - _t0:.1f}s）: "
                  f"指标1 ρ={_rand_metrics.get('合计|_指标1_ρ', float('nan')):.4f} | "
                  f"指标2 ρ={_rand_metrics.get('合计|_指标2_ρ', float('nan')):.4f} | "
                  f"RMSE={_rand_metrics.get('合计|_指标3_RMSE', float('nan')):.4f}")
            # 训练前的评估 loss 单独存一份：Trainer.train() 会重建 TrainerState，
            # 训练前 evaluate() 写进 log_history 的条目会被丢弃 ⇒
            # 自己留着，绘图时补成 epoch 0 的「未训练起点」。
            _rand_eval_loss = float(_rand_metrics.get("loss", float("nan")))
            print(f"随机初始化时的评估 loss = {_rand_eval_loss:.4f}（图 F 的 epoch 0 起点）")
            release_gpu_memory("随机基线评估后")

        # ---- ⑥ 正式训练 ----
        _t0 = time.time()
        try:
            _res = trainer.train()
        except RuntimeError as _e:
            if not _is_oom(_e) or _attempt >= _max_oom_retry:
                raise
            _attempt += 1
            _new_bs = max(1, BATCH_SIZE // 2)
            if _new_bs == BATCH_SIZE:
                raise        # 已经降到 1 还是 OOM ⇒ 不是 batch 的问题 ⇒ 原样抛出（真 bug 不能被吞）
            _new_ga = max(1, int(round(TARGET_EFFECTIVE_BATCH / _new_bs)))
            print("=" * 78)
            print(f"⚠️ 正式训练 OOM（第 {_attempt}/{_max_oom_retry} 次重试）: {str(_e)[:220]}")
            print(f"   micro-batch {BATCH_SIZE} → {_new_bs}，梯度累积 {GRAD_ACCUM} → {_new_ga}"
                  f"（**有效 batch 仍为 {TARGET_EFFECTIVE_BATCH}**，口径不变）")
            print("   ⇒ 重建模型与 Trainer 后重试（权重会被重新初始化，保持可比）")
            print("=" * 78)
            BATCH_SIZE = _new_bs
            GRAD_ACCUM = _new_ga
            del trainer, model
            release_gpu_memory("OOM 重试前")
            continue
        _dt = time.time() - _t0
        print(f"训练完成，用时 {_dt / 60:.1f} 分钟，loss={_res.training_loss:.4f}")
        break

    # 立刻快照 log_history：它是累积的，后续 predict/evaluate 会继续追加；
    # 等全部跑完再取会混进重复评估点（图上出现「回头的折线」）。
    _log_hist = [dict(h) for h in trainer.state.log_history]
    _n_train_logs = sum(1 for h in _log_hist if "loss" in h and "eval_loss" not in h)
    _n_eval_logs = sum(1 for h in _log_hist if "eval_loss" in h)
    print(f"日志快照：训练点 {_n_train_logs} 个 | 评估点 {_n_eval_logs} 个")

    # 训练后回读「实际生效的口径」并断言（不信任配置常量），写进口径文件。
    _actual_bs = int(trainer.args.per_device_train_batch_size)
    _actual_ga = int(trainer.args.gradient_accumulation_steps)
    _dl_len = math.ceil(len(_data["train_ds"]) / _actual_bs)      # DataLoader 每轮的批数
    _actual_max_steps = int(trainer.state.max_steps)
    _expected_max_steps = math.ceil(_dl_len / _actual_ga) * int(trainer.args.num_train_epochs)
    _actual_warmup = int(trainer.args.warmup_steps)
    _actual_eff_batch = _actual_bs * _actual_ga
    _actual_wf = _actual_warmup / max(1, _actual_max_steps)
    assert _actual_max_steps == _expected_max_steps, (
        f"Trainer 的总更新步 {_actual_max_steps} 与按口径算出的 {_expected_max_steps} 不一致")
    assert _actual_eff_batch == BATCH_SIZE * GRAD_ACCUM, (
        f"实际有效 batch = {_actual_eff_batch} ≠ 配置 {BATCH_SIZE * GRAD_ACCUM}")
    _wf_tol = max(0.02, 2.0 / max(1, _actual_max_steps))
    assert abs(_actual_wf - WARMUP_RATIO) <= _wf_tol, (
        f"实际 warmup 占比 {_actual_wf:.3f} ≠ 设定 {WARMUP_RATIO}")
    _loss_terms = trainer.loss_weight_snapshot()   # 辅助项的实际权重（训练后取一次，允许同步）
    print(f"口径回读 ✓ 有效 batch={_actual_eff_batch} | 每轮批数={_dl_len} | "
          f"总更新步={_actual_max_steps} | warmup={_actual_warmup} 步（占 {_actual_wf:.1%}）")
    print(f"损失分项权重（自适应标定后）: {_loss_terms}")

    # ---- ⑦ 训练后评估（与随机基线完全相同的口径）----
    _pred_out = trainer.predict(_data["val_ds"])
    _trained_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
    # ---- ★ v5：采集模块内部诊断量（分布宽度 / 门值 g）----
    #   【为什么放在这里】训练结束、模型已载入 best 权重之后 ⇒ 诊断的是「交出去的那个模型」。
    _diag = collect_model_diagnostics(model, _data)
    print("诊断量采集完成："
          f"宽度 {'有' if _diag['widths'] is not None else '无'} | "
          f"门值 {'有' if _diag['gates'] is not None else '无'}")
    print("--- 训练后指标 ---")
    for k, v in _trained_metrics.items():
        if isinstance(v, float) and ("指标" in k or "目标数" in k or "综合分" in k):
            print(f"   {k:34s} {v:.4f}")

    # ---- ⑧ type-level oracle（参考对照用：每目标赋其真实均值，不看语境）----
    _val_df = _data["val_df"]
    _gold = ds_labels(_data["val_ds"])
    _type_level = np.zeros((len(_val_df), N_OUTPUTS))
    for _name, _cname, _j in _data["eval_cols"]:
        _src = "y0" if _j == 0 else "y1"
        _m = (_val_df["subset_name"] == _name).to_numpy()
        _type_level[_m, _j] = _val_df.groupby("group_key")[_src].transform("mean").to_numpy()[_m]
    _oracle_metrics = compute_compartment_metrics(
        _type_level, _gold, np.asarray(_data["val_ds"]["group_index"]),
        # 必须用全局的 SUBSET_ORDER：subset_index 是全局编号，
        # 传「本次运行的子集清单」会在单子集运行时越界。
        np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        _data["eval_cols"])

    # ---- ⑨ 保存产物并登记 ----
    # 只保留联合运行的权重（900M 模型一个 final ~3.6GB，全存会撑爆磁盘；
    # joint 覆盖全部评测设置，也是提交要用的那一个）。
    if _run_name in SAVE_FINAL_FOR:
        trainer.save_model(str(_out_dir / "final"))
        tokenizer.save_pretrained(str(_out_dir / "final"))
        print(f"模型已保存到: {_out_dir / 'final'}")
    # best 模型已 load 进内存并存进 final/，训练期 checkpoint 没有保留价值。
    for _ck in sorted(_out_dir.glob("checkpoint-*")):
        shutil.rmtree(_ck, ignore_errors=True)
    # 口径落盘：本次运行的全部口径写成一个 json，事后与其它实验逐项核对
    # ——不靠记忆，靠文件；两条基线要对照时先 diff 两个 json。
    _settings = dict(
        model_key=MODEL_KEY, model_id=MODEL_REGISTRY[MODEL_KEY]["model_id"],
        # 读出方式与机制名：两条基线的结构性差异，必须留痕
        head_mode=HEAD_MODE, mech=MECH_NAME,
        # 三个结构模块的开关与结构参数（归因依据）
        v5_modules=V5_MODULES,
        # 臂名与运行计划（多档运行产物隔离在 outputs/arms/<臂名>/ 下）
        arm_name=(globals().get("ARM_NAME", "") or ""),
        run_plan=[list(_rp) for _rp in RUN_PLAN],
        # 损失配方覆盖值（空串 = 未覆盖 = 用模块开关推导的默认配方）
        ce_override=(globals().get("_CE_OVERRIDE", "") or ""),
        swap_override=(globals().get("_SWAP_OVERRIDE", "") or ""),
        softcl_override=(globals().get("_SOFTCL_OVERRIDE", "") or ""),
        m1_dist_head=M1_DIST_HEAD, m1_ce_weight=M1_CE_WEIGHT, m1_classes=M1_CLASSES,
        m2_dual_stream=M2_DUAL_STREAM, m2_type_max_len=M2_TYPE_MAX_LEN,
        m2_gate_supervision=M2_GATE_SUPERVISION, m2_gate_weight=M2_GATE_WEIGHT,
        m4_role_emb=M4_ROLE_EMB, m4_swap_max_rows=M4_SWAP_MAX_ROWS,
        # 模块 4 的 swap 权重按子集（只对 nn 生效）⇒ 记 nn 侧取值
        m4_swap_weight=float(SUBSET_LOSS_PROFILE.get("en-nn", {}).get("swap", 0.0)),
        optim=OPTIM, gradient_checkpointing=USE_GRAD_CKPT,
        # 档位 = 优化器 + 冻结范围 + checkpointing 的打包，决定训练范围
        fit_tier=FIT_TIER_NAME, trainable_scope=FIT_SCOPE_FRAC,
        mixed_precision=("amp" if (USE_BF16 or USE_FP16) else "fp32"),
        gpu_index=GPU_INDEX,
        effective_batch=BATCH_SIZE * GRAD_ACCUM, micro_batch=BATCH_SIZE,
        grad_accum=GRAD_ACCUM, lr=LR, epochs=EPOCHS, max_length=MAX_LENGTH,
        # 本次运行实际用了几轮（与配置 EPOCHS 分开记）
        run_epochs=int(_run_epochs), joint_epochs=int(JOINT_EPOCHS),
        # best checkpoint 的选择指标（决定交出哪个权重）
        metric_for_best_model=BEST_METRIC_KEY,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        huber_delta=HUBER_DELTA,
        # 损失配置：按子集差异化的占比表
        loss_profile={k: dict(v) for k, v in SUBSET_LOSS_PROFILE.items()},
        auto_balance=bool(AUTO_BALANCE),
        order_tau=ORDER_TAU, xrank_margin=XRANK_MARGIN, xrank_tau=XRANK_TAU,
        balance_ema_beta=BALANCE_EMA_BETA,
        # 机制结构参数 + best 准则的 z 分数
        softcl_label_span=SOFTCL_LABEL_SPAN,
        softcl_proj_dim=(SOFTCL_PROJ_DIM if getattr(model, "proj", None) is not None else None),
        ce_soft_sigma=CE_SOFT_SIGMA, readout_cues=READOUT_CUES,
        z_m1=Z_M1, z_m2=Z_M2, z_m3=Z_M3,
        best_metric_formula="m1/Z_M1 + m2/Z_M2 - m3/Z_M3",
        loss_finite_check_every=LOSS_FINITE_CHECK_EVERY,
        # 结论收敛版的口径（defaults 未注入这些字段时 False/None/orig 也合法）
        v6_recipe=bool(V6_RECIPE_ENABLED),
        data_mode=DATA_MODE,
        aug_root=(str(AUG_ROOT) if AUG_ROOT is not None else None),
        aug_rows=(dict(AUG_ROWS) if AUG_ROWS else None),
        tierank_margin=TIERANK_MARGIN, tierank_tau=TIERANK_TAU,
        # 组统计矩的累加精度（按设备能力决定）
        stats_dtype=(str(trainer._gs[0].dtype).replace("torch.", "")
                     if trainer._gs and trainer._gs[0] is not None else "n/a"),
        # 实际生效的口径（从 trainer 回读，不是配置常量）
        actual_effective_batch=_actual_eff_batch,
        actual_max_steps=_actual_max_steps,
        actual_warmup_steps=_actual_warmup,
        actual_warmup_frac=round(_actual_wf, 4),
        dataloader_len=_dl_len,
        loss_term_weights={k: round(v, 4) for k, v in _loss_terms.items()},
        seed=SEED, n_outputs=N_OUTPUTS, pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
        train_subset=_subsets,
        # logging_steps 只影响日志密度，落盘便于核对曲线点数一致
        logging_steps=int(trainer.args.logging_steps),
    )
    _set_dir = OUTPUT_ROOT / "summary"
    _set_dir.mkdir(parents=True, exist_ok=True)
    with open(_set_dir / f"run_settings_{_run_name}.json", "w", encoding="utf-8") as _f:
        json.dump(_settings, _f, ensure_ascii=False, indent=2)
    ALL_RUNS.append(dict(
        run_name=_run_name, mode=_mode, subset=_subset, subsets=list(_subsets),
        eval_cols=_data["eval_cols"], out_dir=_out_dir,
        metrics=_trained_metrics, random_metrics=_rand_metrics, oracle_metrics=_oracle_metrics,
        settings=_settings,
        preds=np.asarray(_pred_out.predictions).reshape(len(_gold), -1),
        gold=_gold,
        groups=np.asarray(_data["val_ds"]["group_index"]),
        subset_ids=np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        val_df=_val_df,
        train_minutes=_dt / 60,
        log_history=_log_hist,
        random_eval_loss=_rand_eval_loss,
        diag=_diag,
    ))
    print(f"→ 运行 {_run_name} 完成，产物在 {_out_dir}")

    # 释放显存：下一轮要重新建模型，不释放容易在长跑中累积
    del trainer, model, _pred_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 78)
print(f"全部 {len(ALL_RUNS)} 次运行完成")
print("=" * 78)

## Step9 模型评估、对照图与结果分析

这一节分成**两个互不混淆的区块**：

### 区块一：评估标准（模型本身的结果）

这就是**官方评测会看到的数字**——模型在 6 个评测设置上的三项指标。
表 1 与图 A / 图 B / 图 C 全部属于这一块。

### 区块二：参考对照（不是评估标准）

两个**人造系统**，用来回答「这个分数算好吗」：

| 对照系统 | 它是什么 | 用来回答什么问题 |
|---|---|---|
| **随机初始化** | 同一个模型、同一份数据，但**没训练过** | 「模型到底学到了什么」——训练后必须显著优于它，否则等于没训 |
| **type-level oracle** | 给每个目标赋它自己的**真实均值**，完全不看语境 | 「天花板在哪」——它在指标 1 / RMSE 上能拿满分，但指标 2 必然是 `nan`（组内 std 恒为 0）。我们的模型如果指标 2 也接近 nan，说明还没真正做到「因句而异」 |

**这两个都不是提交系统的成绩**，所以它们被单独放在区块二，
不与评估标准混在同一张表、同一根柱子里。

**六张图各管一件事**：

| 图 | 内容 | 看什么 |
|---|---|---|
| 图 A | 预测 vs 真值散点（每个「子集×列」一个面板） | 点是否落在对角线上；ρ 是整体排序能力，RMSE 是数值校准 |
| 图 B | 6 个评测设置 × 三项指标 | **评估标准**本身：哪个设置强、哪个设置弱、指标 2 有没有塌成 nan |
| 图 C | 组内 std 对照散点（人工 std vs 预测 std） | 这就是指标 2 的可视化：横轴是金标准，纵轴是系统输出，ρ 即指标 2 |
| 图 D | 五次运行的合计指标横向对照 | 「联合训练」到底有没有比「单子集训练」更好 |
| 图 E | 训练后 / 随机初始化 / type-level oracle | 参照系：这个分数算好吗（**不是**评估标准） |
| 图 F | 训练损失曲线（训练集与测试集同图、不同颜色） | 训练过程健不健康：有没有收敛、有没有过拟合（泛化差距） |

**为什么图 C 最关键**：指标 2 直接决定是否出局。
如果图 C 的点挤成一条水平线（预测的组内 std 几乎不变），
说明模型没有真正对语境敏感——哪怕指标 1 很高，这份提交也会在指标 2 上拿 `nan`。

In [ ]:
# ============ 绘图工具 ============
# 【为什么统一成一个函数】每张图都手写一遍样式，早晚会出现「这张图有右边框、
#   那张图没有」的不一致。统一函数 + 数值直接标在图上（不让读者去对坐标轴）。
def style_axes(ax):
    """去掉上/右边框，网格线淡化到 alpha 0.22，坐标轴置于数据之下。"""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.22, linewidth=0.6)
    ax.set_axisbelow(True)


def annotate_box(ax, x, y, text, color="#2C3E50", ha="left", va="top"):
    """带白色不透明底衬的标注框。
    【为什么不透明】半透明白底（alpha<1）挡不住后面的线条，线会穿字而过。"""
    ax.text(x, y, text, ha=ha, va=va, fontsize=8.4, color=color,
            bbox=dict(boxstyle="round,pad=0.32", facecolor="white",
                      edgecolor=color, linewidth=0.8, alpha=1.0))


def label_bars(ax, items, ylim, fontsize=7.0):
    """把数值直接标在柱子上，并自动错开互相压叠的标签。
    items: [(x, 柱高, 文本)]；ylim: (下界, 上界) 用于计算错开的间距。"""
    lo, hi = ylim
    gap = (hi - lo) * 0.058
    placed = []
    for x, y, t in sorted(items, key=lambda z: (z[0], -z[1])):
        yy = y
        for px, py in placed:
            if abs(px - x) < 0.32 and abs(py - yy) < gap:
                yy = py + gap
        placed.append((x, yy))
        va = "bottom" if y >= 0 else "top"
        ax.text(x, yy, t, ha="center", va=va, fontsize=fontsize, color="#2C3E50",
                bbox=dict(boxstyle="round,pad=0.16", facecolor="white",
                          edgecolor="none", alpha=1.0))


BLUE, ORANGE, GREY, GREEN, RED = "#3C6E9E", "#C2703D", "#8A8F98", "#2E7D5B", "#B03A2E"

# 主图用哪一次运行？优先联合运行（它覆盖全部 6 个评测设置）
MAIN_RUN = next((r for r in ALL_RUNS if str(r["mode"]).startswith("joint")), ALL_RUNS[-1])
print("主图使用的运行:", MAIN_RUN["run_name"], "| 评测设置:", len(MAIN_RUN["eval_cols"]))

In [ ]:
# ============ 表 1：评估标准（官方六项设置 × 三项指标）============
# 【为什么用长表而不是宽表】联合运行有 6 个「子集×列」× 5 次运行 = 30 行，
#   宽表会挤成一屏看不清；长表可以直接排序、筛选、导进 Excel 做后续分析。
_rows = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        _rows.append({
            "运行": _r["run_name"],
            "子集": _n,
            "标签列": _cn,
            "指标1 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标1_ρ", float("nan")),
            "指标2 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标2_ρ", float("nan")),
            "指标3 RMSE": _r["metrics"].get(f"{_n}|{_cn}_指标3_RMSE", float("nan")),
            "组内std退化目标数": _r["metrics"].get(f"{_n}|{_cn}_组内std退化目标数", 0),
            "样例数": _r["metrics"].get(f"{_n}|{_cn}_样例数", 0),
        })
RESULTS = pd.DataFrame(_rows)
MAIN_RESULTS = RESULTS[RESULTS["运行"] == MAIN_RUN["run_name"]].reset_index(drop=True)
print("=" * 96)
print(f"表 1  评估标准 —— {MAIN_RUN['run_name']}（联合运行覆盖全部评测设置）")
print("=" * 96)
print(MAIN_RESULTS.round(4).to_string(index=False))
print(f"\n合计（各评测设置取平均）：指标1 ρ={MAIN_RESULTS['指标1 ρ'].mean():.4f} | "
      f"指标2 ρ={MAIN_RESULTS['指标2 ρ'].mean():.4f} | "
      f"RMSE={MAIN_RESULTS['指标3 RMSE'].mean():.4f}")
print(f"best-checkpoint 选择指标（{BEST_METRIC_KEY}，= 指标1/{Z_M1} + 指标2/{Z_M2} − RMSE/{Z_M3}，"
      f"即在 z 分数空间里等权）= "
      f"{MAIN_RUN['metrics'].get(BEST_METRIC_KEY, float('nan')):.4f}")
print("→ 指标 2 若是 nan，说明该设置下模型对同一目标的所有例句给出了相同分数，"
      "会在官方评测中直接出局。")

# 全部运行的汇总表落盘
_summary_dir = OUTPUT_ROOT / "summary"
_summary_dir.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(_summary_dir / "results_all_runs.csv", index=False)
print("\n已保存:", _summary_dir / "results_all_runs.csv")

# ---- 臂级口径摘要（这一档「跑的是什么」的一行说明，汇总脚本按它读取）----
try:
    with open(_summary_dir / "arm_config.json", "w", encoding="utf-8") as _f:
        json.dump({
            "baseline": MODEL_KEY,
            "arm": ARM_NAME,
            "modules": ",".join([k for k in ("m1", "m2", "m4") if V5_MODULES.get(k)]),
            "ce_weight": _CE_OVERRIDE,
            "swap_weight": _SWAP_OVERRIDE,
            "run_plan": _RUN_PLAN_ENV,
            "source": "本次消融新跑",
        }, _f, ensure_ascii=False, indent=2)
    print("已保存:", _summary_dir / "arm_config.json")
except Exception as _e:
    print("arm_config.json 写入失败（不影响训练与指标）:", _e)

# ============ 口径清单：本实验的独立口径记录 ============
# 两条基线是两个独立的机制实验，口径各自独立成立、不要求互相对齐；
# 这张表的职责：① 如实记录本实验的全部口径；
# ② 把「声明值」与「实际生效值」并排列出（清单只打印配置常量会出现静默漂移）。
#
# 【A 组】本实验的口径（声明值与实际生效值必须一致）
# 【B 组】机制项 —— 本实验要验证的东西
# 【C 组】实现细节 / 环境（允许不同，不影响数值）
_DECLARED = [
    "fit_tier", "trainable_scope", "mixed_precision", "optim",
    "effective_batch", "actual_effective_batch",
    "lr", "epochs", "run_epochs", "joint_epochs",
    "warmup_ratio", "actual_warmup_frac",
    "huber_delta", "seed", "n_outputs", "max_length", "weight_decay",
    "metric_for_best_model", "auto_balance", "order_tau", "xrank_margin",
    "xrank_tau", "balance_ema_beta", "loss_finite_check_every",
]
_MECH_ITEMS = {
    # ---- 三个结构模块的开关与结构参数（消融归因的第一依据）----
    "v5_modules": "本次运行开了哪些结构模块（m1 / m2 / m4）",
    "m1_dist_head": "模块 1：是否用序数-分布头（回归头 → 6 类 logits + 期望值读数）",
    "m1_ce_weight": "模块 1：分布头的软标签交叉熵占比",
    "m2_dual_stream": "模块 2：是否用类型-语境双流 + 敏感度门",
    "m2_gate_supervision": "模块 2：门控的监督方式（none = 自由门 / share = 用语境有效度监督）",
    "m2_type_max_len": "模块 2：类型流的最大长度（显存约束：必须短）",
    "m4_role_emb": "模块 4：是否用角色嵌入（Mod/Head = 同一 MWE 的两个视角）",
    "m4_swap_max_rows": "模块 4：每个 batch 最多给多少行构造成分交换视图",
    "m4_swap_weight": "模块 4：成分交换一致性的目标占比",
    "ce_override": "损失配方覆盖（空串 = 未覆盖 = 用模块开关推导的默认配方）",
    "swap_override": "损失配方覆盖（空串 = 未覆盖）",
    "softcl_override": "损失配方覆盖（空串 = 未覆盖）",
    "head_mode": "机制项：读出方式（regression / lm_head）",
    "mech": "机制项：本实验验证的机制名",
    "loss_profile": "机制项：损失配方按机制分叉（DeBERTa 开 softcl、Qwen 开 ce）",
    "softcl_label_span": "机制项：softcl 的标签量程",
    "ce_soft_sigma": "机制项：软目标分布带宽",
    "readout_cues": "机制项：lm_head 的读出锚点",
    "z_m1": "best 准则的 z 分数常数",
    "z_m2": "best 准则的 z 分数常数",
    "z_m3": "best 准则的 z 分数常数",
    # ---- 结论收敛版（V6）----
    "v6_recipe": "结论收敛配方是否启用（nn 关 order / 开 tierank / swap·softcl 关 / pv guard↑）",
    "data_mode": "数据口径（orig = 原始 / aug = 原始+train_data_v2 过滤版 / synth = 只合成）",
    "aug_root": "合成数据目录（DATA_MODE=aug/synth 时必须非空）",
    "aug_rows": "各子集合成行数（增广倍数的直接证据）",
    "tierank_margin": "tierank 只比较「目标均值差 > 此值」的对",
    "tierank_tau": "tierank 的 softplus 温度",
}
_FREE_ITEMS = {
    "model_key": "模型身份", "model_id": "模型身份",
    "pooling": "架构决定的池化方式（mean / last）",
    "micro_batch": "batch 分解方式（乘积 = effective_batch 即等价）",
    "grad_accum": "batch 分解方式（乘积 = effective_batch 即等价）",
    "gradient_checkpointing": "只影响速度与显存，数学上等价",
    "gpu_index": "用哪一张**同型号**卡（只影响并行调度）",
    "train_subset": "本次运行的子集（single vs joint 本就不同）",
    "actual_max_steps": "本次运行的总更新步数（随子集大小/轮数变）",
    "actual_warmup_steps": "本次运行的 warmup 绝对步数",
    "dataloader_len": "本次运行每轮的批数",
    "loss_term_weights": "自适应标定出的实际权重（随训练进程变，只作记录）",
    "stats_dtype": "组统计矩的累加精度（CUDA/CPU=float64；MPS 不支持 float64）",
    "arm_name": "臂名（产物隔离目录 outputs/arms/<臂名>/）",
    "run_plan": "本次实际跑了哪几次（run_plan 裁剪后）",
}
print("\n" + "=" * 78)
print("本实验的口径（独立记录；A 组里「声明值」与「实际值」必须一致）")
print("=" * 78)
_s = ALL_RUNS[0]["settings"]
print(f"\n【A. 本实验口径】{len(_DECLARED)} 项")
for _k in _DECLARED:
    print(f"  {_k:24s} = {_s.get(_k)}")
print("\n  声明值 vs 实际值核对（必须一致）：")
for _d, _a, _why in [
    ("effective_batch", "actual_effective_batch", "有效 batch：声明值 = 实际生效值"),
    ("warmup_ratio", "actual_warmup_frac", "warmup 占比：声明值 = 实际生效值"),
    ("fit_tier", "trainable_scope", "档位 = 可训层比例（t3 ⇒ 1.0 全参）"),
]:
    print(f"    {_d}={_s.get(_d)} | {_a}={_s.get(_a)}    ← {_why}")
print(f"\n【B. 机制项】{len(_MECH_ITEMS)} 项")
for _k, _why in _MECH_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")
print(f"\n【C. 实现细节 / 环境】{len(_FREE_ITEMS)} 项 —— 允许不同，不影响数值")
for _k, _why in _FREE_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")
print("\n口径文件:", _summary_dir / f"run_settings_{ALL_RUNS[0]['run_name']}.json")
print("判读规则：")
print("   ① 本实验独立成立——不要求与另一个实验口径一致；")
print("   ② A 组的「声明值」必须等于「实际生效值」；")
print("   ③ 与其它实验比较时，用这张表核对差异，不要假设两边一定一样。")

In [ ]:
# ============ 诊断表：模块的「内部读出」（官方指标里看不到，但决定模块是否成立）============
# 模块 1 的分布宽度是否反映歧义度（ρ 明显 > 0 ⇒ 成立）、
# 模块 2 的门 g 是否在四个子集上分化（nn 小 / pv 大 ⇒ 成立；几乎一样 ⇒ 否掉）。
# 这张表是「模块是否值得继续投入」的唯一依据，必须落盘。
# 人工歧义度必须先按子集切片、再在切片内求组内均值：joint 的 val_df 是四个子集
# 拼起来的，直接全局 groupby 会让 pv 组在 nn 的 std 列上取到 NaN、污染整列 ρ。
# 抽成独立函数以便对这条规则写回归单测。
def compute_width_rho(val_df, widths, subset_name, col_idx, std_col):
    """目标级的「预测宽度」与「人工歧义度」的 Spearman ρ。

    参数
      val_df      : 该次运行的验证集表（joint 时含全部子集的行）
      widths      : [N, n_out] 的分布宽度矩阵（行序与 val_df 一致），可为 None
      subset_name : 只算这个子集
      col_idx     : 该子集的第几个标签列
      std_col     : 该子集对应的人工 std 列名

    返回 (rho, n_targets) —— rho 为 nan 表示算不出来（样本不足 / 列不存在）。
    """
    if widths is None or std_col is None:
        return float("nan"), 0
    _mask = (val_df["subset_name"] == subset_name).to_numpy() \
        if "subset_name" in val_df.columns else np.ones(len(val_df), dtype=bool)
    if int(_mask.sum()) == 0:
        return float("nan"), 0
    _svdf = val_df.loc[_mask]
    if std_col not in _svdf.columns:
        return float("nan"), 0
    _gk = _svdf["group_key"].astype(str).to_numpy()
    _sw = np.asarray(widths)[_mask]
    if _sw.ndim != 2 or col_idx >= _sw.shape[1]:
        return float("nan"), 0
    _tgw = pd.DataFrame({"gk": _gk, "w": _sw[:, col_idx]}).groupby("gk")["w"].mean()
    _amb = _svdf.groupby("group_key")[std_col].mean()
    # 人工 std 必须是有限值：不可用的组**剔除**，而不是把 NaN 混进秩
    _amb = _amb[np.isfinite(_amb.to_numpy(dtype=float))]
    _cm = _tgw.index.intersection(_amb.index)
    if len(_cm) < 3:
        return float("nan"), int(len(_cm))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConstantInputWarning)
        _rho = float(stats.spearmanr(_tgw.loc[_cm].to_numpy(),
                                     _amb.loc[_cm].to_numpy())[0])
    return _rho, int(len(_cm))


_diag_rows = []
for _r in ALL_RUNS:
    _d = _r.get("diag") or {}
    _w = _d.get("widths")
    _g = _d.get("gates")
    _vdf = _r["val_df"].reset_index(drop=True)
    _gk = _vdf["group_key"].astype(str).to_numpy()
    for _n, _cn, _j in _r["eval_cols"]:
        _row = {"运行": _r["run_name"], "子集": _n, "标签列": _cn}
        # ---- 模块 1：目标级的「预测宽度」vs「人工歧义度」（按子集切片，见上文根因）----
        _sp = SUBSET_SPECS[_n]
        _stdcol = _sp["stds"][_j] if _j < len(_sp["stds"]) else None
        _rho, _ntgt = compute_width_rho(_vdf, _w, _n, _j, _stdcol)
        _row["宽度ρ(歧义度)"] = _rho
        _row["宽度ρ目标数"] = _ntgt
        # ---- 模块 2：平均门值 g（逐子集；同一子集两列应相同，因为 g 只看 MWE）----
        _row["平均门值g"] = float("nan")
        # ★ v5 消融新增：**门值在目标之间是否分化**、**门值在目标内部是否随语境变化**
        #   v5 实测发现 DeBERTa 的门值在四个子集上几乎是常数（nn↔pv 只差 0.0198），
        #   但「常数」有两种完全不同的形态，指标上看不出区别：
        #     (a) 门是**结构性常量**（所有目标、所有句子都同一个值）⇒ 门没学到任何东西；
        #     (b) 门**在目标之间有区分**（每个目标不同）只是 nn 与 pv 的**均值**接近
        #         ⇒ 门学到了「这个词有多敏感」，只是没学到「品类差异」。
        #   ⇒ 必须分别测：**目标级离散度**（跨目标）与**组内离散度**（同一目标跨句子）。
        _row["门值g目标级std"] = float("nan")
        _row["门值g组内std"] = float("nan")
        _sub_mask = (_vdf["subset_name"] == _n).to_numpy() if "subset_name" in _vdf.columns \
            else np.ones(len(_vdf), dtype=bool)
        if _g is not None and int(_sub_mask.sum()) > 0:
            _sg = np.asarray(_g)[_sub_mask]
            if len(_sg) > 0:
                _row["平均门值g"] = float(np.mean(_sg))
            # (a) 目标级离散度：先把 g 平均到目标，再看目标之间差多少
            _tgm = pd.DataFrame({"gk": _gk[_sub_mask], "g": _sg}).groupby("gk")["g"].mean()
            if len(_tgm) >= 2:
                _row["门值g目标级std"] = float(np.std(_tgm.to_numpy(dtype=float)))
            # (b) 组内离散度：同一个目标内部，g 随句子变化多少（门是否真的"读语境"）
            if len(_tgm) >= 1:
                _gin = pd.DataFrame({"gk": _gk[_sub_mask], "g": _sg}).groupby("gk")["g"].std()
                _row["门值g组内std"] = float(np.nanmean(_gin.to_numpy(dtype=float)))
        _diag_rows.append(_row)
DIAG_RESULTS = pd.DataFrame(_diag_rows)
if len(DIAG_RESULTS):
    print("=" * 96)
    print("★ v5 诊断表 —— 模块的内部读出（**不属于官方指标**，是「模块是否成立」的判据）")
    print("=" * 96)
    print(DIAG_RESULTS.round(4).to_string(index=False))
    _dg_dir = OUTPUT_ROOT / "summary"
    _dg_dir.mkdir(parents=True, exist_ok=True)
    DIAG_RESULTS.to_csv(_dg_dir / "v5_module_diagnostics.csv", index=False)
    print("\n已保存:", _dg_dir / "v5_module_diagnostics.csv")
    if M1_DIST_HEAD:
        _wr = DIAG_RESULTS["宽度ρ(歧义度)"].dropna()
        if len(_wr):
            print(f"→ 模块 1 诊断：宽度与人工歧义度的目标级 ρ 均值 = {_wr.mean():.4f}"
                  f"（> 0 ⇒ 分布头学到了真实不确定度；≈ 0 ⇒ 只有「分数更准」这一半收益）")
    if M2_DUAL_STREAM:
        _gr = DIAG_RESULTS.drop_duplicates(["运行", "子集"])[
            ["运行", "子集", "平均门值g", "门值g目标级std", "门值g组内std"]]
        print("→ 模块 2 诊断（期望 nn 小 / pv 大；四个子集几乎一样 ⇒ 门没学到东西 ⇒ 模块否掉）：")
        print(_gr.round(4).to_string(index=False))
        # ---- ★ 消融判读：区分「结构性常量」与「学到了但均值接近」----
        #   v5 实测 DeBERTa 四子集门值差仅 0.0198（判据要求 ≥ 0.05）。
        #   但只看均值差无法区分两种机制，必须同时看两个离散度：
        #     · 目标级 std ≈ 0  ⇒ 门对**所有目标**给同一个值 ⇒ 真·结构性常量（模块否掉）；
        #     · 目标级 std > 0 且 nn 与 pv 的均值仍接近
        #                    ⇒ 门学到了「目标敏感度」，但没学到「品类差异」
        #                       （模块**部分**成立：要改的是门控的监督/特征，不是门本身）。
        print("→ 模块 2 判读（区分结构性常量 vs 部分成立）：")
        for _rn, _sub in _gr.groupby("运行"):
            _nn = _sub[_sub["子集"].str.endswith("-nn")]["平均门值g"]
            _pv = _sub[_sub["子集"].str.endswith("-pv")]["平均门值g"]
            _gap = float(np.mean(_pv) - np.mean(_nn)) if len(_nn) and len(_pv) else float("nan")
            _tstd = float(np.nanmean(_sub["门值g目标级std"])) if len(_sub) else float("nan")
            _istd = float(np.nanmean(_sub["门值g组内std"])) if len(_sub) else float("nan")
            if not np.isfinite(_gap):
                _verdict = "判据不适用（缺子集）"
            elif _tstd < 1e-4:
                _verdict = "✗ 结构性常量（目标级 std ≈ 0）⇒ 门没学到任何东西"
            elif _gap < 0.05:
                _verdict = (f"△ 部分成立（目标级 std={_tstd:.4f} > 0 ⇒ 门对目标有区分，"
                            f"但 nn↔pv 均值差 {abs(_gap):.4f} < 0.05 ⇒ 没学到品类差异）")
            else:
                _verdict = f"✓ 成立（nn↔pv 均值差 {abs(_gap):.4f} ≥ 0.05）"
            print(f"  [{_rn}] nn→pv 门值差={_gap:+.4f} | 目标级std={_tstd:.4f} | "
                  f"组内std={_istd:.4f} ⇒ {_verdict}")

In [ ]:
# ============ 图 A：预测 vs 真值（按「子集 × 标签列」分面）============
# 【为什么放对角线】理想模型的点全部落在 y=x 上；
#   点整体在直线**上方**说明系统性高估，在下方说明系统性低估。
_panels = MAIN_RUN["eval_cols"]
_pred = MAIN_RUN["preds"]
_gold = MAIN_RUN["gold"]
_groups = MAIN_RUN["groups"]
_subsets = MAIN_RUN["subset_ids"]
_nc = min(3, len(_panels))
_nr = math.ceil(len(_panels) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.8 * _nc, 4.5 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_panels):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    ax.scatter(_gold[_m, _j], _pred[_m, _j], s=14, alpha=0.42, color=BLUE, edgecolors="none")
    _lo = max(0.0, float(min(_gold[_m, _j].min(), _pred[_m, _j].min())) - 0.3)
    _hi = float(max(_gold[_m, _j].max(), _pred[_m, _j].max())) + 0.3
    ax.plot([_lo, _hi], [_lo, _hi], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(_lo, _hi); ax.set_ylim(_lo, _hi)
    ax.set_xlabel(T("gold")); ax.set_ylabel(T("pred"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    _r = stats.spearmanr(_pred[_m, _j], _gold[_m, _j])[0]
    _e = float(np.sqrt(((_pred[_m, _j] - _gold[_m, _j]) ** 2).mean()))
    annotate_box(ax, _lo + 0.04 * (_hi - _lo), _hi - 0.04 * (_hi - _lo),
                 f"ρ = {_r:.3f}\nRMSE = {_e:.3f}")
    style_axes(ax)
for _k in range(len(_panels), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figA')} ({MAIN_RUN['run_name']})", fontsize=12.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
_figA = _summary_dir / "figA_pred_vs_gold.png"
fig.savefig(_figA, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figA)

In [ ]:
# ============ 图 B：评估标准 —— 六个评测设置 × 三项指标 ============
# 【这张图只画模型本身】参考对照（随机初始化 / oracle）不在这里出现，
#   它们被放到后面的「参考对照」区，避免读者把对照系统的数字误当成成绩。
_metric_defs = [("指标1_ρ", T("rho_up"), T("m1")),
                ("指标2_ρ", T("rho_up"), T("m2")),
                ("指标3_RMSE", T("rmse_down"), T("m3"))]
_xlabels = [f"{n}\n{c}" for n, c, _j in _panels]

fig, axes = plt.subplots(1, 3, figsize=(2.0 * len(_panels) + 3.6, 5.2))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [MAIN_RUN["metrics"].get(f"{n}|{c}_{_suffix}", float("nan")) for n, c, _j in _panels]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [BLUE if np.isfinite(v) else "#D8DBE0" for v in _vals]
    ax.bar(np.arange(len(_panels)), _heights, 0.6, color=_colors, edgecolor="white")
    # 【y 轴必须包含 0】柱状图的基准是 0；若 0 落在范围外，柱子会被裁切，
    #   读者会误以为柱子的高度是「从面板顶端算起」的。
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.22 * _span)
    _items = [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)]
    label_bars(ax, _items, ax.get_ylim(), fontsize=7.4)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):
            # 指标 2 在常数输出下是 nan：必须在图上写出来，不能画成 0
            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(_panels)))
    ax.set_xticklabels(_xlabels, fontsize=7.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figB')}: {MAIN_RUN['run_name']} on {len(_panels)} {T('n_settings')}",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figB = _summary_dir / "figB_evaluation_standard.png"
fig.savefig(_figB, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figB)

In [ ]:
# ============ 图 C：组内 std 对照（指标 2 的可视化，按「子集×列」分面）============
# 【为什么这是最关键的一张图】指标 2 的被解释变量就是「每个目标跨例句的人工 std」，
#   分子是「系统给出的组内 std」。把它画出来，才能一眼看出模型有没有「因句而异」。
#   点挤成一条水平线 = 预测的组内 std 几乎不变 = 指标 2 拿不到分。
# 【为什么分面数 = 评测设置数】指标 2 是**按「子集×列」分别算**的，
#   复合词的修饰语列与中心语列是两个独立的评测设置，必须各画一panel，
#   否则会漏掉「某个设置塌了」的情况。
_c_sub = list(_panels)
_nc = min(3, len(_c_sub))
_nr = math.ceil(len(_c_sub) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 4.3 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_c_sub):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    _gs = pd.DataFrame({"g": _groups[_m], "p": _pred[_m, _j], "y": _gold[_m, _j]}).groupby("g")
    _ps = _gs["p"].std(ddof=1)
    _ys = _gs["y"].std(ddof=1)
    _ok = _ps.notna() & _ys.notna()
    _r2 = stats.spearmanr(_ps[_ok], _ys[_ok])[0] if _ok.sum() > 2 else float("nan")
    ax.scatter(_ys[_ok], _ps[_ok], s=20, alpha=0.55, color=ORANGE, edgecolors="none")
    _mx = float(max(_ys[_ok].max(), _ps[_ok].max())) * 1.10 if _ok.sum() else 1.0
    ax.plot([0, _mx], [0, _mx], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(0, _mx); ax.set_ylim(0, _mx)
    ax.set_xlabel(T("gold_std"))
    ax.set_ylabel(T("pred_std"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    annotate_box(ax, _mx * 0.03, _mx * 0.97,
                 f"{T('m2rho')} {_r2:.3f}\n{T('n_targets')} {int(_ok.sum())}\n"
                 f"{T('const_std')} {int((_ps[_ok] == 0).sum())}")
    style_axes(ax)
for _k in range(len(_c_sub), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figC')}: {T('gold_axis')} ({MAIN_RUN['run_name']})",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figC = _summary_dir / "figC_group_std.png"
fig.savefig(_figC, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figC)

In [ ]:
# ============ 图 D：各次运行的横向对照（single 四个子集 vs 联合）============
# 【为什么要有这张图】五个运行里，四个是「单子集」、一个是「联合」。
#   这张图直接回答：「联合训练到底有没有比单子集训练更好」——
#   指标 1 更高？还是指标 2 更高？还是两头都不占？
_run_names = [r["run_name"] for r in ALL_RUNS]
_run_labels = []
for _r in ALL_RUNS:
    if str(_r["mode"]).startswith("joint"):
        _run_labels.append(T("joint_short"))
    else:
        _run_labels.append(T("single_short", short=short_of(_r["subset"])))

fig, axes = plt.subplots(1, 3, figsize=(2.1 * len(ALL_RUNS) + 3.2, 4.8))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [_r["metrics"].get(f"合计|_{_suffix}", float("nan")) for _r in ALL_RUNS]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [ORANGE if str(_r["mode"]).startswith("joint") else BLUE for _r in ALL_RUNS]
    ax.bar(np.arange(len(ALL_RUNS)), _heights, 0.62, color=_colors, edgecolor="white")
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)],
               ax.get_ylim(), fontsize=7.6)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):
            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(ALL_RUNS)))
    ax.set_xticklabels(_run_labels, fontsize=7.4)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
_handles = [plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)]
fig.legend(_handles, [T("single"), T("joint")],
           loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=9, frameon=False)
fig.suptitle(T("figD"), fontsize=12.5, fontweight="bold", y=1.06)
fig.tight_layout()
_figD = _summary_dir / "figD_run_comparison.png"
fig.savefig(_figD, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figD)

In [ ]:
# ============ 区块二：参考对照（不是评估标准）============
# 【为什么要做这一栏】一个孤零零的 ρ 数字无法判断好坏：
#   训练后 ρ=0.6 —— 是学到了东西，还是随机初始化本来就能有 0.3？
#   训练后 ρ=0.6 —— 离「完全不看语境」的 oracle 还有多远？
# 【为什么必须分开】这三个系统的数字不在同一张表、同一根柱子里，
#   避免读者把对照系统的成绩误当成提交系统的成绩。
print("=" * 96)
print("区块二  参考对照（不是评估标准，仅用于解读上面的数字）")
print("=" * 96)
_cmp_rows = []
for _r in ALL_RUNS:
    for _sys, _m in [("训练后模型", _r["metrics"]),
                     ("随机初始化", _r["random_metrics"]),
                     ("type-level oracle", _r["oracle_metrics"])]:
        _cmp_rows.append({
            "运行": _r["run_name"], "对照系统": _sys,
            "指标1 ρ": _m.get("合计|_指标1_ρ", float("nan")),
            "指标2 ρ": _m.get("合计|_指标2_ρ", float("nan")),
            "指标3 RMSE": _m.get("合计|_指标3_RMSE", float("nan")),
        })
COMPARE = pd.DataFrame(_cmp_rows)
print(COMPARE.round(4).to_string(index=False))
COMPARE.to_csv(_summary_dir / "reference_comparison.csv", index=False)

print("\n--- 三个系统分别是什么 ---")
print("随机初始化      ：同一模型、同一份数据，但一步都没训练。"
      "它是「瞎猜」的下界；训练后模型必须显著高于它，否则等于没训。")
print("type-level oracle：给每个目标赋它自己的真实均值（不看语境）。"
      "它在指标 1 / RMSE 上接近满分，但指标 2 必然是 nan"
      "（同一目标的所有例句拿到同一个分数 → 组内 std 恒为 0 → 相关系数未定义）。")
print("→ 我们的模型如果指标 2 也接近 nan，说明还没有真正做到「因句而异」。")

In [ ]:
# ============ 图 E：参考对照（训练后 / 随机初始化 / type-level oracle）============
# 【这张图属于参考对照区】标题与图例都明确标出「参考」，不与图 B 混为一谈。
_cmp_systems = [(T("trained"), "metrics", BLUE),
                (T("random_init"), "random_metrics", GREY),
                (T("oracle"), "oracle_metrics", ORANGE)]
fig, axes = plt.subplots(1, 3, figsize=(11.0, 4.6))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _items = []
    for _si, (_sys, _key, _c) in enumerate(_cmp_systems):
        _v = MAIN_RUN[_key].get(f"合计|_{_suffix}", float("nan"))
        _x = _si
        if np.isfinite(_v):
            ax.bar(_x, _v, 0.55, color=_c, edgecolor="white")
            _items.append((_x, _v, f"{_v:.3f}"))
        else:
            ax.bar(_x, 0.0, 0.55, color="#D8DBE0", edgecolor="white")
    _finite = [y for _x, y, _t in _items]
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, _items, ax.get_ylim(), fontsize=8.4)
    # oracle 在指标 2 上是 nan：写出来，不画成 0
    if _suffix == "指标2_ρ":
        ax.text(2, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                fontsize=9.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(range(len(_cmp_systems)))
    ax.set_xticklabels([s for s, _k, _c in _cmp_systems], fontsize=8.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figE')} ({T('not_standard')}): {MAIN_RUN['run_name']}",
             fontsize=12.0, fontweight="bold", y=1.03)
fig.tight_layout()
_figE = _summary_dir / "figE_reference_comparison.png"
fig.savefig(_figE, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figE)

In [ ]:
# ============ 图 F：训练损失曲线（训练集与测试集在同一幅图里、不同颜色）============
# 【★ 为什么必须有这张图】前面五张图全都在看「模型输出得好不好」，
#   只有这张图在看「训练过程本身健不健康」：
#     · 训练损失不降 → 学习率 / 优化器 / 数据有问题，再高的 ρ 也说明不了什么；
#     · 训练损失一直降、评估损失先降后升 → 过拟合，该早停或加正则；
#     · 两条一起降 → 正常收敛。
#   官方评测只看最终分数，但**过拟合只能靠这张图提前发现**。
# 【★ 为什么训练与评估必须画在同一幅图里】分开画会漏掉「泛化差距」：
#   只看训练曲线，一切看起来都好；只看评估曲线，只知道自己现在在哪、
#   不知道是不是还能更好。放在一起，两条曲线之间的**间距**就是泛化差距。
def _loss_series(hist):
    """把 log_history 拆成 (训练点, 评估点) 两条序列。
    【为什么要拆开】一条记录里要么有 'loss'（训练），要么有 'eval_loss'（评估），
       混在一起画会出现「评估点被当成训练点」的错位。"""
    tr = [(float(h.get("epoch", 0.0)), float(h["loss"]))
          for h in hist if "loss" in h and "eval_loss" not in h]
    ev = [(float(h.get("epoch", 0.0)), float(h["eval_loss"]))
          for h in hist if "eval_loss" in h]
    tr.sort()
    ev.sort()
    return tr, ev


_RUN_COLORS = [BLUE, GREEN, ORANGE, "#7A5195", "#3F7F93"]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.7))

# ---- (a) 五次运行的训练损失（看「哪些子集难学」）----
ax = axes[0]
for _ri, _r in enumerate(ALL_RUNS):
    _tr, _ev = _loss_series(_r.get("log_history", []))
    if not _tr:
        continue
    _lbl = (T("joint_short").replace("\n", " ") if str(_r["mode"]).startswith("joint")
            else short_of(_r["subset"]))
    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], marker="o", markersize=3.0,
            linewidth=1.5, color=_RUN_COLORS[_ri % len(_RUN_COLORS)], label=_lbl)
ax.set_xlabel(T("epoch"))
ax.set_ylabel(T("loss"))
ax.set_title(f"(a) {T('loss_all_runs')}", fontsize=10.5)
ax.legend(fontsize=8.2, frameon=False)
style_axes(ax)

# ---- (b) 主运行：训练损失 vs 评估损失（同图、不同颜色）----
ax = axes[1]
_tr, _ev = _loss_series(MAIN_RUN.get("log_history", []))
# 【补上「未训练起点」】log_history 里没有 epoch 0（原因见训练循环里的注释），
#   用训练前那次评估的 loss 补一个锚点，曲线才有「起点 → 收敛」的完整形状。
_rl = MAIN_RUN.get("random_eval_loss", float("nan"))
if np.isfinite(_rl):
    _ev = [(0.0, float(_rl))] + _ev
if _tr:
    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], color=BLUE, linewidth=1.6,
            marker="o", markersize=3.2, label=T("train_loss"))
if _ev:
    _ex = [p[0] for p in _ev]
    _ey = [p[1] for p in _ev]
    ax.plot(_ex, _ey, color=RED, linewidth=1.8, marker="s", markersize=6.0,
            label=T("eval_loss"))
    for _i0, (_x0, _y0) in enumerate(zip(_ex, _ey)):
        # 【标签摆位】第一个点（未训练起点）画在**上方**，其余画在**下方**：
        #   起点在 x=0，画在下方会被 y 轴切掉半个框；其余点在下方可以避开折线。
        _va = "bottom" if _i0 == 0 else "top"
        annotate_box(ax, _x0, _y0, f"{_y0:.3f}", color=RED, ha="center", va=_va)
    # epoch 0 的评估点 = **训练前**（随机初始化）的水平，用来标出「起点」
    # 【标注摆位】放在 axes 分数 (0.30, 0.52) 而不是紧贴起点：
    #   起点 x=0 处已经有数值标签框，文字再压上去会互相遮挡。
    if _ex and abs(_ex[0]) < 1e-9:
        ax.annotate(T("untrained"), xy=(_ex[0], _ey[0]), xytext=(0.30, 0.52),
                    textcoords="axes fraction", fontsize=8.2, color=RED,
                    arrowprops=dict(arrowstyle="->", color=RED, linewidth=0.9))
ax.set_xlabel(T("epoch"))
ax.set_ylabel(T("loss"))
ax.set_title(f"(b) {T('loss_main')}", fontsize=10.5)
ax.legend(fontsize=8.6, frameon=False)
style_axes(ax)

fig.suptitle(f"{T('figF')}: {MAIN_RUN['run_name']}", fontsize=12.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
_figF = _summary_dir / "figF_loss_curves.png"
fig.savefig(_figF, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figF)

## 附录：推理与提交（`*-pred.tsv` 导出）

**为什么把导出放在最后**：提交格式是本任务**最容易在最后一天翻车**的环节。
下面这份导出函数已经把所有已知的坑写成了断言——任何一条不满足都直接 `raise`，
而不是让它静默走到提交那天。

**格式规范（官方）**：

- 提交物是**一个 zip**，内含 4 个文件：`en-nn-pred.tsv` / `en-pv-pred.tsv` /
  `de-nn-pred.tsv` / `de-pv-pred.tsv`
- **无表头**，制表符分隔
- 名词复合词每行 **3** 列：`ContextID` ⇥ `修饰语预测` ⇥ `中心语预测`（**修饰语在前**）
- 小品词动词每行 **2** 列：`ContextID` ⇥ `单一预测`
- 测试集**不含** `*Avg` / `*Std` 列，载入逻辑必须容忍缺列
- 预测值必须落在 **[0, 5]**；输出**不得是常数**（否则指标 2 = nan）

In [ ]:
# ============ 导出函数 ============
def export_pred(tsv_path, preds, lang, task):
    """把预测导出成官方格式的 *-pred.tsv，并做全部自检。
    【为什么每条都写成断言而不是 print】提交脚本一旦出错，
       官方评测器**不会报格式错误**，只会静默给出离谱的分数。
       与其提交后怀疑人生，不如在这里就把红线画出来。"""
    test = pd.read_csv(tsv_path, sep="\t")        # 容忍缺列：只取需要的字段
    ids = test["ContextID"].astype(str)           # 原样透传，不重编号、不转 int
    preds = np.asarray(preds, dtype=float)
    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)

    if task == "nn":
        assert preds.shape[1] == 2, f"名词复合词需要 2 列（修饰语、中心语），收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "mod": preds[:, 0], "head": preds[:, 1]})
    else:
        assert preds.shape[1] == 1, f"小品词动词需要 1 列，收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "pred": preds[:, 0]})

    assert len(body) == len(test), "行数与测试集不一致"
    assert set(body["ContextID"]) == set(ids), "ContextID 集合与测试集不一致"
    _v = body.select_dtypes("number").to_numpy()
    assert np.isfinite(_v).all(), "预测里出现 NaN/inf"
    assert _v.min() >= 0.0 and _v.max() <= 5.0, f"预测超出 [0,5]：[{_v.min():.3f}, {_v.max():.3f}]"

    out = OUTPUT_ROOT / f"{lang}-{task}-pred.tsv"
    body.to_csv(out, sep="\t", index=False, header=False)   # 【关键】不写表头
    return out


# ============ 演示：按运行逐个子集导出，检查格式 ============
# 【为什么用合成的 ContextID】模型用的 DataFrame 里 ContextID 已被删除（防行序泄漏），
#   而真实测试集会自带 ContextID。所以这里造一份**带合成 ID 的假测试文件**，
#   只为验证导出函数的格式与断言；正式导出时把官方测试 TSV 的路径传进去即可。
_exported = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        if _cn != SUBSET_SPECS[_n]["cols"][0]:
            continue        # 每个子集只导一次（复合词的两列在同一文件里）
        _sp = SUBSET_SPECS[_n]
        _m = _r["subset_ids"] == _n
        _cols = [_r["preds"][_m, jj] for jj in range(len(_sp["cols"]))]
        _tsv = OUTPUT_ROOT / f"_demo_test_{_n}.tsv"
        pd.DataFrame({"ContextID": [f"{i:05d}" for i in range(int(_m.sum()))],
                      "Context": _r["val_df"].loc[_m, "Context"].to_numpy()
                      }).to_csv(_tsv, sep="\t", index=False)
        _out = export_pred(_tsv, np.stack(_cols, axis=1), _sp["lang"], _sp["task"])
        _exported.append((_r["run_name"], _n, _out))
        print(f"[{_r['run_name']} / {_n}] -> {_out.name}")
print("\n→ 无表头、列数正确、值域 [0,5]、ContextID 原样透传。")
print("→ 正式提交时把 4 个 *-train.tsv 换成官方测试 TSV，文件名保持 en-nn-pred.tsv 等 4 个。")

## Step10 扫描分析（逐运行汇总 / 逐轮轨迹 / 跨 seed 稳定性 / 多 seed 集成）

**输入**：`ALL_RUNS` 里全部联合运行（网格 = seed × 轮数，全部使用同一数据切分 ⇒
验证集逐行对齐）。**判读阈值**：合计口径 `analysis/metric_noise.json`
（指标1 **0.0652** / 指标2 **0.1526** / RMSE **0.0463**）；
单设置口径 `analysis/metric_noise_by_setting.json`；两类不可混用。
**历史参照**：V8 云端 joint 3 轮 seed42 = 0.5783 / 0.3867 / 0.7711（人类数据、
md5 通过、同口径可比）。

In [ ]:
# ============ Step10.1 逐运行汇总 + 逐轮轨迹 + 类型分诊断 ============
V9_MDE = {"m1": 0.0652, "m2": 0.1526, "rmse": 0.0463}
V9_HIST = dict(m1=0.5783, m2=0.3867, rmse=0.7711)   # V8 云端 joint 3轮 seed42（人类数据，md5 通过）
V9_JOINTS = [r for r in ALL_RUNS if str(r["mode"]).startswith("joint")]
assert V9_JOINTS, "ALL_RUNS 里没有联合运行（mode 以 joint 开头），Step10 无从分析"


def _v9_total_metrics(run):
    """一个运行的合计三指标 + 组内 std 退化目标数（R4 红线在产物层可查）。"""
    m = run["metrics"]
    return dict(m1=float(m["合计|_指标1_ρ"]), m2=float(m["合计|_指标2_ρ"]),
                rmse=float(m["合计|_指标3_RMSE"]),
                ndeg=int(m["合计|_组内std退化目标数"]))


print("=" * 96)
print("表 1  逐运行汇总（全验证口径，best-checkpoint 权重；参照 V8 = 0.5783/0.3867/0.7711）")
print("=" * 96)
print(f"{'运行':16s} {'seed':>4s} {'轮数':>4s} {'指标1ρ':>8s} {'Δm1(vs V8)':>11s} "
      f"{'指标2ρ':>8s} {'RMSE':>8s} {'退化':>4s} {'训练min':>8s}")
V9_ROWS = []
for r in V9_JOINTS:
    t = _v9_total_metrics(r)
    seed = int(r["settings"]["seed"])
    eps = int(r["settings"]["run_epochs"])
    d1 = t["m1"] - V9_HIST["m1"]
    assert t["ndeg"] == 0, (f"[{r['run_name']}] 组内 std 退化目标数 {t['ndeg']} ≠ 0："
                            "指标2 红线被破坏，停止分析")
    print(f"{r['run_name']:16s} {seed:4d} {eps:4d} {t['m1']:8.4f} {d1:+11.4f} "
          f"{t['m2']:8.4f} {t['rmse']:8.4f} {t['ndeg']:4d} {r['train_minutes']:8.1f}")
    V9_ROWS.append(dict(run=r["run_name"], seed=seed, epochs=eps, **t,
                        train_min=round(float(r["train_minutes"]), 1)))
V9_SUM = pd.DataFrame(V9_ROWS)


def _v9_traj(run, key="指标1"):
    """从 log_history 取逐轮合计指标轨迹。

    【键名必须带 eval_ 前缀】log_history 是 trainer.state 的原始快照，
    Trainer.evaluate() 写入的键全部带 eval_ 前缀（AGENTS.md 记过的坑）；
    run["metrics"] 才是 strip_eval_prefix 之后的无前缀版本。
    首点（epoch≈0）是训练前的随机初始化评估 ⇒ 跳过；训练结束后 best-checkpoint
    的重评与最后一轮同 epoch ⇒ 去重保留**原序后者**（log_history 按时间单调，
    后者 = best 口径的重评；不能 sorted——那会按数值排序、保留较大值而不是较晚者）。"""
    k = f"eval_合计|_{key}_ρ"
    pts = [(float(h["epoch"]), float(h[k]))
           for h in run["log_history"] if k in h and h.get("epoch") is not None]
    out = [p for p in pts if p[0] > 1e-6]
    dedup = []
    for ep, v in out:
        if dedup and abs(ep - dedup[-1][0]) < 1e-6:
            dedup[-1] = (ep, v)
        else:
            dedup.append((ep, v))
    return dedup


print("\n" + "=" * 96)
print("表 2  逐轮合计指标1 轨迹（回答「欠训练是否持续」；best 准则 = z 分数综合分）")
print("=" * 96)
V9_TRAJ = {}
for r in V9_JOINTS:
    tr = _v9_traj(r)
    V9_TRAJ[r["run_name"]] = tr
    line = " ".join(f"e{int(ep)}:{v:.4f}" for ep, v in tr)
    still_up = bool(len(tr) >= 2 and tr[-1][1] > tr[-2][1])
    print(f"{r['run_name']:16s} {line}   末轮仍涨: {'是' if still_up else '否'}")

# 类型分诊断（解释变量：语境聚合的目标级质量；与 V8 报告同口径 = 0.46–0.79）
print("\n类型分诊断（语境聚合 vs 金标类型均值的目标级 Spearman，按 6 设置顺序）：")
for r in V9_JOINTS:
    rhos = []
    for n, cname, j in r["eval_cols"]:
        m = (r["subset_ids"] == n)
        keys = r["val_df"].loc[m, "group_key"].to_numpy()
        p = np.asarray(r["preds"])[m, j]
        g = r["gold"][m, j]
        ctx = pd.Series(p).groupby(keys).mean()
        gt = pd.Series(g).groupby(keys).mean()
        rhos.append(float(stats.spearmanr(ctx.reindex(gt.index), gt)[0]))
    print(f"{r['run_name']:16s} " + " ".join(f"{v:+.3f}" for v in rhos))
print("（读法：与逐例句 m1 的差距 = 剩余可聚合空间；V8 判读已确认强主干上此空间很小）")

In [ ]:
# ============ Step10.2 跨 seed 稳定性 + 轮数效应 ============
print("=" * 96)
print("表 3  跨 seed 稳定性（同轮数档内；V8 单种子 3 轮 = 0.5783/0.3867/0.7711）")
print("=" * 96)
V9_STAB = []
for eps, grp in V9_SUM.groupby("epochs"):
    m1s = grp["m1"].to_numpy()
    m2s = grp["m2"].to_numpy()
    rms = grp["rmse"].to_numpy()
    spread = float(m1s.max() - m1s.min())
    verdict = ("seed 间稳定（极差 ≤ 0.0652）" if spread <= V9_MDE["m1"]
               else "seed 间差异超合计阈值（V8 新高部分是种子运气）")
    print(f"{int(eps)} 轮档：m1 = {' / '.join(f'{v:.4f}' for v in m1s)}"
          f"（均值 {m1s.mean():.4f} ± {m1s.std(ddof=1):.4f}，极差 {spread:.4f} ⇒ {verdict}）")
    print(f"{'':9s}m2 均值 {m2s.mean():.4f} ± {m2s.std(ddof=1):.4f} | "
          f"RMSE 均值 {rms.mean():.4f} ± {rms.std(ddof=1):.4f}")
    V9_STAB.append(dict(epochs=int(eps), n_seeds=int(len(grp)),
                        m1_mean=float(m1s.mean()), m1_sd=float(m1s.std(ddof=1)),
                        m1_spread=spread, m2_mean=float(m2s.mean()),
                        rmse_mean=float(rms.mean())))
V9_STAB = pd.DataFrame(V9_STAB)

if len(V9_STAB) == 2:
    d = float(V9_STAB.iloc[1]["m1_mean"] - V9_STAB.iloc[0]["m1_mean"])
    print(f"\n轮数效应（4→5 轮，跨 seed 均值差）：Δm1 = {d:+.4f}"
          f"（阈值 {V9_MDE['m1']:.4f} ⇒ {'显著' if abs(d) > V9_MDE['m1'] else '噪声内'}）")
    print("旁证：表 2 各 5 轮运行的「末轮仍涨」计数（全否 ⇒ 已到平台）")

print("\n相对 V8 云端（joint 3 轮 seed42）的逐运行 Δm1 判读（R1/R2 的读数）：")
for _, row in V9_SUM.iterrows():
    d = row["m1"] - V9_HIST["m1"]
    print(f"  {row['run']:16s} Δm1 {d:+.4f}"
          f"（{'超合计阈值' if abs(d) > V9_MDE['m1'] else '噪声内'}）")

In [ ]:
# ============ Step10.3 多 seed 预测均值集成 ============
print("=" * 96)
print("表 4  多 seed 集成（逐例句分数均值；同一验证集、6 个评测设置分别算后取官方平均）")
print("=" * 96)
V9_ALIGN_OK = all(
    (r["val_df"]["group_key"].to_numpy()
     == V9_JOINTS[0]["val_df"]["group_key"].to_numpy()).all()
    and np.allclose(np.asarray(r["gold"]), np.asarray(V9_JOINTS[0]["gold"]))
    for r in V9_JOINTS)
print(f"验证集逐行对齐检验：{'通过（数据切分种子恒为 42）' if V9_ALIGN_OK else '失败（集成禁用）'}")


def _v9_eval_preds(preds_by_run, runs):
    """对一组运行的预测求逐例句均值，按 6 设置分别算三指标再官方 nan 安全平均。"""
    per = []
    for n, cname, j in runs[0]["eval_cols"]:
        m = (runs[0]["subset_ids"] == n)
        q = np.mean([np.asarray(pr)[m, j] for pr in preds_by_run], axis=0)
        g = runs[0]["gold"][m, j]
        grp = np.asarray(runs[0]["groups"])[m]
        m1, m2, rmse, ndeg = _three_metrics(q, g, grp)
        per.append((m1, m2, rmse, ndeg))
    a = np.asarray(per, dtype=float)
    return dict(m1=float(np.nanmean(a[:, 0])), m2=float(np.nanmean(a[:, 1])),
                rmse=float(np.nanmean(a[:, 2])), ndeg=int(a[:, 3].sum()))


V9_ENS = {}
if V9_ALIGN_OK:
    _groups = {}
    for r in V9_JOINTS:
        _groups.setdefault(int(r["settings"]["run_epochs"]), []).append(r)
    for eps, rs in sorted(_groups.items()):
        V9_ENS[f"ens_e{eps}"] = _v9_eval_preds([r["preds"] for r in rs], rs)
    V9_ENS["ens_all"] = _v9_eval_preds([r["preds"] for r in V9_JOINTS], V9_JOINTS)
    _best = V9_SUM.loc[V9_SUM["m1"].idxmax()]
    print(f"最佳单运行：{_best['run']}  "
          f"{_best['m1']:.4f}/{_best['m2']:.4f}/{_best['rmse']:.4f}")
    for name, t in V9_ENS.items():
        d = t["m1"] - float(_best["m1"])
        print(f"{name:10s} 指标1 {t['m1']:.4f}（vs 最佳单运行 {d:+.4f}，"
              f"阈值 {V9_MDE['m1']:.4f} ⇒ {'显著' if d > V9_MDE['m1'] else '噪声内'}）"
              f" | 指标2 {t['m2']:.4f} | RMSE {t['rmse']:.4f} | 退化目标数 {t['ndeg']}")
    print("注：① 集成只用「同一验证集上的逐例句分数均值」，不涉及任何标签调参；")
    print("    ② 集成会改变组内 std ⇒ 指标2 没有校准那样的不变性保证，")
    print("       退化目标数必须为 0（R4 红线，上面逐位打印）；")
    print("    ③ 提交用法：对测试集逐例句做同样的多种子均值（测试侧零额外假设）。")

In [ ]:
# ============ Step10.4 图 J/K/L + 落盘 ============
_out_v9 = OUTPUT_ROOT / "summary"
_out_v9.mkdir(parents=True, exist_ok=True)

# 图 J：逐运行合计指标1（颜色按 seed；虚线 = V8 云端参照 + 阈值带）
_figJ, _axJ = plt.subplots(figsize=(11.5, 4.6))
_seed_colors = {42: "#2980B9", 43: "#C0392B", 44: "#27AE60"}
_xj = np.arange(len(V9_SUM))
_axJ.bar(_xj, V9_SUM["m1"], color=[_seed_colors[s] for s in V9_SUM["seed"]])
for _x, _v in zip(_xj, V9_SUM["m1"]):
    _axJ.annotate(f"{_v:.4f}", (_x, _v), textcoords="offset points",
                  xytext=(0, 3), ha="center", fontsize=7)
_axJ.axhline(V9_HIST["m1"], color="#7F8C8D", lw=1.1, ls="--",
             label=f"V8 cloud (3ep, s42) = {V9_HIST['m1']:.4f}")
_axJ.axhspan(V9_HIST["m1"] - V9_MDE["m1"], V9_HIST["m1"] + V9_MDE["m1"],
             color="#7F8C8D", alpha=0.12, label=f"±MDE {V9_MDE['m1']:.4f}")
_axJ.set_xticks(_xj, [f"{r}\n(s{int(s)},e{int(e)})"
                      for r, s, e in zip(V9_SUM["run"], V9_SUM["seed"], V9_SUM["epochs"])],
                fontsize=7)
_axJ.set_ylabel("Metric1 rho (total)")
_axJ.set_title("Fig J  Sweep: Metric1 per run (color = seed)", fontsize=10)
_axJ.legend(fontsize=8)
_figJ.tight_layout()
_figJ.savefig(_out_v9 / "figJ_v9_sweep_runs.png", dpi=150)
plt.show()

# 图 K：逐轮轨迹（每运行一条；末端箭头看是否仍涨）
_figK, _axK = plt.subplots(figsize=(10.5, 4.6))
for _run, _tr in V9_TRAJ.items():
    if not _tr:
        continue
    _eps = [e for e, _ in _tr]
    _vs = [v for _, v in _tr]
    _axK.plot(_eps, _vs, marker="o", ms=4, lw=1.4, label=_run)
_axK.axhline(V9_HIST["m1"], color="#7F8C8D", lw=1.1, ls="--", label="V8 cloud (3ep)")
_axK.set_xlabel("epoch")
_axK.set_ylabel("Metric1 rho (total)")
_axK.set_title("Fig K  Epoch trajectories (best ckpt re-eval deduped)", fontsize=10)
_axK.legend(fontsize=7, ncol=3)
_figK.tight_layout()
_figK.savefig(_out_v9 / "figK_v9_trajectories.png", dpi=150)
plt.show()

# 图 L：集成 vs 单运行（指标1 / 指标2 / RMSE 三面板）
if V9_ENS:
    _names = list(V9_SUM["run"]) + list(V9_ENS.keys())
    _vals = {"m1": list(V9_SUM["m1"]) + [V9_ENS[k]["m1"] for k in V9_ENS],
             "m2": list(V9_SUM["m2"]) + [V9_ENS[k]["m2"] for k in V9_ENS],
             "rmse": list(V9_SUM["rmse"]) + [V9_ENS[k]["rmse"] for k in V9_ENS]}
    _figL, _axesL = plt.subplots(1, 3, figsize=(13.5, 4.2))
    for _ax, _mk, _lab in zip(_axesL, ("m1", "m2", "rmse"),
                              ("Metric1 rho", "Metric2 rho", "RMSE")):
        _cols = (["#BDC3C7"] * len(V9_SUM)) + (["#8E44AD"] * len(V9_ENS))
        _ax.bar(np.arange(len(_names)), _vals[_mk], color=_cols)
        _ax.set_xticks(np.arange(len(_names)), _names, rotation=60, fontsize=6)
        _ax.set_title(_lab, fontsize=10)
        _ax.axhline(0, color="black", lw=0.8)
    _figL.suptitle("Fig L  Ensembles (purple) vs single runs (gray)", fontsize=11)
    _figL.tight_layout(rect=[0, 0, 1, 0.93])
    _figL.savefig(_out_v9 / "figL_v9_ensembles.png", dpi=150)
    plt.show()

# ---- 落盘（口径 + 全部读数，事后可 diff 可复算）----
_v9_dump = dict(
    hist_ref_v8_joint=V9_HIST, thresholds=V9_MDE,
    grid=[list(x) for x in V9_SWEEP], cloud_sps=V9_CLOUD_SPS,
    per_run=V9_SUM.to_dict(orient="records"),
    stability=V9_STAB.to_dict(orient="records"),
    trajectories={k: [list(p) for p in v] for k, v in V9_TRAJ.items()},
    ensembles={k: dict(v) for k, v in V9_ENS.items()},
    run_settings={r["run_name"]: dict(seed=int(r["settings"]["seed"]),
                                      run_epochs=int(r["settings"]["run_epochs"]),
                                      effective_batch=int(r["settings"]["effective_batch"]))
                  for r in V9_JOINTS},
)
with open(_out_v9 / "v9_sweep.json", "w", encoding="utf-8") as _f:
    json.dump(_v9_dump, _f, ensure_ascii=False, indent=1, default=str)
V9_SUM.to_csv(_out_v9 / "v9_sweep_runs.csv", index=False)
V9_STAB.to_csv(_out_v9 / "v9_sweep_stability.csv", index=False)
print("已写出 v9_sweep.json / v9_sweep_runs.csv / v9_sweep_stability.csv")
print("判读顺序：R4 红线（退化目标数全 0、指标2 全非 nan）→ R1 稳定性 → R2 轮数 → R3 集成")

## 结论与延伸

### 这份实验的运行前置条件

> 本文件是 **XLM-RoBERTa-large（V9 多 seed × 轮数扫描）** 的专属**扫描实验**，由共享真源
> （`tools/nbgen/cells_a/b/c.py`，**只读**）经 `v9/nbgen/build_v9.py` 的 overlay
> 补丁生成：数据处理、切分、指标、绘图、显存档位与 v4–v8 逐字同源；
> V9 的差异只有「主干注册 xlmr + 扫描网格（种子/轮数表驱动）+ Step10 扫描分析」，
> 全部可逐条审计（每条补丁都断言原串恰好出现一次）。
>
> 三个模型结构模块（1 / 2 / 4）各有独立开关（环境变量 `COMPARTMENT_V5_MODULES`，
> 默认 `m1,m2,m4` 全开），用于归因「收益来自哪个模块」：
>   · 逐设置差值 → `analysis/metric_noise_by_setting.json`（单设置阈值）；
>   · 合计差值   → `analysis/metric_noise.json`（合计阈值）；
>   · **两者不可混用**。
> 三个模块另外各有自带诊断（Step9 的「诊断表」）：
>   模块 1 看「分布宽度 vs 人工歧义度」的相关；模块 2 看「门 g 在四个子集上是否分化」；
>   模块 4 看「两列是否同时改善（只改善一列 = 容量问题，不是结构问题）」。
>
> **本文件验证的机制**：**多 seed × 轮数扫描 + 多 seed 预测均值集成（V8 主干的稳定性与欠训练跟进）**。

| 项 | 要求 | 说明 |
|---|---|---|
| Python | ≥ 3.10 | 已验证 3.12 |
| torch | ≥ 2.4 | 已验证 2.14 |
| transformers | ≥ 5.0 | **5.x 的 API 与 4.x 不同**：`eval_strategy`、`processing_class`、`warmup_steps`、`dtype=` 都是 5.x 写法；代码已做双兼容（签名探测自动切换） |
| datasets | ≥ 3.0 | 已验证 3.6 |
| 网络 | 云端 GPU 可直连 `huggingface.co`；本机通常不通 | Step1 会**自动探测**官方源，不通就换 `hf-mirror.com`，下载时再兜一层重试。也可设 `COMPARTMENT_HF_MODE=official\|mirror\|cache` 强制指定 |
| **显存** | **单卡 ≥ 12GiB 可用**（T4 14.56GiB 即可） | 训练「档位」默认写死 t3（全参微调 · Adafactor · 关 AMP），共 7 档且固定开销单调递减。Step8 会用真实模型真跑 2 步验证，OOM 自动降一档 |
| 内存 | ≥ 16GB | 主干 fp32 权重约 3.6GB / 6.2GB |
| 磁盘 | ≥ 10GB 可用 | 5 次运行只保留**联合运行**的权重（~3.6GB），训练用 checkpoint 训完即删 |
| 时间 | 5 次运行（4 个单子集 + 1 个联合） | 轮数 `EPOCHS=JOINT_EPOCHS`（注入档可覆盖）；总样本·轮 = 0.8×总行数×(EPOCHS+JOINT_EPOCHS)。Step8 开跑前会打印**时长投影**，超预算会报警 |

### 显存够不够：先看这张表

全参微调 + AdamW 的**固定开销**（权重 fp32 4 B/param + 梯度 fp32 4 B/param
+ AdamW 的 m/v 8 B/param = **16 B/param**，再加上开混合精度时 AMP 的 fp16 权重副本 2 B/param
→ **18 B/param**）**与 batch 大小无关**，所以「降 batch / 加梯度累积」救不了它。
真正能降下来的只有三样：**优化器状态**、**AMP 的 fp16 副本**、**可训练参数的范围**。

| 模型 | 参数量 | fp32 AdamW+AMP | 8bit+AMP | Adafactor+AMP | Adafactor 无 AMP（t3） |
|---|---|---|---|---|---|
| `microsoft/deberta-v2-xlarge` | 885M | 14.8 GiB | 9.9 GiB | 8.4 GiB | 6.8 GiB ✓ |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1544M | **25.9 GiB** | 17.3 GiB | 14.7 GiB ✗ | **11.8 GiB ✓** |

（表中为「固定开销」，不含激活；激活 + 显存碎片 + CUDA context 另计约 3 GiB。）

**T4 只有 14.56 GiB**，所以在 T4 上能保住「全参微调」语义的档位是
**t3（Adafactor + 关 AMP，8.2 B/param ≈ 11.8 GiB）** —— 没有任何参数被冻结。

### 为什么「9B 都能全量微调，1.5B 却超了」

这两句话说的**不是同一件事**，差别在三个地方：

| | 9B 全参微调（业界常见） | 本项目 1.5B 全参微调 |
|---|---|---|
| 硬件 | 2 × A100 **80GB**（合计 160GB） | 2 × T4 **14.56GB**（合计 29.1GB） |
| 固定开销 | 9B × 18 B = **162GB**，靠 ZeRO-3 分片塞进 160GB（还要再省） | 1.5B × 18 B = **25.9GB**，单卡 14.56GB 装不下 |
| 是否分片 | 用 ZeRO-3 / FSDP 把参数切到多卡 | notebook 是**单进程**，每卡都要放完整一份 |

关键在于：**DDP（数据并行）不省显存** —— 每张卡仍要放一整份模型+梯度+优化器状态，
只是把数据分半（提速用）。只有 **ZeRO-3 / FSDP 参数分片**才会把 25.9GB 摊到两张卡
（13GB/卡），而那需要 `accelerate launch --num_processes 2`，
**在 papermill 的单进程里跑不了**；而且本 notebook 自定义了分组采样器
（让同一目标的例句落在同一个 batch），DDP 下若不改成分布式采样器，
两个进程会取到**同一批样本** → 训练结果悄悄错掉。

所以本 notebook 的选择是：**单卡 + 把固定开销从 18 B/param 降到 8.2 B/param
（Adafactor + 关 AMP）**，这样 1.5B 也能在 T4 上做**完整的全参微调**。

### 训练档位（tier）：默认写死 t3（全参微调）

> 档位写死 `FIT_MODE = "t3"`：降档会改变训练范围（t4/t5 只训顶部若干层），
> 而机制实验要求主干整体可学 ⇒ 写死后训练范围可复现、不会悄悄漂移；
> 真跑不通就直接报错并给出诊断。两条基线是两个独立的机制实验，
> 各自锁定 t3，口径记录在「口径清单」里。
> 下面的档位表仍保留：它是 `FIT_MODE="auto"` 时的选档逻辑，也是降档链的依据。

**显存模型**：

```
每参数开销 = 权重 fp32 4
           + （开 AMP 时）fp16 权重副本 2   ← 反向传播要用它算 grad_input，
                                               被 autograd 存到 backward 结束
           + 梯度 fp32 4 × 可训比例
           + 优化器状态 × 可训比例   AdamW 8 / 8bit AdamW 2 / Adafactor ≈0.2
```

（AMP 的 fp16 副本 2 B/param 不能漏算：漏算会低估约 3 GiB，把「装不下」误判成「装得下」。）

| 档位 | 含义 | optim | AMP | 可训层 | 固定开销 | 什么时候选它 |
|---|---|---|---|---|---|---|
| t0 | 全参微调 · fp32 AdamW | `adamw_torch` | 开 | 100% | 18 B/param | 40GB+ 的卡（A100 等） |
| t1 | 全参微调 · 8bit AdamW | `adamw_bnb_8bit` | 开 | 100% | 12 B/param | 装了 bitsandbytes 且 ≥24GB 卡 |
| t2 | 全参微调 · Adafactor | `adafactor` | 开 | 100% | 10.2 B/param | 16GB+ 的卡（L4 24GB 会选 t1） |
| **t3** | **全参微调 · Adafactor · 关 AMP** | `adafactor` | 关 | 100% | **8.2 B/param** | **T4 14.56GB 就是这一档**（仍全参微调，只是不用混合精度） |
| t4 | 顶部 50% 层 · Adafactor | `adafactor` | 开 | 50% | 8.1 B/param | 想要 AMP 速度、可以接受冻一半层时 |
| t5 | 顶部 50% 层 · Adafactor | `adafactor` | 关 | 50% | 6.1 B/param | 更省显存 |
| t6 | 顶部 25% 层 · Adafactor | `adafactor` | 关 | 25% | 5.05 B/param | 兜底 |

> ⚠️ **上表的前提是「单卡模式」**。若进程能看见 2 张卡，transformers 的 Trainer 会自动套
> `nn.DataParallel`，其反向传播要**再复制一整份梯度**（Qwen-1.5B 多 5.75 GiB）→ 必然 OOM。
> Step1 用 `CUDA_VISIBLE_DEVICES` 把进程限制成单卡，`build_training_args` 里再钉 `args._n_gpu = 1` 兜底。

**选档规则（仅 `FIT_MODE="auto"` 时启用）**：用**两条基线中较大者的参数量**
（`FIT_REF_PARAMS = 1.55e9`）作参考，配合本机显存选第一个装得下的档位
（判据用「micro-batch=1 的最小激活」，因为激活是 batch 的事、由探测阶段逐档降）。
若估算仍偏乐观，Step8 会用**真实模型真跑 2 步**验证，OOM 就自动降一档重来；
全部档位都失败时给出精确诊断（而不是一句 out of memory）。

> Qwen 的有效 batch 是 8（micro 8 × accum 1）：梯度累积 ≥2 会让绑定 embedding 的
> 稠密梯度在反向时同时存在两份（+890 MiB）⇒ t3 装不下；
> accum=1 时 autograd 直接复用那块临时缓冲（全程只 1 份）⇒ 装得下。

> t3 档为什么「关 AMP」：开 AMP 时 PyTorch 会把权重的 fp16 副本留到反向传播用，
> 多占 2 B/param。Qwen-1.5B 正是差这 2 B/param（3.1 GiB）而装不下 14.56 GiB 的卡。
> 关掉 AMP 就省出这 3.1 GiB，代价是纯 fp32 训练慢一些（T4 没有 TF32）。

### 「两张 T4 能不能凑成 30G 用？」——不能

**结论先说**：CUDA 显存**不能跨卡合并**。一个张量只能待在一张卡上，
「2 × 14.56 GiB」不会变成「一张 30 GiB 的卡」。

**更糟的是**：只要进程能看见 2 张卡，transformers 的 Trainer 就会**自动**把模型套上
`nn.DataParallel`：

```python
# transformers/trainer.py :: _wrap_model()
if self.args.n_gpu > 1 and not is_loaded_in_8bit and not is_loaded_in_4bit:
    model = nn.DataParallel(model)
```

`nn.DataParallel` 的反向传播要经过 `Broadcast` → `ReduceAddCoalesced`，
**额外分配并复制一整份梯度** → 梯度显存翻倍。Qwen-1.5B 多 5.75 GiB → 14.56 GiB 的卡必然 OOM。

所以本项目的做法是：**用 `CUDA_VISIBLE_DEVICES` 把进程限制成只看见 1 张卡**
（Step1 已设，`build_training_args` 里再钉一次 `args._n_gpu = 1` 兜底），
让 Trainer 走单卡路径，梯度显存不翻倍。

| 想做的事 | 能不能把两张卡的显存合起来用 | 说明 |
|---|---|---|
| **DDP（数据并行）** | **不能** | 每张卡各存**一整份**模型+梯度+优化器状态，只是把 batch 分半 → 提速，不省显存 |
| **nn.DataParallel（Trainer 自动套的）** | **不能，而且更糟** | 反向传播还要**再复制一份梯度** → 显存 ×2 |
| **ZeRO-3 / FSDP（参数分片）** | **能** | 每卡只放 1/2 → 这才是「29 GiB 一起用」 |
| **两条基线各占一张卡** | 不需要合并 | 两个独立实验各跑各的，墙钟时间直接减半 |

**为什么本 notebook 不直接上 ZeRO-3/FSDP**（三个原因）：

1. 需要**多进程**启动：`accelerate launch --num_processes 2` 或 `torchrun`。
   papermill / `jupyter nbconvert` 起的是**普通单进程**，不会自动开 DDP。
2. 本 notebook 自定义了 `TargetGroupedSampler`（让同一目标的例句落在同一个 batch，
   否则组内损失形同虚设）。DDP 下若不改成分布式采样器，
   **两个进程会取到同一批样本** → 训练结果悄悄错掉，而且**不报错**。
3. 改造要动的地方多（采样器、Trainer 子类、跨进程结果汇总），属于大改，
   而当前阻塞点（Qwen 装不下）用单卡 + 档位机制已经能解。

**最省事的方案**：

```bash
COMPARTMENT_GPU=0   # DeBERTa 版（单卡，梯度显存不翻倍）
COMPARTMENT_GPU=1   # Qwen 版（同一台机器、两条基线各占一张卡，各自独立运行）
```

两条基线各占一张卡同时跑，**两张卡的算力都用满**，墙钟时间减半（同型号卡对数值无影响）；
「29 GiB 合成一张卡」等以后真的要训 7B/9B 时再用 FSDP 脚本形态来做。

### 多卡怎么用：先分清「提速」和「省显存」

| 用法 | 每卡上放什么 | 能降低单卡显存需求吗 | 能提速吗 |
|---|---|---|---|
| 单进程 | 全部在一张卡 | — | 基准 |
| **DDP（数据并行）** | **每卡各存一整份**模型+梯度+优化器状态，只是数据分半 | **不能**（每卡仍是完整那一份） | 能（约 1.8×） |
| **ZeRO-3 / FSDP（参数分片）** | 参数/梯度/优化器状态**切分**到多卡 | **能**（这才是用满多卡显存的办法） | 能 |
| 两条基线各占一张卡 | 两条基线是**互不相干的实验**，各占一卡同时跑 | 不需要（各跑各的） | 能（墙钟减半） |

**对 Qwen2.5-1.5B 具体意味着什么**：DDP **救不了**它 —— 每卡仍要装 23.0 GiB 的固定开销。
本 notebook 的做法是走 **t3 档**（全参微调 + Adafactor + 关 AMP = 8.2 B/param ≈ 11.8 GiB，单卡就够）；
真想用满 29 GiB 训更大的模型，则要上 ZeRO-3/FSDP 脚本形态（见上一节的三条理由）。

**注意**：papermill / `jupyter nbconvert` 起的是**普通单进程**，**不会自动开 DDP**。
真要用 DDP 得显式 `accelerate launch --num_processes 2 notebook.ipynb`，
而且必须把自定义采样器改成分布式采样器，否则两个进程会取到同一批样本。

### 训练时间够不够：开跑前就有投影

Step8 探测时会量出「每样本耗时」，据此投影 **5 次运行**（4 个单子集 + 1 个联合）
的总时长并打印；若超过 `TIME_BUDGET_H`（本文件为 8 小时，给评估与扫描分析留余量），
会给出三条可选做法。**它只报警、不自动改 EPOCHS** —— 自动改会让两条基线的
训练轮数不同，对照口径就废了。


### 图上的文字：一律英文

Kaggle 等云端镜像的默认字体（DejaVu Sans）**不含中文字形**，中文会静默变成方块且不报错
⇒ 图上文字固定英文（`PLOT_LANG="en"`），字体只挑能显示拉丁字母与德语变音的通用字体。

### 自动跑了哪些运行

`RUN_PLAN`（Step1）是唯一的开关：

| 运行 | 训练数据 | 覆盖的评测设置 | 用途 |
|---|---|---|---|
| `single / en-nn` | 只有 en-nn | en-nn·ModAvg、en-nn·HeadAvg | 单子集分析 |
| `single / de-nn` | 只有 de-nn | de-nn·ModAvg、de-nn·HeadAvg | 单子集分析 |
| `single / en-pv` | 只有 en-pv | en-pv·Avg | 单子集分析 |
| `single / de-pv` | 只有 de-pv | de-pv·Avg | 单子集分析 |
| `joint` | 四个子集拼起来 | 全部 6 个 | **一个模型覆盖 6 个评测设置**，正式提交用 |

**联合模式的三处技术处理**（这也是 single 与 joint 能共用一套代码的原因）：

1. **统一输出维度**：四个子集都用 2 列输出（复合词：修饰语/中心语；小品词动词：只有第 1 列有标签，
   第 2 列用 `loss_mask` 屏蔽）。模型结构、损失、指标、绘图因此完全不用判断子集类型。
2. **分组键带子集**：`group_index` 编的是 **(子集, 目标)** 而不是单纯的目标。
   指标 2 的组内 std 必须按这个分组算，否则不同子集里同名目标会被串成一组。
3. **切分在子集内部做**：每个子集各自按「目标级互斥 + 先分箱后分层抽样」切分，
   再把四个子集的 train/val 拼起来。这样每个子集的验证集难度分布都保持正确。

**想少跑几次**：直接注释掉 `RUN_PLAN` 里对应的行，其余代码一行不用动。

### 云端跑法（推荐命令）

```bash
# 1) 拉代码与数据（数据放在仓库的 train/ 下）
git clone <your-repo> ComPartMENT && cd ComPartMENT

# 2) 环境变量（都可省略，notebook 会自动探测）
export COMPARTMENT_TRAIN_DIR=/kaggle/input/datasets/kintsugi0v0/compartment-data-v2/train_data_v2
export COMPARTMENT_OUTPUT=/kaggle/working/outputs
export COMPARTMENT_HF_MODE=auto        # auto=先探测官方源，不通换镜像
export TOKENIZERS_PARALLELISM=false

# 3) 依赖
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" \
            scikit-learn pandas matplotlib seaborn scipy jupyter nbconvert

# 4) 先用 30 秒跑通整条链路（2 层随机主干，不下权重）
#    用环境变量开冒烟模式（COMPARTMENT_SMOKE=1，2 层随机主干，不下权重）
COMPARTMENT_SMOKE=1 jupyter nbconvert --to notebook --execute \
  "v9/ComPartMENT(XLM-RoBERTa-large-V9多seed轮数扫描)/ComPartMENT_XLM-RoBERTa-large_V9.ipynb" \
  --ExecutePreprocessor.timeout=1800

# 5) 正式跑（不设 COMPARTMENT_SMOKE 即为正式模式）
nohup jupyter nbconvert --to notebook --execute \
  "v9/ComPartMENT(XLM-RoBERTa-large-V9多seed轮数扫描)/ComPartMENT_XLM-RoBERTa-large_V9.ipynb" \
  --ExecutePreprocessor.timeout=7200 > train.log 2>&1 &
```

**数据路径不用改代码**：notebook 会按「环境变量 `COMPARTMENT_TRAIN_DIR` → Kaggle 数据集
`compartment-data-v2`（原始 TSV 在 `train_data_v2/` 根目录）→ 仓库内 `train/`」的顺序自动探测，
找不到时再用递归 glob 兜底（排除 `filtered/` 下的合成数据）。Kaggle 上挂上
`compartment-data-v2` dataset 直接跑即可，不用改任何一行。

**云端唯一必须改的地方**（Step1）：

| 位置 | 改什么 | 为什么 |
|---|---|---|
| `SMOKE` | 不设 `COMPARTMENT_SMOKE`（默认正式模式） | 冒烟模式（`COMPARTMENT_SMOKE=1`）不下载权重、不产出可用模型 |
| `HF_MODE` | 一般不用改 | `auto` 会先探测官方源；海外云端直通用它最稳 |

### 显存不够时的降级顺序（代价从小到大）

Step8 的**训练档位机制**已经把这套顺序自动化了（见上文的档位表），
这里只列「为什么是这个顺序」：

1. **降 `per_device_train_batch_size`，用 `gradient_accumulation_steps` 把有效 batch 补回去**
   —— 数学等价、几乎无损，第一选择（显存探测会自动做这件事）
   —— ⚠️ 但只省**激活**显存，救不了「权重+梯度+优化器状态」这块固定开销
2. **开 `gradient_checkpointing`**（记得同时 `model.config.use_cache = False`）
   —— 激活降到约 1/N，慢约 30%
3. **换优化器**（`adafactor` 把 m/v 从 8 B/param 压到 ≈0.2；`adamw_bnb_8bit` 压到 2）
   —— 唯一能显著降低**固定开销**的办法
4. **关掉混合精度**（AMP 的 fp16 权重副本占 2 B/param，Qwen-1.5B 就是差这 3.1 GiB）
   —— 代价是纯 fp32 训练慢一些
5. **只训顶部若干层**（`apply_fit_scope`）—— 会冻结底层与 embedding，是**方法层面的改变**，
   必须记进口径清单（本版两条基线各自独立，改动只影响本条基线）
6. **降 `MAX_LENGTH`** —— 最有效，但要改全局配置并**从头重跑**（分词在 Step3 已固化）
7. **换更小的同族模型** —— `microsoft/deberta-v3-base` / `Qwen2.5-0.5B-Instruct`，
   管线一行不用改

### 三个判断「实验有没有做对」的指标

1. **梯度流验证必须通过**：回归头与主干的梯度绝对值和都 > 0，且 loss 不是 NaN。
   否则模型可能训了十几分钟而参数纹丝不动。
2. **指标 2 不能是 `nan`**：组内预测 std 恒为 0 会让相关系数未定义。
   图 C 上如果点全挤在水平线上，就说明模型没有真正对语境敏感。
3. **训练后必须显著优于随机初始化**（参考对照区）：「训练后模型」的指标 1 要明显高于
   「随机初始化」，否则模型没有学到任何东西（此时优先查：标签列是否对齐、
   输入模板是否恒定、学习率是否过大）。

### 后续可以做什么（对应本项目的建模决策）

| 方向 | 做法 | 开关位置 |
|---|---|---|
| 调组内排序的比重（指标 1） | 已开启（占比 0.08，四个子集都有）；调大更重「组内顺序」、调小更重「绝对值」 | `SUBSET_LOSS_PROFILE[子集]["order"]` |
| 调指标 3 的比重 | 已开启（占比 0.05，四个子集都有） | `SUBSET_LOSS_PROFILE[子集]["mse"]` |
| 调防塌陷的比重（指标 2） | 已开启（nn 0.06 / pv 0.10），只在下界被突破时发力 | `SUBSET_LOSS_PROFILE[子集]["guard"]` |
| 调指标 2 的下界 κ | nn 用 0.15（只挡真塌陷）、pv 用 0.50（主动顶住离散度） | `SUBSET_LOSS_PROFILE[子集]["kappa"]` |
| 开关「跨目标 std 序」 | **只在 pv 开**（nn 的指标 2 天花板只有 0.00–0.08） | `SUBSET_LOSS_PROFILE[子集]["xrank"]` |
| 关掉自适应标定 | 改为固定权重（用于消融「自动标定」本身有没有用） | `AUTO_BALANCE=False` |
| **恢复「组内 std 等值匹配」** | ⚠️ **不建议**：人工 std 掺着不可复现的标注噪声，等于让模型拟合噪声 | 见 Step7 的说明 |
| 换模型规模 | `deberta-v2-xlarge` → `deberta-v3-xlarge`；`Qwen2.5-1.5B` → `7B` | `MODEL_REGISTRY` |
| 减少运行次数 | 只跑 joint 一次即可覆盖全部 6 个评测设置 | `RUN_PLAN` |
| 改进定位 | 德语复合词必须预切分（不能字符串切片还原），切分版与未切分版 ensemble | Step3 的 `locate_target` |
| 帕累托前沿 | 把输出缩放系数当旋钮，在 dev 上画「指标1 ρ vs 指标2 ρ」的前沿，而不是用单一指标选点 | Step9 之后 |

### 复现命令

```bash
cd ComPartMENT
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" scikit-learn pandas matplotlib seaborn scipy
jupyter nbconvert --to notebook --execute \
  "v9/ComPartMENT(XLM-RoBERTa-large-V9多seed轮数扫描)/ComPartMENT_XLM-RoBERTa-large_V9.ipynb"
```